In [1]:
import os

ROOT = "/kaggle/input"

keywords = [
    "bird_group_folds_seed16.csv",
    "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy",
    "tta_test_crop_probabilities.npz",
    "test_effnet_b2_audio8s_tta3.npy",
    "test_convnext_tiny_pretrained_v4_resize224_tta3.npy",
    "beats_iter3plus_as2m_8s_train_embeddings.npy",
    "beats_iter3plus_as2m_8s_test_embeddings.npy",
    "triple_v4_nonbird_ood_test_scores.csv",
]

print("=" * 70)
print("MEMERIKSA INPUT NOTEBOOK")
print("=" * 70)

for current_dir, _, files in os.walk(ROOT):
    for filename in files:
        if any(key in filename for key in keywords):
            print(os.path.join(current_dir, filename))

MEMERIKSA INPUT NOTEBOOK
/kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
/kaggle/input/notebooks/radithiafatir/testttrisan/test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy
/kaggle/input/notebooks/radithiafatir/testttrisan/test_convnext_tiny_pretrained_v4_resize224_tta3.npy
/kaggle/input/notebooks/radithiafatir/testttrisan/tta_test_crop_probabilities.npz
/kaggle/input/notebooks/radithiafatir/testttrisan/test_effnet_b2_audio8s_tta3.npy
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_nonbird_ood_test_scores.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/beats_iter3plus_as2m_8s_test_embeddings.npy
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/beats_iter3plus_as2m_8s_train_embeddings.npy


In [2]:
import os
import shutil

TRIPLE_SRC = (
    "/kaggle/input/notebooks/radithiafatir/testttrisan"
)
DIAG_SRC = (
    "/kaggle/input/notebooks/radithiafatir/rill-versi-baru"
)
OUT = "/kaggle/working"

# Salin output diagnostik dan cache BEATs yang sudah tersedia
files_to_reuse = [
    (DIAG_SRC, "triple_v4_nonbird_ood_test_scores.csv"),
    (DIAG_SRC, "beats_iter3plus_as2m_8s_train_embeddings.npy"),
    (DIAG_SRC, "beats_iter3plus_as2m_8s_test_embeddings.npy"),
]

for source_dir, filename in files_to_reuse:
    source_path = os.path.join(source_dir, filename)
    target_path = os.path.join(OUT, filename)

    if not os.path.isfile(source_path):
        raise FileNotFoundError(source_path)

    shutil.copy2(source_path, target_path)
    print("Siap:", target_path)

# Pastikan file penting dari Triple V4 juga tersedia
for filename in [
    "bird_group_folds_seed16.csv",
    "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy",
    "tta_test_crop_probabilities.npz",
]:
    path = os.path.join(TRIPLE_SRC, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)
    print("Triple V4:", path)

print("\nSemua input utama tersedia.")
print("Tidak ada model yang dilatih ulang atau submission dibuat.")

Siap: /kaggle/working/triple_v4_nonbird_ood_test_scores.csv
Siap: /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy
Siap: /kaggle/working/beats_iter3plus_as2m_8s_test_embeddings.npy
Triple V4: /kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
Triple V4: /kaggle/input/notebooks/radithiafatir/testttrisan/test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy
Triple V4: /kaggle/input/notebooks/radithiafatir/testttrisan/tta_test_crop_probabilities.npz

Semua input utama tersedia.
Tidak ada model yang dilatih ulang atau submission dibuat.


In [3]:
import os
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

SCORE_PATH = os.path.join(
    OUT, "triple_v4_nonbird_ood_test_scores.csv"
)
SAMPLE_PATH = os.path.join(ROOT, "sample_submission.csv")

N_UNKNOWN = 245
W_UNCERTAINTY = 0.50
W_KNN = 0.50

scores = pd.read_csv(SCORE_PATH)
sample = pd.read_csv(SAMPLE_PATH)

required = {
    "file_id",
    "triple_v4_pred_known_class",
    "triple_v4_confidence",
    "beats_knn_cosine_distance",
}

missing = required - set(scores.columns)
if missing:
    raise ValueError(f"Kolom tidak ditemukan: {missing}")

scores["file_id"] = scores["file_id"].astype(str)
sample["file_id"] = sample["file_id"].astype(str)

if scores["file_id"].duplicated().any():
    raise ValueError("Ada file_id duplikat pada skor.")

if len(scores) != len(sample):
    raise ValueError("Jumlah skor tidak sama dengan jumlah test.")

if set(scores["file_id"]) != set(sample["file_id"]):
    raise ValueError("file_id skor tidak cocok dengan sample_submission.")

confidence = pd.to_numeric(
    scores["triple_v4_confidence"], errors="coerce"
)
knn_distance = pd.to_numeric(
    scores["beats_knn_cosine_distance"], errors="coerce"
)

if not np.isfinite(confidence).all():
    raise ValueError("Confidence memiliki nilai tidak valid.")

if not np.isfinite(knn_distance).all():
    raise ValueError("KNN distance memiliki nilai tidak valid.")

# Ranking tinggi berarti lebih berpotensi menjadi Unknown.
uncertainty_rank = pd.Series(-confidence).rank(pct=True)
knn_rank = pd.Series(knn_distance).rank(pct=True)

scores["ood_score_conf_knn50"] = (
    W_UNCERTAINTY * uncertainty_rank.to_numpy()
    + W_KNN * knn_rank.to_numpy()
)

# Pilih 245 file dengan skor OOD tertinggi.
selected = scores.sort_values(
    ["ood_score_conf_knn50", "file_id"],
    ascending=[False, True],
    kind="mergesort",
).head(N_UNKNOWN)

unknown_ids = set(selected["file_id"])

# Klasifikasi utama tetap berasal dari Triple V4.
pred_map = scores.set_index("file_id")[
    "triple_v4_pred_known_class"
]

prediction = sample["file_id"].map(pred_map)

if prediction.isna().any():
    raise ValueError("Ada prediksi kelas yang tidak ditemukan.")

prediction = prediction.astype(str)
prediction[sample["file_id"].isin(unknown_ids)] = "Unknown"

target_cols = [c for c in sample.columns if c != "file_id"]
if len(target_cols) != 1:
    raise ValueError(
        f"Format sample_submission tidak dikenali: {sample.columns.tolist()}"
    )

submission = sample.copy()
submission[target_cols[0]] = prediction

submission_path = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_u245.csv"
)
audit_path = os.path.join(
    OUT, "triple_v4_conf_knn50_selected_245.csv"
)

submission.to_csv(submission_path, index=False)
selected.to_csv(audit_path, index=False)

# Validasi hasil
check = pd.read_csv(submission_path)

assert len(check) == len(sample)
assert check["file_id"].equals(sample["file_id"])
assert check[target_cols[0]].notna().all()
assert (check[target_cols[0]] == "Unknown").sum() == N_UNKNOWN
assert set(check[target_cols[0]].unique()).issubset(
    {"Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"}
)

print("=" * 65)
print("TRIPLE V4 + BEATs kNN 50/50 — REPRODUCED")
print("=" * 65)
print("Jumlah Unknown:", int((prediction == "Unknown").sum()))

print("\nDistribusi label final:")
print(pd.Series(prediction).value_counts().to_string())

print("\nDistribusi kelas awal pada kandidat Unknown:")
print(selected["triple_v4_pred_known_class"].value_counts().to_string())

print("\nSubmission:", submission_path)
print("Daftar 245 kandidat:", audit_path)
print("Validasi format: LULUS")
print("Status: BELUM DISUBMIT")

TRIPLE V4 + BEATs kNN 50/50 — REPRODUCED
Jumlah Unknown: 245

Distribusi label final:
file_id
Kutilang       263
Unknown        245
Bondol Jawa    229
Cendrawasih     77

Distribusi kelas awal pada kandidat Unknown:
triple_v4_pred_known_class
Cendrawasih    105
Bondol Jawa     92
Kutilang        48

Submission: /kaggle/working/submission_triple_v4_conf_knn50_u245.csv
Daftar 245 kandidat: /kaggle/working/triple_v4_conf_knn50_selected_245.csv
Validasi format: LULUS
Status: BELUM DISUBMIT


In [4]:
# SCHEMATICS NPC DSC 2026 — Triple V4 Unknown-count sensitivity
# Offline diagnostics only. Does NOT submit to Kaggle.
# Keeps the champion ranking formula fixed: 50% inverse-confidence rank + 50% BEATs kNN-distance rank.

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
ROOT = "/kaggle/input/competitions/schematics-2026"
INPUT_NOTEBOOKS = "/kaggle/input/notebooks/radithiafatir"
TRIPLE_SRC = os.path.join(INPUT_NOTEBOOKS, "testttrisan")
DIAG_SRC = os.path.join(INPUT_NOTEBOOKS, "rill-versi-baru")
OUT = "/kaggle/working"

N_TEST_EXPECTED = 814
U_VALUES = [200, 220, 245, 270, 290]
CHAMPION_FILENAME = "submission_triple_v4_conf_knn50_u245.csv"
CLASS_NAMES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
RESNET_OOF_FILENAME = "oof_resnet50_ensemble_tta3_seed16.npy"
B2_OOF_FILENAME = "oof_efficientnet_b2_audio8s_tta3_seed16.npy"
CONVNEXT_OOF_FILENAME = "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy"
TRIPLE_TEST_PROBS_FILENAME = "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"
TEST_SCORES_FILENAME = "triple_v4_nonbird_ood_test_scores.csv"
OOF_SCORES_FILENAME = "triple_v4_nonbird_ood_oof_known_scores.csv"
SAMPLE_FILENAME = "sample_submission.csv"

os.makedirs(OUT, exist_ok=True)


def find_file(basename, preferred=()):
    """Find file in known locations first, then search notebook outputs/input tree."""
    for path in preferred:
        if path and os.path.isfile(path):
            return path
    roots = [OUT, TRIPLE_SRC, DIAG_SRC, INPUT_NOTEBOOKS, "/kaggle/input/competitions/schematics-2026"]
    seen = set()
    for root in roots:
        if not os.path.isdir(root) or root in seen:
            continue
        seen.add(root)
        # Known output directories first; searching notebook outputs avoids scanning audio files.
        for dirpath, _, filenames in os.walk(root):
            if basename in filenames:
                return os.path.join(dirpath, basename)
    return None


def require_file(basename, preferred=()):
    path = find_file(basename, preferred)
    if path is None:
        raise FileNotFoundError(
            f"Could not find {basename}. Attach the saved output notebook containing this file "
            f"or place it under /kaggle/working."
        )
    return path


def as_probabilities(path, expected_rows=None):
    arr = np.asarray(np.load(path), dtype=np.float64)
    if arr.ndim != 2 or arr.shape[1] != len(CLASS_NAMES):
        raise ValueError(f"Unexpected probability shape for {path}: {arr.shape}")
    if expected_rows is not None and arr.shape[0] != expected_rows:
        raise ValueError(f"Unexpected row count for {path}: {arr.shape[0]} != {expected_rows}")
    if not np.isfinite(arr).all() or np.any(arr < 0):
        raise ValueError(f"Invalid probabilities in {path}")
    row_sums = arr.sum(axis=1, keepdims=True)
    if np.any(row_sums <= 0):
        raise ValueError(f"Invalid zero-sum row in {path}")
    return arr / row_sums


def js_divergence(p, q):
    """Jensen-Shannon divergence (base 2), range 0..1."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(p.sum(), 1e-12)
    q = q / max(q.sum(), 1e-12)
    return float(jensenshannon(p, q, base=2.0) ** 2)


def normalise_label_series(series, context):
    s = series.astype(str).str.strip()
    invalid = sorted(set(s) - set(CLASS_NAMES))
    if invalid:
        raise ValueError(f"Unexpected labels in {context}: {invalid}")
    return s


def empirical_cdf(reference, values):
    """Empirical CDF of values under reference; works without sklearn."""
    ref = np.sort(np.asarray(reference, dtype=float))
    vals = np.asarray(values, dtype=float)
    if len(ref) == 0:
        return np.full(vals.shape, np.nan, dtype=float)
    # right-side counts ties as <=, consistently for OOF and test.
    return np.searchsorted(ref, vals, side="right") / len(ref)


def class_conditional_calibration(oof_df, test_df, oof_pred, test_pred):
    """Create OOF cross-fold and test calibrated OOD scores by predicted class.

    Each OOF row is calibrated against OOF rows of the same predicted class in
    other folds. Test rows are calibrated against all OOF rows of their predicted
    class. High scores suggest unusual samples. This is diagnostic, not a true
    estimator of the number of unseen species.
    """
    oof = oof_df.copy().reset_index(drop=True)
    test = test_df.copy().reset_index(drop=True)
    oof["predicted_class"] = np.asarray(oof_pred)
    test["predicted_class"] = np.asarray(test_pred)
    oof["uncertainty_raw"] = -pd.to_numeric(oof["triple_v4_confidence"]).to_numpy()
    test["uncertainty_raw"] = -pd.to_numeric(test["triple_v4_confidence"]).to_numpy()
    oof["knn_raw"] = pd.to_numeric(oof["beats_knn_cosine_distance"]).to_numpy()
    test["knn_raw"] = pd.to_numeric(test["beats_knn_cosine_distance"]).to_numpy()

    oof_cal = np.full(len(oof), np.nan, dtype=float)
    for idx, row in oof.iterrows():
        cls = row["predicted_class"]
        fold = row["fold"]
        ref = oof[(oof["predicted_class"] == cls) & (oof["fold"] != fold)]
        if len(ref) < 10:
            ref = oof[(oof["predicted_class"] == cls) & (oof.index != idx)]
        u_pct = empirical_cdf(ref["uncertainty_raw"].to_numpy(), [row["uncertainty_raw"]])[0]
        k_pct = empirical_cdf(ref["knn_raw"].to_numpy(), [row["knn_raw"]])[0]
        oof_cal[idx] = 0.5 * u_pct + 0.5 * k_pct

    test_cal = np.full(len(test), np.nan, dtype=float)
    for idx, row in test.iterrows():
        cls = row["predicted_class"]
        ref = oof[oof["predicted_class"] == cls]
        if len(ref) < 10:
            ref = oof
        u_pct = empirical_cdf(ref["uncertainty_raw"].to_numpy(), [row["uncertainty_raw"]])[0]
        k_pct = empirical_cdf(ref["knn_raw"].to_numpy(), [row["knn_raw"]])[0]
        test_cal[idx] = 0.5 * u_pct + 0.5 * k_pct

    return oof_cal, test_cal


print("=" * 88)
print("SCHEMATICS NPC DSC 2026 — TRIPLE V4 UNKNOWN COUNT SENSITIVITY")
print("Offline only; no CNN training; no submission is sent.")
print("=" * 88)

# -----------------------------------------------------------------------------
# Load score tables and expected sample submission
# -----------------------------------------------------------------------------
test_scores_path = require_file(
    TEST_SCORES_FILENAME,
    [os.path.join(OUT, TEST_SCORES_FILENAME), os.path.join(DIAG_SRC, TEST_SCORES_FILENAME)],
)
oof_scores_path = require_file(
    OOF_SCORES_FILENAME,
    [os.path.join(OUT, OOF_SCORES_FILENAME), os.path.join(DIAG_SRC, OOF_SCORES_FILENAME)],
)
sample_path = require_file(SAMPLE_FILENAME, [os.path.join(ROOT, SAMPLE_FILENAME)])

# OOF blend arrays let us reconstruct predicted classes from the baseline R55/B2-05/C40 blend.
r_path = require_file(RESNET_OOF_FILENAME, [os.path.join(TRIPLE_SRC, RESNET_OOF_FILENAME)])
b2_path = require_file(B2_OOF_FILENAME, [os.path.join(TRIPLE_SRC, B2_OOF_FILENAME)])
cx_path = require_file(CONVNEXT_OOF_FILENAME, [os.path.join(TRIPLE_SRC, CONVNEXT_OOF_FILENAME)])
test_probs_path = require_file(
    TRIPLE_TEST_PROBS_FILENAME,
    [os.path.join(TRIPLE_SRC, TRIPLE_TEST_PROBS_FILENAME)],
)

print("Test score table:", test_scores_path)
print("OOF score table:", oof_scores_path)
print("Sample submission:", sample_path)
print("OOF probability arrays:", r_path, b2_path, cx_path, sep="\n  ")
print("Test probability array:", test_probs_path)

test_scores = pd.read_csv(test_scores_path)
oof_scores = pd.read_csv(oof_scores_path)
sample = pd.read_csv(sample_path)

required_test_cols = {"file_id", "triple_v4_pred_known_class", "triple_v4_confidence", "beats_knn_cosine_distance"}
required_oof_cols = {"file_id", "label", "fold", "triple_v4_confidence", "beats_knn_cosine_distance"}
if required_test_cols - set(test_scores.columns):
    raise ValueError(f"Missing test score columns: {required_test_cols - set(test_scores.columns)}")
if required_oof_cols - set(oof_scores.columns):
    raise ValueError(f"Missing OOF score columns: {required_oof_cols - set(oof_scores.columns)}")
if "file_id" not in sample.columns:
    raise ValueError("sample_submission.csv must contain file_id")

for frame, name in [(test_scores, "test scores"), (oof_scores, "OOF scores"), (sample, "sample submission")]:
    frame["file_id"] = frame["file_id"].astype(str)
    if frame["file_id"].duplicated().any():
        raise ValueError(f"Duplicate file_id in {name}")

if len(test_scores) != len(sample):
    raise ValueError(f"Test scores ({len(test_scores)}) and sample rows ({len(sample)}) differ")
if set(test_scores["file_id"]) != set(sample["file_id"]):
    raise ValueError("file_id set in test score table differs from sample submission")

test_scores["triple_v4_pred_known_class"] = normalise_label_series(
    test_scores["triple_v4_pred_known_class"], "test predictions"
)
oof_scores["label"] = normalise_label_series(oof_scores["label"], "OOF labels")

# -----------------------------------------------------------------------------
# Reconstruct baseline probability predictions and validate them against tables
# -----------------------------------------------------------------------------
P_R = as_probabilities(r_path, expected_rows=len(oof_scores))
P_B2 = as_probabilities(b2_path, expected_rows=len(oof_scores))
P_CX = as_probabilities(cx_path, expected_rows=len(oof_scores))
P_OOF = 0.55 * P_R + 0.05 * P_B2 + 0.40 * P_CX
P_OOF = P_OOF / P_OOF.sum(axis=1, keepdims=True)

oof_pred = np.asarray(CLASS_NAMES, dtype=object)[P_OOF.argmax(axis=1)]
oof_conf_from_probs = P_OOF.max(axis=1)

# Sanity-check array order/weighting against the documented Triple V4 OOF score.
from sklearn.metrics import f1_score, accuracy_score
_label_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
y_oof_idx = oof_scores["label"].map(_label_to_idx).to_numpy(dtype=int)
oof_f1_reconstructed = f1_score(
    y_oof_idx, P_OOF.argmax(axis=1), labels=np.arange(len(CLASS_NAMES)),
    average="macro", zero_division=0
)
oof_acc_reconstructed = accuracy_score(y_oof_idx, P_OOF.argmax(axis=1))
print("  Reconstructed OOF Macro F1:", round(float(oof_f1_reconstructed), 6))
print("  Reconstructed OOF accuracy:", round(float(oof_acc_reconstructed), 6))
if abs(oof_f1_reconstructed - 0.942038) > 0.005:
    warnings.warn(
        "Reconstructed OOF F1 differs from saved baseline 0.942038 by >0.005. "
        "Check array order, label order, and the R55/B2-05/C40 blend before trusting calibration."
    )

P_TEST = as_probabilities(test_probs_path, expected_rows=len(sample))
test_pred_from_probs = np.asarray(CLASS_NAMES, dtype=object)[P_TEST.argmax(axis=1)]
test_conf_from_probs = P_TEST.max(axis=1)

# Ensure order by file_id aligns across score table and arrays. OOF files are expected to be in train_df order;
# if the source tables have file_id order different from saved arrays, the folds CSV is the only alignment guarantee.
# Score table and array ordering are created together in the diagnostic notebook; report consistency diagnostics.
score_test_conf = pd.to_numeric(test_scores["triple_v4_confidence"], errors="coerce").to_numpy()
score_test_pred = test_scores["triple_v4_pred_known_class"].to_numpy()
if not np.isfinite(score_test_conf).all():
    raise ValueError("Invalid confidence values in test score table")

print("\nReconstructed probability diagnostics:")
print("  OOF blend shape:", P_OOF.shape)
print("  Test blend shape:", P_TEST.shape)
print("  Test confidence max absolute difference:", float(np.max(np.abs(score_test_conf - test_conf_from_probs))))
print("  Test predicted-class agreement:", float(np.mean(score_test_pred == test_pred_from_probs)))

# If the diagnostic table's OOF confidence was created from the same blend, this checks alignment.
oof_score_conf = pd.to_numeric(oof_scores["triple_v4_confidence"], errors="coerce").to_numpy()
print("  OOF confidence max absolute difference:", float(np.nanmax(np.abs(oof_score_conf - oof_conf_from_probs))))

# For exact consistency and robust class-conditional comparisons, use the table's per-row score values.
test_conf = score_test_conf
test_knn = pd.to_numeric(test_scores["beats_knn_cosine_distance"], errors="coerce").to_numpy()
oof_conf = oof_score_conf
oof_knn = pd.to_numeric(oof_scores["beats_knn_cosine_distance"], errors="coerce").to_numpy()
if not np.isfinite(test_knn).all() or not np.isfinite(oof_knn).all() or not np.isfinite(oof_conf).all():
    raise ValueError("Non-finite confidence or kNN distance")

# -----------------------------------------------------------------------------
# Champion exact 245-unknown set reconstruction and verification
# -----------------------------------------------------------------------------
# This matches the candidate construction already used in Kaggle:
# high inverse-confidence percentile + high kNN-distance percentile => Unknown.
test_scores = test_scores.copy()
test_scores["confidence_uncertainty_rank"] = pd.Series(-test_conf).rank(pct=True).to_numpy()
test_scores["knn_distance_rank"] = pd.Series(test_knn).rank(pct=True).to_numpy()
test_scores["combined_rank_score_50_50"] = 0.50 * test_scores["confidence_uncertainty_rank"] + 0.50 * test_scores["knn_distance_rank"]
test_scores["triple_v4_pred_known_class"] = test_pred_from_probs if np.mean(score_test_pred == test_pred_from_probs) > 0.999 else score_test_pred

champion_path = find_file(CHAMPION_FILENAME, [os.path.join(OUT, CHAMPION_FILENAME)])
champion_found = champion_path is not None
champion_exact_match = None
champion_overlap = None

reconstructed_245 = test_scores.sort_values(
    ["combined_rank_score_50_50", "file_id"],
    ascending=[False, True],
    kind="mergesort",
).head(245)
reconstructed_245_ids = set(reconstructed_245["file_id"])

if champion_found:
    champ = pd.read_csv(champion_path)
    if "file_id" not in champ.columns:
        raise ValueError(f"Champion CSV has no file_id: {champion_path}")
    target_cols = [c for c in champ.columns if c != "file_id"]
    if len(target_cols) != 1:
        raise ValueError(f"Can't identify champion target column: {champ.columns.tolist()}")
    champ["file_id"] = champ["file_id"].astype(str)
    champ_unknown_ids = set(champ.loc[champ[target_cols[0]].astype(str) == "Unknown", "file_id"])
    champion_overlap = len(champ_unknown_ids & reconstructed_245_ids)
    champion_exact_match = (champion_overlap == 245 and len(champ_unknown_ids) == 245)
    print("\nChampion file found:", champion_path)
    print("  Unknown count in champion:", len(champ_unknown_ids))
    print("  Reconstructed overlap:", f"{champion_overlap}/245")
    print("  Exact 245/245 match:", champion_exact_match)
    if not champion_exact_match:
        warnings.warn("Champion reconstruction does not match 245/245. Continue sensitivity analysis, but inspect score formula/order before trusting any candidate.")
else:
    print("\nWARNING: original champion CSV wasn't found among current working/input files.")
    print("Reconstruction is available, but exact identity against the saved champion could not be independently verified.")

# Save per-row raw test scores (rank ranking only; no labels are changed).
score_rows_path = os.path.join(OUT, "triple_v4_unknown_count_rank_scores.csv")
test_scores.to_csv(score_rows_path, index=False)

# -----------------------------------------------------------------------------
# Sensitivity sweep for U = 200, 220, 245, 270, 290
# -----------------------------------------------------------------------------
# Compare the remaining known predicted-class mix to two references:
# (a) OOF model-predicted class distribution, (b) training true-label distribution.
oof_pred_series = pd.Series(oof_pred, name="oof_predicted_class")
oof_pred_counts = oof_pred_series.value_counts().reindex(CLASS_NAMES, fill_value=0)
oof_pred_prior = (oof_pred_counts / len(oof_pred_series)).to_numpy(dtype=float)
oof_true_counts = oof_scores["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
train_true_prior = (oof_true_counts / oof_true_counts.sum()).to_numpy(dtype=float)

test_baseline_pred = test_scores["triple_v4_pred_known_class"].astype(str)
summary_rows = []
selection_manifest = []
all_scores = test_scores["combined_rank_score_50_50"].to_numpy(dtype=float)
rank_order = test_scores.assign(_score=all_scores).sort_values(
    ["_score", "file_id"], ascending=[False, True], kind="mergesort"
)

for U in U_VALUES:
    if not (0 < U < len(test_scores)):
        raise ValueError(f"Invalid U={U}")
    chosen = rank_order.head(U).copy()
    chosen_ids = set(chosen["file_id"])
    remain = test_scores[~test_scores["file_id"].isin(chosen_ids)].copy()

    chosen_counts = chosen["triple_v4_pred_known_class"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    remaining_counts = remain["triple_v4_pred_known_class"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    remaining_prior = (remaining_counts / max(remaining_counts.sum(), 1)).to_numpy(dtype=float)

    summary_rows.append({
        "U_unknown": U,
        "unknown_fraction_test": U / len(test_scores),
        "score_cutoff_lowest_selected": float(chosen["combined_rank_score_50_50"].min()),
        "selected_from_pred_Bondol": int(chosen_counts["Bondol Jawa"]),
        "selected_from_pred_Cendrawasih": int(chosen_counts["Cendrawasih"]),
        "selected_from_pred_Kutilang": int(chosen_counts["Kutilang"]),
        "remaining_Bondol": int(remaining_counts["Bondol Jawa"]),
        "remaining_Cendrawasih": int(remaining_counts["Cendrawasih"]),
        "remaining_Kutilang": int(remaining_counts["Kutilang"]),
        "remaining_Bondol_share": float(remaining_prior[0]),
        "remaining_Cendrawasih_share": float(remaining_prior[1]),
        "remaining_Kutilang_share": float(remaining_prior[2]),
        "JS_divergence_remaining_vs_OOF_predicted_prior": js_divergence(remaining_prior, oof_pred_prior),
        "JS_divergence_remaining_vs_train_true_prior": js_divergence(remaining_prior, train_true_prior),
        "selected_confidence_mean": float(chosen["triple_v4_confidence"].mean()),
        "selected_knn_distance_mean": float(chosen["beats_knn_cosine_distance"].mean()),
        "overlap_with_champion_245": (
            len(chosen_ids & reconstructed_245_ids) if U != 245 else len(chosen_ids & reconstructed_245_ids)
        ),
    })

    for rank, (_, row) in enumerate(chosen.iterrows(), start=1):
        selection_manifest.append({
            "U_unknown": U,
            "rank_unknown_score": rank,
            "file_id": row["file_id"],
            "predicted_class_before_rejection": row["triple_v4_pred_known_class"],
            "triple_v4_confidence": row["triple_v4_confidence"],
            "beats_knn_cosine_distance": row["beats_knn_cosine_distance"],
            "combined_rank_score_50_50": row["combined_rank_score_50_50"],
        })

sensitivity = pd.DataFrame(summary_rows)
manifest = pd.DataFrame(selection_manifest)
sensitivity_path = os.path.join(OUT, "triple_v4_unknown_count_sensitivity.csv")
manifest_path = os.path.join(OUT, "triple_v4_unknown_count_selection_manifest.csv")
sensitivity.to_csv(sensitivity_path, index=False)
manifest.to_csv(manifest_path, index=False)

# -----------------------------------------------------------------------------
# Prior-aware class-conditional tail diagnostic
# -----------------------------------------------------------------------------
# Reconstruct OOF predicted class from the R55/B2-05/C40 probability blend.
# Use fold-excluded reference distributions for OOF; use all OOF references for test.
# This estimates score-tail excess after conditioning on predicted class; it is NOT a true Unknown count.
if "fold" not in oof_scores.columns:
    raise ValueError("OOF score table must contain fold for cross-fold calibration")

oof_fold = pd.to_numeric(oof_scores["fold"], errors="coerce")
if not np.isfinite(oof_fold).all():
    raise ValueError("Invalid OOF fold values")

oof_tail_scores, test_tail_scores = class_conditional_calibration(
    oof_scores, test_scores, oof_pred, test_baseline_pred.to_numpy()
)
oof_diag = oof_scores.copy()
oof_diag["predicted_class"] = oof_pred
oof_diag["ood_score_class_conditional_50_50"] = oof_tail_scores
test_diag = test_scores.copy()
test_diag["ood_score_class_conditional_50_50"] = test_tail_scores

# Calculate tail rates within each predicted class versus the corresponding OOF predicted-class score distribution.
tail_rows = []
quantiles = [0.90, 0.95, 0.975, 0.99]
for cls in CLASS_NAMES:
    ref = oof_diag.loc[oof_diag["predicted_class"] == cls, "ood_score_class_conditional_50_50"].dropna().to_numpy()
    tst = test_diag.loc[test_diag["triple_v4_pred_known_class"] == cls, "ood_score_class_conditional_50_50"].dropna().to_numpy()
    if len(ref) == 0 or len(tst) == 0:
        continue
    for q in quantiles:
        threshold = float(np.quantile(ref, q))
        observed = int((tst > threshold).sum())
        observed_rate = observed / len(tst)
        expected_known_tail = (1.0 - q) * len(tst)
        excess = max(0.0, observed - expected_known_tail)
        tail_rows.append({
            "predicted_class": cls,
            "OOF_reference_count": len(ref),
            "test_count_predicted_as_class": len(tst),
            "OOF_tail_quantile": q,
            "OOF_threshold": threshold,
            "test_count_above_threshold": observed,
            "test_tail_rate": observed_rate,
            "expected_tail_count_if_all_known_and_matched": expected_known_tail,
            "positive_excess_tail_count_diagnostic_only": excess,
        })

tail_df = pd.DataFrame(tail_rows)
tail_path = os.path.join(OUT, "triple_v4_prior_adjusted_class_conditional_tail.csv")
tail_df.to_csv(tail_path, index=False)

# -----------------------------------------------------------------------------
# Visual diagnostics: overall score distribution and class-conditional calibrated score.
# Plots are exploratory; rank-score KDE/valleys are not a label oracle.
# -----------------------------------------------------------------------------
fig_path = os.path.join(OUT, "triple_v4_unknown_count_sensitivity.png")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(test_scores["combined_rank_score_50_50"], bins=30, density=True, alpha=0.75)
for U in U_VALUES:
    chosen = rank_order.head(U)
    cutoff = float(chosen["combined_rank_score_50_50"].min())
    axes[0].axvline(cutoff, linestyle="--", linewidth=1, label=f"U={U}")
axes[0].set_title("Test combined rank score and U cutoffs")
axes[0].set_xlabel("50% inverse-confidence rank + 50% kNN-distance rank")
axes[0].set_ylabel("Density")
axes[0].legend()

for cls in CLASS_NAMES:
    ref = oof_diag.loc[oof_diag["predicted_class"] == cls, "ood_score_class_conditional_50_50"].dropna().to_numpy()
    tst = test_diag.loc[test_diag["triple_v4_pred_known_class"] == cls, "ood_score_class_conditional_50_50"].dropna().to_numpy()
    if len(ref):
        axes[1].hist(ref, bins=20, density=True, alpha=0.22, label=f"OOF {cls}")
    if len(tst):
        axes[1].hist(tst, bins=20, density=True, alpha=0.22, histtype="step", linewidth=1.5, label=f"Test {cls}")
axes[1].set_title("Class-conditional calibrated score: OOF vs test")
axes[1].set_xlabel("Higher = more unusual vs OOF predicted-class reference")
axes[1].set_ylabel("Density")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(fig_path, dpi=150)
plt.close(fig)

# -----------------------------------------------------------------------------
# Save a clear run summary
# -----------------------------------------------------------------------------
summary_json = {
    "test_rows": int(len(test_scores)),
    "oof_rows": int(len(oof_scores)),
    "u_values_tested": U_VALUES,
    "ranking_weights": {"inverse_confidence_rank": 0.50, "beats_knn_distance_rank": 0.50},
    "champion_csv_found": bool(champion_found),
    "champion_exact_unknown_set_match_245_of_245": champion_exact_match,
    "champion_csv_path": champion_path,
    "oof_predicted_class_share": {k: float(v) for k, v in zip(CLASS_NAMES, oof_pred_prior)},
    "train_true_class_share": {k: float(v) for k, v in zip(CLASS_NAMES, train_true_prior)},
    "interpretation_warning": (
        "Offline diagnostics only. Class-conditional tail excess, JS divergence, and plots are not a validated estimator of true Unknown count. "
        "OOF LOCO is a proxy; test class priors and domains may differ. Do not infer the optimal U from these tables alone."
    ),
    "outputs": {
        "sensitivity_csv": sensitivity_path,
        "selection_manifest_csv": manifest_path,
        "prior_adjusted_tail_csv": tail_path,
        "rank_scores_csv": score_rows_path,
        "plot_png": fig_path,
    },
}
summary_json_path = os.path.join(OUT, "triple_v4_unknown_count_sensitivity_summary.json")
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
print("\n" + "=" * 120)
print("UNKNOWN COUNT SENSITIVITY — DISTRIBUTION SUMMARY")
print("=" * 120)
print(sensitivity.round(4).to_string(index=False))
print("\n" + "=" * 120)
print("PRIOR-AWARE CLASS-CONDITIONAL TAIL DIAGNOSTIC")
print("Positive excess tail is a diagnostic, NOT an estimate of true unknown count.")
print("=" * 120)
if len(tail_df):
    print(tail_df.round(4).to_string(index=False))
else:
    print("No tail diagnostics produced.")
print("\nOOF predicted-class share:", dict(zip(CLASS_NAMES, np.round(oof_pred_prior, 4))))
print("Training true-class share:", dict(zip(CLASS_NAMES, np.round(train_true_prior, 4))))
print("\nSaved outputs:")
for name, path in summary_json["outputs"].items():
    print(f" - {name}: {path}")
print(" - summary_json:", summary_json_path)
print("\nNo submission CSV created and nothing was submitted.")


SCHEMATICS NPC DSC 2026 — TRIPLE V4 UNKNOWN COUNT SENSITIVITY
Offline only; no CNN training; no submission is sent.
Test score table: /kaggle/working/triple_v4_nonbird_ood_test_scores.csv
OOF score table: /kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_nonbird_ood_oof_known_scores.csv
Sample submission: /kaggle/input/competitions/schematics-2026/sample_submission.csv
OOF probability arrays:
  /kaggle/input/notebooks/radithiafatir/testttrisan/oof_resnet50_ensemble_tta3_seed16.npy
  /kaggle/input/notebooks/radithiafatir/testttrisan/oof_efficientnet_b2_audio8s_tta3_seed16.npy
  /kaggle/input/notebooks/radithiafatir/testttrisan/oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy
Test probability array: /kaggle/input/notebooks/radithiafatir/testttrisan/test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy
  Reconstructed OOF Macro F1: 0.942038
  Reconstructed OOF accuracy: 0.984118

Reconstructed probability diagnostics:
  OOF blend shape: (182

In [5]:
# SCHEMATICS NPC DSC 2026 — Pre-submit sensitivity checks
# Offline diagnostics only. Does NOT create a submission or submit to Kaggle.
# Checks: champion reproduction, component-wise tails, bulk quantiles, confidence-floor audit,
# class-conditional implicit pi(p) with mid-rank CDF/bootstrap CI, N_imp(p), and fold-averaging notes.
# Important: pi(p) is only interpretable under an upper-tail mixture assumption and matched known-score
# distribution; class-conditional and global prior/domain shifts can violate that assumption.

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = "/kaggle/input/competitions/schematics-2026"
INPUT_NOTEBOOKS = "/kaggle/input/notebooks/radithiafatir"
TRIPLE_SRC = os.path.join(INPUT_NOTEBOOKS, "testttrisan")
DIAG_SRC = os.path.join(INPUT_NOTEBOOKS, "rill-versi-baru")
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

CLASS_NAMES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
N_TEST_EXPECTED = 814
CHAMPION_U = 245
CHAMPION_FILENAME = "submission_triple_v4_conf_knn50_u245.csv"
TEST_SCORES_FILENAME = "triple_v4_nonbird_ood_test_scores.csv"
OOF_SCORES_FILENAME = "triple_v4_nonbird_ood_oof_known_scores.csv"
SAMPLE_FILENAME = "sample_submission.csv"
RESNET_OOF_FILENAME = "oof_resnet50_ensemble_tta3_seed16.npy"
B2_OOF_FILENAME = "oof_efficientnet_b2_audio8s_tta3_seed16.npy"
CONVNEXT_OOF_FILENAME = "oof_convnext_tiny_audio8s_tta3_seed16_pretrained_v4_resize224.npy"
TEST_PROBS_FILENAME = "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"


def find_file(basename, preferred=()):
    for path in preferred:
        if path and os.path.isfile(path):
            return path
    roots = [OUT, TRIPLE_SRC, DIAG_SRC, INPUT_NOTEBOOKS]
    visited = set()
    for root in roots:
        if not os.path.isdir(root) or root in visited:
            continue
        visited.add(root)
        for dirpath, _, filenames in os.walk(root):
            if basename in filenames:
                return os.path.join(dirpath, basename)
    return None


def require_file(basename, preferred=()):
    path = find_file(basename, preferred)
    if path is None:
        raise FileNotFoundError(
            f"Cannot find {basename}. Attach the saved output notebook containing this file as input."
        )
    return path


def load_probs(path, expected_rows):
    p = np.asarray(np.load(path), dtype=np.float64)
    if p.ndim != 2 or p.shape != (expected_rows, len(CLASS_NAMES)):
        raise ValueError(f"Unexpected probability shape for {path}: {p.shape}")
    if not np.isfinite(p).all() or np.any(p < 0):
        raise ValueError(f"Invalid probability values in {path}")
    sums = p.sum(axis=1, keepdims=True)
    if np.any(sums <= 0):
        raise ValueError(f"Zero-sum probability row in {path}")
    return p / sums


def normalize_label(s):
    return s.astype(str).str.strip()


def quantile_summary(ref, test, p_values, pi_values, method, stratum):
    """Compare test lower-tail quantiles with OOF quantiles under hypothetical Unknown fractions."""
    rows = []
    ref = np.asarray(ref, dtype=float)
    test = np.asarray(test, dtype=float)
    ref = ref[np.isfinite(ref)]
    test = test[np.isfinite(test)]
    if len(ref) < 10 or len(test) < 5:
        return rows
    q25, q75 = np.quantile(ref, [0.25, 0.75])
    scale = max(float(q75 - q25), 1e-8)
    for pi in pi_values:
        for p in p_values:
            q_ref = p / (1.0 - pi)
            if not 0 < q_ref < 1:
                continue
            test_val = float(np.quantile(test, p))
            ref_val = float(np.quantile(ref, q_ref))
            rows.append({
                "predicted_class_stratum": stratum,
                "signal_high_means_OOD": method,
                "assumed_unknown_fraction_pi": pi,
                "test_quantile_p": p,
                "mapped_OOF_quantile_p_over_1_minus_pi": q_ref,
                "n_test_stratum": len(test),
                "n_oof_reference_stratum": len(ref),
                "test_score_quantile": test_val,
                "OOF_reference_score_quantile": ref_val,
                "test_minus_reference": test_val - ref_val,
                "difference_in_OOF_IQR_units": (test_val - ref_val) / scale,
                "OOF_IQR_scale": scale,
            })
    return rows


def empirical_midrank_cdf(sorted_ref, x):
    """Empirical mid-rank CDF: (F(x-) + F(x))/2, handling exact ties symmetrically."""
    sorted_ref = np.asarray(sorted_ref, dtype=float)
    sorted_ref = sorted_ref[np.isfinite(sorted_ref)]
    if len(sorted_ref) == 0 or not np.isfinite(x):
        return np.nan
    left = int(np.searchsorted(sorted_ref, x, side="left"))
    right = int(np.searchsorted(sorted_ref, x, side="right"))
    return float((left + 0.5 * (right - left)) / len(sorted_ref))


def implicit_pi_summary(ref, test, p_values, method, stratum):
    """Calculate raw pi(p)=1-p/F_OOF(q_test(p)) using the empirical mid-rank CDF."""
    rows = []
    ref = np.asarray(ref, dtype=float)
    test = np.asarray(test, dtype=float)
    ref = np.sort(ref[np.isfinite(ref)])
    test = test[np.isfinite(test)]
    if len(ref) < 10 or len(test) < 5:
        return rows
    for p in p_values:
        q_test = float(np.quantile(test, p))
        f_oof = empirical_midrank_cdf(ref, q_test)
        pi_raw = (1.0 - p / f_oof) if np.isfinite(f_oof) and f_oof > 0 else np.nan
        rows.append({
            "predicted_class_stratum": stratum,
            "signal_high_means_OOD": method,
            "test_quantile_p": p,
            "n_test_stratum": len(test),
            "n_oof_reference_stratum": len(ref),
            "test_score_quantile": q_test,
            "F_OOF_midrank_at_test_quantile": f_oof,
            "pi_implicit_raw_midrank": pi_raw,
            "note": "Raw diagnostic only; not clipped. Requires matched known-score distribution and upper-tail mixture assumptions.",
        })
    return rows


def bootstrap_pi_by_class(oof_df, test_df, signal, p_values, n_boot, rng):
    """Bootstrap OOF source_groups globally and TEST rows within each predicted-class stratum."""
    out_rows = []
    boot_map = {}
    oof_all = oof_df.copy()
    test_all = test_df.copy()
    if "source_group" not in oof_all.columns or oof_all["source_group"].isna().any():
        raise ValueError("OOF source_group must be present and non-missing for cluster bootstrap.")

    oof_all[signal] = pd.to_numeric(oof_all[signal], errors="coerce")
    test_all[signal] = pd.to_numeric(test_all[signal], errors="coerce")
    oof_all = oof_all[np.isfinite(oof_all[signal].to_numpy(dtype=float))].copy()
    test_all = test_all[np.isfinite(test_all[signal].to_numpy(dtype=float))].copy()
    all_groups = pd.unique(oof_all["source_group"].astype(str).to_numpy())
    if len(all_groups) < 2:
        raise ValueError("Too few OOF source groups for cluster bootstrap.")
    all_group_array = np.asarray(all_groups, dtype=object)

    # Build each class's class-conditional score reference as group -> values.
    ref_by_class = {}
    test_by_class = {}
    point_pi_by_class = {}
    floor_by_class = {}
    for cls in CLASS_NAMES:
        ref_df = oof_all[oof_all["predicted_class_reconstructed"] == cls].copy()
        test_cls = test_all[test_all["predicted_class_reconstructed"] == cls].copy()
        ref_scores = ref_df[signal].to_numpy(dtype=float)
        test_scores = test_cls[signal].to_numpy(dtype=float)
        if len(ref_scores) < 10 or len(test_scores) < 5:
            continue

        ref_group_map = {}
        for group, gdf in ref_df.groupby(ref_df["source_group"].astype(str), sort=False):
            vals = gdf[signal].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals):
                ref_group_map[str(group)] = vals
        ref_by_class[cls] = ref_group_map
        test_by_class[cls] = test_scores
        sorted_ref = np.sort(ref_scores)
        point_pi = {}
        for p in p_values:
            q_test = float(np.quantile(test_scores, p))
            f_mid = empirical_midrank_cdf(sorted_ref, q_test)
            point_pi[p] = (1.0 - p / f_mid) if np.isfinite(f_mid) and f_mid > 0 else np.nan
        point_pi_by_class[cls] = point_pi

        floor_oof = float(np.mean(ref_df["inverse_confidence_ood"].to_numpy(dtype=float) <= CONF_EPS)) if "inverse_confidence_ood" in ref_df.columns else np.nan
        floor_test = float(np.mean(test_cls["inverse_confidence_ood"].to_numpy(dtype=float) <= CONF_EPS)) if "inverse_confidence_ood" in test_cls.columns else np.nan
        floor_by_class[cls] = (floor_oof, floor_test)

        boot_map[cls] = np.full((n_boot, len(p_values)), np.nan, dtype=float)

    # Shared group draw per replicate preserves dependence between predicted-class strata.
    for b in range(n_boot):
        sampled_groups = rng.choice(all_group_array, size=len(all_group_array), replace=True)
        for cls in CLASS_NAMES:
            if cls not in ref_by_class or cls not in test_by_class:
                continue
            group_map = ref_by_class[cls]
            ref_parts = [group_map[str(group)] for group in sampled_groups if str(group) in group_map]
            ref_boot = np.concatenate(ref_parts) if ref_parts else np.asarray([], dtype=float)
            if len(ref_boot) == 0:
                continue
            ref_boot.sort()
            test_scores = test_by_class[cls]
            test_boot = rng.choice(test_scores, size=len(test_scores), replace=True)
            for j, p in enumerate(p_values):
                q_test = float(np.quantile(test_boot, p))
                f_mid = empirical_midrank_cdf(ref_boot, q_test)
                if np.isfinite(f_mid) and f_mid > 0:
                    boot_map[cls][b, j] = 1.0 - p / f_mid

    for cls in CLASS_NAMES:
        if cls not in boot_map:
            continue
        floor_oof, floor_test = floor_by_class[cls]
        floor_warning = (
            signal == "log10_inverse_confidence_ood"
            and (floor_oof > 0.05 or floor_test > 0.05)
        )
        for j, p in enumerate(p_values):
            vals = boot_map[cls][:, j]
            vals = vals[np.isfinite(vals)]
            ci_low, ci_high = (
                np.quantile(vals, [0.025, 0.975]).tolist()
                if len(vals) >= max(30, 0.5 * n_boot) else [np.nan, np.nan]
            )
            ignore_by_floor = bool(floor_warning and p <= 0.25)
            out_rows.append({
                "signal": signal,
                "predicted_class": cls,
                "p": p,
                "n_test_class": len(test_by_class[cls]),
                "n_oof_rows_class": sum(len(v) for v in ref_by_class[cls].values()),
                "n_oof_source_groups_class": len(ref_by_class[cls]),
                "pi_point_raw_midrank": point_pi_by_class[cls][p],
                "pi_bootstrap_ci95_low": ci_low,
                "pi_bootstrap_ci95_high": ci_high,
                "bootstrap_reps_requested": n_boot,
                "bootstrap_reps_valid": int(len(vals)),
                "oof_confidence_floor_fraction": floor_oof,
                "test_confidence_floor_fraction": floor_test,
                "ignored_due_to_confidence_floor_rule": ignore_by_floor,
                "pi_for_plot_or_decision": np.nan if ignore_by_floor else point_pi_by_class[cls][p],
                "interpretation_status": (
                    "IGNORE: confidence floor fraction >5% and p<=0.25"
                    if ignore_by_floor else "Diagnostic only; mixture/domain assumptions still apply"
                ),
            })
    return pd.DataFrame(out_rows), boot_map

print("=" * 96)
print("SCHEMATICS NPC DSC 2026 — PRE-SUBMIT SENSITIVITY CHECKS")
print("Offline diagnostics only; no submission is created or sent.")
print("=" * 96)

# 1. Locate and load saved score tables, arrays, sample submission.
test_scores_path = require_file(
    TEST_SCORES_FILENAME,
    [os.path.join(OUT, TEST_SCORES_FILENAME), os.path.join(DIAG_SRC, TEST_SCORES_FILENAME)],
)
oof_scores_path = require_file(
    OOF_SCORES_FILENAME,
    [os.path.join(OUT, OOF_SCORES_FILENAME), os.path.join(DIAG_SRC, OOF_SCORES_FILENAME)],
)
sample_path = require_file(SAMPLE_FILENAME, [os.path.join(ROOT, SAMPLE_FILENAME)])
r_path = require_file(RESNET_OOF_FILENAME, [os.path.join(TRIPLE_SRC, RESNET_OOF_FILENAME)])
b2_path = require_file(B2_OOF_FILENAME, [os.path.join(TRIPLE_SRC, B2_OOF_FILENAME)])
cx_path = require_file(CONVNEXT_OOF_FILENAME, [os.path.join(TRIPLE_SRC, CONVNEXT_OOF_FILENAME)])
test_probs_path = require_file(TEST_PROBS_FILENAME, [os.path.join(TRIPLE_SRC, TEST_PROBS_FILENAME)])

test = pd.read_csv(test_scores_path).copy()
oof = pd.read_csv(oof_scores_path).copy()
sample = pd.read_csv(sample_path).copy()
for frame in (test, oof, sample):
    frame["file_id"] = frame["file_id"].astype(str)

if test["file_id"].duplicated().any() or oof["file_id"].duplicated().any() or sample["file_id"].duplicated().any():
    raise ValueError("Duplicate file_id found in a score table or sample submission.")
if len(test) != len(sample):
    raise ValueError(f"Test score rows {len(test)} != sample_submission rows {len(sample)}")
if set(test["file_id"]) != set(sample["file_id"]):
    raise ValueError("Test score file IDs do not match sample_submission.")

for col in ["triple_v4_confidence", "beats_knn_cosine_distance"]:
    if col not in test.columns or col not in oof.columns:
        raise ValueError(f"Missing required score column: {col}")
if "fold" not in oof.columns or "label" not in oof.columns:
    raise ValueError("OOF score table requires fold and label columns.")

P_R = load_probs(r_path, len(oof))
P_B2 = load_probs(b2_path, len(oof))
P_CX = load_probs(cx_path, len(oof))
P_OOF = 0.55 * P_R + 0.05 * P_B2 + 0.40 * P_CX
P_OOF /= P_OOF.sum(axis=1, keepdims=True)
P_TEST = load_probs(test_probs_path, len(sample))

# Validate arrays against score CSVs. This also validates order, not just shape.
oof_conf_csv = pd.to_numeric(oof["triple_v4_confidence"], errors="coerce").to_numpy()
test_conf_csv = pd.to_numeric(test["triple_v4_confidence"], errors="coerce").to_numpy()
if not np.isfinite(oof_conf_csv).all() or not np.isfinite(test_conf_csv).all():
    raise ValueError("Invalid confidence values in score tables.")
oof_conf_rebuilt = P_OOF.max(axis=1)
test_conf_rebuilt = P_TEST.max(axis=1)
oof_conf_diff = float(np.max(np.abs(oof_conf_csv - oof_conf_rebuilt)))
test_conf_diff = float(np.max(np.abs(test_conf_csv - test_conf_rebuilt)))
oof_pred = np.asarray(CLASS_NAMES, dtype=object)[P_OOF.argmax(axis=1)]
test_pred = np.asarray(CLASS_NAMES, dtype=object)[P_TEST.argmax(axis=1)]
oof_pred_agreement = float(np.mean(oof_pred == oof.get("predicted_class", pd.Series(oof_pred)).astype(str).to_numpy())) if "predicted_class" in oof.columns else np.nan
test_pred_csv = normalize_label(test["triple_v4_pred_known_class"]).to_numpy()
test_pred_agreement = float(np.mean(test_pred == test_pred_csv))

print("\nRECONSTRUCTION / ALIGNMENT")
print("  OOF blend shape:", P_OOF.shape)
print("  Test blend shape:", P_TEST.shape)
print("  OOF confidence max abs difference:", f"{oof_conf_diff:.3e}")
print("  Test confidence max abs difference:", f"{test_conf_diff:.3e}")
print("  Test predicted-class agreement with score table:", f"{test_pred_agreement:.6f}")
print("  Reconstructed OOF predicted class from probability arrays; confirm OOF row order via F1/accuracy below.")

from sklearn.metrics import f1_score, accuracy_score
label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
y = normalize_label(oof["label"]).map(label_to_idx)
if y.isna().any():
    raise ValueError("Unknown label in OOF score table.")
y = y.to_numpy(dtype=int)
oof_f1 = float(f1_score(y, P_OOF.argmax(axis=1), labels=np.arange(3), average="macro", zero_division=0))
oof_acc = float(accuracy_score(y, P_OOF.argmax(axis=1)))
print("  Reconstructed OOF Macro F1:", f"{oof_f1:.6f}")
print("  Reconstructed OOF accuracy:", f"{oof_acc:.6f}")
if abs(oof_f1 - 0.942038) > 0.005 or oof_conf_diff > 1e-4 or test_conf_diff > 1e-4 or test_pred_agreement < 0.999:
    warnings.warn("At least one alignment/sanity check differs from known baseline. Inspect array order and blend weighting before interpreting output.")

# Attach reconstructed predicted class and raw OOD-oriented component scores.
oof = oof.reset_index(drop=True)
test = test.reset_index(drop=True)
oof["predicted_class_reconstructed"] = oof_pred
test["predicted_class_reconstructed"] = test_pred
oof["confidence"] = oof_conf_rebuilt
test["confidence"] = test_conf_rebuilt
oof["inverse_confidence_ood"] = 1.0 - oof_conf_rebuilt
test["inverse_confidence_ood"] = 1.0 - test_conf_rebuilt
# Confidence uncertainty is strongly skewed near zero, so use log10(1-confidence)
# for cross-split quantile comparisons. Ranking is unchanged by this monotonic transform.
CONF_EPS = 1e-6
oof["log10_inverse_confidence_ood"] = np.log10(np.clip(oof["inverse_confidence_ood"].to_numpy(), CONF_EPS, None))
test["log10_inverse_confidence_ood"] = np.log10(np.clip(test["inverse_confidence_ood"].to_numpy(), CONF_EPS, None))
# A floor hit means raw (1-confidence) <= CONF_EPS and is collapsed to the same log score.
oof["confidence_floor_hit"] = oof["inverse_confidence_ood"].to_numpy() <= CONF_EPS
test["confidence_floor_hit"] = test["inverse_confidence_ood"].to_numpy() <= CONF_EPS
oof["knn_distance_ood"] = pd.to_numeric(oof["beats_knn_cosine_distance"], errors="coerce")
test["knn_distance_ood"] = pd.to_numeric(test["beats_knn_cosine_distance"], errors="coerce")
for frame in (oof, test):
    if not np.isfinite(frame[["inverse_confidence_ood", "log10_inverse_confidence_ood", "knn_distance_ood"]].to_numpy(dtype=float)).all():
        raise ValueError("Non-finite component score found.")

# 2. Verify champion set exactly reproduces 245/245 where the CSV is accessible.
champion_path = find_file(CHAMPION_FILENAME, [os.path.join(OUT, CHAMPION_FILENAME)])
champion_exact = None
champion_overlap = None
u245_rebuilt_ids = set(
    test.assign(
        inv_rank=pd.Series(test["inverse_confidence_ood"]).rank(pct=True).to_numpy(),
        knn_rank=pd.Series(test["knn_distance_ood"]).rank(pct=True).to_numpy(),
    ).assign(combined=lambda x: 0.5 * x.inv_rank + 0.5 * x.knn_rank)
    .sort_values(["combined", "file_id"], ascending=[False, True], kind="mergesort")
    .head(CHAMPION_U)["file_id"]
)
if champion_path:
    champion = pd.read_csv(champion_path)
    champion["file_id"] = champion["file_id"].astype(str)
    target_cols = [c for c in champion.columns if c != "file_id"]
    if len(target_cols) != 1:
        raise ValueError(f"Cannot identify champion label column: {champion.columns.tolist()}")
    champion_ids = set(champion.loc[champion[target_cols[0]].astype(str) == "Unknown", "file_id"])
    champion_overlap = len(champion_ids & u245_rebuilt_ids)
    champion_exact = len(champion_ids) == CHAMPION_U and champion_overlap == CHAMPION_U
    print("\nCHAMPION RECONSTRUCTION")
    print("  File:", champion_path)
    print("  Champion Unknown count:", len(champion_ids))
    print(f"  Overlap with reconstructed 245: {champion_overlap}/245")
    print("  Exact set match:", champion_exact)
else:
    print("\nCHAMPION CSV is not mounted; use the previously verified champion result as the reference.")

# 3. COMPONENT-WISE TAIL CHECK, stratified by predicted class.
TAIL_QUANTILES = [0.90, 0.95, 0.975, 0.99]
tail_rows = []
for cls in ["ALL"] + CLASS_NAMES:
    oref = oof if cls == "ALL" else oof[oof["predicted_class_reconstructed"] == cls]
    tref = test if cls == "ALL" else test[test["predicted_class_reconstructed"] == cls]
    for signal in ["log10_inverse_confidence_ood", "knn_distance_ood"]:
        ref = oref[signal].to_numpy(dtype=float)
        tst = tref[signal].to_numpy(dtype=float)
        if len(ref) < 10 or len(tst) < 5:
            continue
        for q in TAIL_QUANTILES:
            threshold = float(np.quantile(ref, q))
            observed = int((tst > threshold).sum())
            expected = (1.0 - q) * len(tst)
            tail_rows.append({
                "predicted_class_stratum": cls,
                "component": signal,
                "OOF_reference_n": len(ref),
                "test_n": len(tst),
                "OOF_quantile": q,
                "OOF_threshold": threshold,
                "test_count_above_threshold": observed,
                "test_tail_rate": observed / len(tst),
                "expected_tail_count_if_all_known_and_matched": expected,
                "positive_excess_tail_diagnostic_only": max(0.0, observed - expected),
            })
tail_df = pd.DataFrame(tail_rows)
tail_path = os.path.join(OUT, "triple_v4_component_tail_by_predicted_class.csv")
tail_df.to_csv(tail_path, index=False)

# 4. Bulk quantiles vs OOF reference under assumed Unknown share pi.
# Scores are HIGHER = more OOD. The log10 transform keeps the near-zero confidence tail readable.
# Per-class tables are essential; ALL is confounded by predicted-class composition shifts.
P_VALUES = [0.05, 0.10, 0.25, 0.50]
PI_VALUES = [0.0, 0.30, 0.40]
OOD_SIGNALS = ["log10_inverse_confidence_ood", "knn_distance_ood"]
bulk_rows = []
implicit_rows = []
for cls in ["ALL"] + CLASS_NAMES:
    oref = oof if cls == "ALL" else oof[oof["predicted_class_reconstructed"] == cls]
    tref = test if cls == "ALL" else test[test["predicted_class_reconstructed"] == cls]
    for signal in OOD_SIGNALS:
        ref_values = oref[signal].to_numpy()
        test_values = tref[signal].to_numpy()
        bulk_rows.extend(quantile_summary(
            ref_values,
            test_values,
            P_VALUES,
            PI_VALUES,
            signal,
            cls,
        ))
        implicit_rows.extend(implicit_pi_summary(
            ref_values,
            test_values,
            P_VALUES,
            signal,
            cls,
        ))
bulk_df = pd.DataFrame(bulk_rows)
bulk_path = os.path.join(OUT, "triple_v4_bulk_quantile_prior_sensitivity.csv")
bulk_df.to_csv(bulk_path, index=False)

implicit_df = pd.DataFrame(implicit_rows)

# 4A. Confidence-floor audit by split and predicted-class stratum.
CONF_FLOOR_WARN_FRAC = 0.05
floor_rows = []
for split_name, frame in [("OOF", oof), ("TEST", test)]:
    for cls in ["ALL"] + CLASS_NAMES:
        sub = frame if cls == "ALL" else frame[frame["predicted_class_reconstructed"] == cls]
        if len(sub) == 0:
            continue
        hits = int(sub["confidence_floor_hit"].sum())
        frac = hits / len(sub)
        floor_rows.append({
            "split": split_name,
            "predicted_class_stratum": cls,
            "n_rows": len(sub),
            "n_confidence_floor_hits": hits,
            "confidence_floor_fraction": frac,
            "warn_gt_5pct": frac > CONF_FLOOR_WARN_FRAC,
        })
floor_df = pd.DataFrame(floor_rows)
floor_path = os.path.join(OUT, "triple_v4_confidence_floor_audit.csv")
floor_df.to_csv(floor_path, index=False)

# Add floor diagnostics and suppress interpretation of confidence pi at p<=0.25 where floor mass >5%.
for idx, row in implicit_df.iterrows():
    cls = row["predicted_class_stratum"]
    signal = row["signal_high_means_OOD"]
    p_quant = float(row["test_quantile_p"])
    if cls == "ALL":
        oof_floor = float(floor_df[(floor_df.split == "OOF") & (floor_df.predicted_class_stratum == "ALL")]["confidence_floor_fraction"].iloc[0])
        test_floor = float(floor_df[(floor_df.split == "TEST") & (floor_df.predicted_class_stratum == "ALL")]["confidence_floor_fraction"].iloc[0])
    else:
        oof_floor = float(floor_df[(floor_df.split == "OOF") & (floor_df.predicted_class_stratum == cls)]["confidence_floor_fraction"].iloc[0])
        test_floor = float(floor_df[(floor_df.split == "TEST") & (floor_df.predicted_class_stratum == cls)]["confidence_floor_fraction"].iloc[0])
    floor_warning = signal == "log10_inverse_confidence_ood" and (oof_floor > CONF_FLOOR_WARN_FRAC or test_floor > CONF_FLOOR_WARN_FRAC)
    ignored = bool(floor_warning and p_quant <= 0.25)
    implicit_df.loc[idx, "oof_confidence_floor_fraction"] = oof_floor
    implicit_df.loc[idx, "test_confidence_floor_fraction"] = test_floor
    implicit_df.loc[idx, "ignored_due_to_confidence_floor_rule"] = ignored
    implicit_df.loc[idx, "interpretation_status"] = (
        "IGNORE: confidence floor fraction >5% and p<=0.25"
        if ignored else "Diagnostic only; mixture/domain assumptions still apply"
    )

implicit_path = os.path.join(OUT, "triple_v4_implicit_pi_by_class.csv")
implicit_df.to_csv(implicit_path, index=False)

# Additional classwise distribution summary, without interpreting it as true Unknown prevalence.
class_rows = []
for cls in CLASS_NAMES:
    oo = oof[oof["predicted_class_reconstructed"] == cls]
    tt = test[test["predicted_class_reconstructed"] == cls]
    class_rows.append({
        "predicted_class": cls,
        "oof_count": len(oo),
        "oof_share": len(oo) / len(oof),
        "test_count": len(tt),
        "test_share": len(tt) / len(test),
        "oof_mean_confidence": oo["confidence"].mean(),
        "test_mean_confidence": tt["confidence"].mean(),
        "oof_median_confidence": oo["confidence"].median(),
        "test_median_confidence": tt["confidence"].median(),
        "oof_confidence_floor_fraction": oo["confidence_floor_hit"].mean(),
        "test_confidence_floor_fraction": tt["confidence_floor_hit"].mean(),
        "oof_median_log10_inverse_confidence_ood": oo["log10_inverse_confidence_ood"].median(),
        "test_median_log10_inverse_confidence_ood": tt["log10_inverse_confidence_ood"].median(),
        "oof_mean_knn_distance": oo["knn_distance_ood"].mean(),
        "test_mean_knn_distance": tt["knn_distance_ood"].mean(),
        "oof_median_knn_distance": oo["knn_distance_ood"].median(),
        "test_median_knn_distance": tt["knn_distance_ood"].median(),
    })
class_summary = pd.DataFrame(class_rows)
class_summary_path = os.path.join(OUT, "triple_v4_component_distribution_by_class.csv")
class_summary.to_csv(class_summary_path, index=False)

# 4B. Cluster bootstrap OOF source_groups and row bootstrap TEST, by predicted class.
# This estimates uncertainty for class-conditional pi(p); it does not correct domain shift.
N_BOOTSTRAP = 500
BOOTSTRAP_SEED = 20261020
boot_rng = np.random.default_rng(BOOTSTRAP_SEED)
P_BOOT = [0.05, 0.10, 0.25, 0.50]
BOOT_SIGNALS = ["log10_inverse_confidence_ood", "knn_distance_ood"]
boot_curve_frames = []
boot_distributions = {}
for signal in BOOT_SIGNALS:
    curve_df, class_boot = bootstrap_pi_by_class(
        oof, test, signal, P_BOOT, N_BOOTSTRAP, boot_rng
    )
    boot_curve_frames.append(curve_df)
    for cls, boot_values in class_boot.items():
        boot_distributions[(signal, cls)] = boot_values
pi_boot_df = pd.concat(boot_curve_frames, ignore_index=True) if boot_curve_frames else pd.DataFrame()
pi_boot_path = os.path.join(OUT, "triple_v4_implicit_pi_bootstrap_ci_by_class.csv")
pi_boot_df.to_csv(pi_boot_path, index=False)

# 4C. N_imp(p) = sum_k n_k * max(0, pi_k(p)); primary interpretation is kNN.
# CI propagates class-conditional group-bootstrap OOF and row-bootstrap TEST uncertainty.
nimp_rows = []
n_test_by_class = {
    cls: int((test["predicted_class_reconstructed"] == cls).sum())
    for cls in CLASS_NAMES
}
for signal in BOOT_SIGNALS:
    for j, p_quant in enumerate(P_BOOT):
        class_point = {}
        class_boot_vals = {}
        class_valid = True
        class_floor_ignored = False
        for cls in CLASS_NAMES:
            row = pi_boot_df[(pi_boot_df["signal"] == signal) & (pi_boot_df["predicted_class"] == cls) & (pi_boot_df["p"] == p_quant)]
            if len(row) != 1:
                class_valid = False
                break
            r = row.iloc[0]
            class_point[cls] = float(r["pi_point_raw_midrank"]) if pd.notna(r["pi_point_raw_midrank"]) else np.nan
            boot = boot_distributions.get((signal, cls))
            class_boot_vals[cls] = boot[:, j] if boot is not None else np.full(N_BOOTSTRAP, np.nan)
            class_floor_ignored = class_floor_ignored or bool(r["ignored_due_to_confidence_floor_rule"])
        if not class_valid:
            continue
        point_components = [n_test_by_class[cls] * max(0.0, class_point[cls]) for cls in CLASS_NAMES if np.isfinite(class_point[cls])]
        nimp_point = float(np.sum(point_components)) if len(point_components) == len(CLASS_NAMES) else np.nan
        boot_nimp = np.full(N_BOOTSTRAP, np.nan, dtype=float)
        for b in range(N_BOOTSTRAP):
            contributions = []
            for cls in CLASS_NAMES:
                pi_b = class_boot_vals[cls][b]
                if not np.isfinite(pi_b):
                    contributions = []
                    break
                contributions.append(n_test_by_class[cls] * max(0.0, float(pi_b)))
            if len(contributions) == len(CLASS_NAMES):
                boot_nimp[b] = float(np.sum(contributions))
        valid_boot = boot_nimp[np.isfinite(boot_nimp)]
        if len(valid_boot) >= max(30, 0.5 * N_BOOTSTRAP):
            nimp_low, nimp_high = np.quantile(valid_boot, [0.025, 0.975]).tolist()
        else:
            nimp_low, nimp_high = np.nan, np.nan
        confidence_floor_invalid = signal == "log10_inverse_confidence_ood" and class_floor_ignored
        nimp_rows.append({
            "signal": signal,
            "p": p_quant,
            "N_imp_raw_pi_positive_parts": nimp_point,
            "N_imp_bootstrap_ci95_low": nimp_low,
            "N_imp_bootstrap_ci95_high": nimp_high,
            "bootstrap_reps_valid": int(len(valid_boot)),
            "all_class_pi_available": bool(np.isfinite(list(class_point.values())).all()),
            "ignored_due_to_confidence_floor_rule": bool(confidence_floor_invalid),
            "N_imp_for_decision": np.nan if confidence_floor_invalid else nimp_point,
            "test_predicted_class_counts": json.dumps(n_test_by_class, ensure_ascii=False),
            "interpretation_note": "Diagnostic only; mixture assumptions and OOF/test comparability may fail. For kNN, inspect p=0.25 and 0.50 first.",
        })
nimp_df = pd.DataFrame(nimp_rows)
nimp_path = os.path.join(OUT, "triple_v4_implicit_unknown_count_by_knn_and_confidence.csv")
nimp_df.to_csv(nimp_path, index=False)

# 5. Confirm test probability averaging scheme from the source inference pipeline.
# Inspected source code: ResNet 5s/8s probabilities are averaged over 4 fold checkpoints per crop;
# EfficientNet-B2 averages each fold's test probabilities; ConvNeXt-v4 averages fold_probs/N_FOLDS;
# final test probabilities are a 55:5:40 blend of those already fold-averaged model outputs.
fold_note = {
    "resnet5_and_8_seconds": "mean of 4 fold checkpoints per crop, then TTA crop aggregation",
    "efficientnet_b2": "fold probabilities accumulated as fold_probs / number of fold checkpoints (4 folds)",
    "convnext_v4": "fold probabilities accumulated as fold_probs / N_FOLDS (4 folds)",
    "final_test_ensemble": "55% ResNet50 + 5% EfficientNet-B2 + 40% ConvNeXt-v4; component probabilities already fold-averaged",
    "implication": "Test confidence is based on fold-averaged component predictions; it may be smoother than individual-fold OOF predictions. Treat OOF-vs-test confidence shifts cautiously.",
    "confidence_floor_rule": "Compute fraction with 1-confidence <= CONF_EPS in each class/split. If either OOF or TEST exceeds 5%, ignore confidence-derived pi(p) at p <= 0.25.",
    "implicit_pi_uncertainty": "pi(p) uses empirical mid-rank F_OOF and 95% percentile intervals from OOF source_group cluster bootstrap plus within-class test-row bootstrap.",
}
with open(os.path.join(OUT, "triple_v4_fold_averaging_note.json"), "w", encoding="utf-8") as f:
    json.dump(fold_note, f, ensure_ascii=False, indent=2)

# 6. Plot class-conditional implicit pi(p) with bootstrap intervals; no ALL-stratum plot.
fig_path = os.path.join(OUT, "triple_v4_implicit_pi_by_class_with_ci.png")
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True)
colors = {"Bondol Jawa": None, "Cendrawasih": None, "Kutilang": None}
for ax, signal, title in zip(
    axes,
    ["log10_inverse_confidence_ood", "knn_distance_ood"],
    ["Confidence OOD score (log10(1-confidence))", "BEATs kNN distance (primary signal)"],
):
    view = pi_boot_df[pi_boot_df["signal"] == signal]
    for cls in CLASS_NAMES:
        sub = view[view["predicted_class"] == cls].sort_values("p")
        if sub.empty:
            continue
        # For confidence, honor the floor rule: do not plot p<=0.25 when >5% hit the floor.
        good = sub[~sub["ignored_due_to_confidence_floor_rule"]].copy()
        if not good.empty:
            line, = ax.plot(good["p"], good["pi_point_raw_midrank"], marker="o", label=cls)
            ax.fill_between(
                good["p"].to_numpy(dtype=float),
                good["pi_bootstrap_ci95_low"].to_numpy(dtype=float),
                good["pi_bootstrap_ci95_high"].to_numpy(dtype=float),
                color=line.get_color(), alpha=0.16,
            )
    ax.axhline(0.0, linestyle="--", linewidth=1, color="gray")
    ax.axhline(0.3, linestyle=":", linewidth=1, color="gray", alpha=0.8)
    ax.axhline(0.4, linestyle=":", linewidth=1, color="gray", alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Test lower-tail quantile p")
    ax.set_ylabel("Raw implicit pi(p), 95% bootstrap CI")
    ax.legend()
    ax.grid(alpha=0.2)
fig.suptitle("Class-conditional implicit Unknown fraction diagnostic (not a true prevalence estimate)")
fig.tight_layout()
fig.savefig(fig_path, dpi=160)
plt.close(fig)

# Plot N_imp(p) for kNN as the main evidence; confidence is secondary and may be suppressed by floor rule.
nimp_fig_path = os.path.join(OUT, "triple_v4_implicit_unknown_count_knn_ci.png")
fig, ax = plt.subplots(figsize=(8, 5))
for signal, label in [
    ("knn_distance_ood", "BEATs kNN (primary)"),
    ("log10_inverse_confidence_ood", "Confidence (secondary; may be floor-invalid)"),
]:
    sub = nimp_df[nimp_df["signal"] == signal].sort_values("p")
    if sub.empty:
        continue
    if signal == "log10_inverse_confidence_ood":
        sub = sub[~sub["ignored_due_to_confidence_floor_rule"]]
    if sub.empty:
        continue
    line, = ax.plot(sub["p"], sub["N_imp_for_decision"], marker="o", label=label)
    ax.fill_between(
        sub["p"].to_numpy(dtype=float),
        sub["N_imp_bootstrap_ci95_low"].to_numpy(dtype=float),
        sub["N_imp_bootstrap_ci95_high"].to_numpy(dtype=float),
        color=line.get_color(), alpha=0.16,
    )
ax.axhline(245, linestyle="--", color="black", linewidth=1, label="Champion U=245")
ax.axhline(330, linestyle=":", color="gray", linewidth=1, label="Approx. U=330 (40% of test)")
ax.set_xlabel("Test lower-tail quantile p")
ax.set_ylabel("N_imp(p) = sum class test count × max(0, pi_k(p))")
ax.set_title("Implicit unknown-count diagnostic with bootstrap CI")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(nimp_fig_path, dpi=160)
plt.close(fig)

print("\n" + "=" * 108)
print("CHECK 1 — COMPONENT TAILS BY PREDICTED CLASS (LOG-CONFIDENCE AND kNN)")
print("Scores are oriented higher = more OOD. Excess is diagnostic, not a true-Unknown estimate.")
print("=" * 108)
print(tail_df[tail_df["predicted_class_stratum"].isin(["ALL"] + CLASS_NAMES)].round(4).to_string(index=False))

print("\n" + "=" * 108)
print("CHECK 2A — ALL-TEST BULK QUANTILES (MIXED-CLASS VIEW; INTERPRET CAUTIOUSLY)")
print("Signals: log10(1-confidence) and kNN distance; higher = more OOD.")
print("pi is hypothetical; quantiles compare test p with OOF p/(1-pi).")
print("=" * 108)
print(bulk_df[bulk_df["predicted_class_stratum"] == "ALL"].round(4).to_string(index=False))

print("\n" + "=" * 108)
print("CHECK 2B — CLASS-CONDITIONAL BULK QUANTILES (PRIORITIZE kNN; LOG-CONFIDENCE AS SUPPORT)")
print("OOF and test are stratified by each split's reconstructed predicted class.")
print("Per-class pi scenarios are sensitivity comparisons, not class-specific truth.")
print("=" * 108)
class_bulk = bulk_df[bulk_df["predicted_class_stratum"] != "ALL"].copy()
print(class_bulk.sort_values(
    ["signal_high_means_OOD", "predicted_class_stratum", "assumed_unknown_fraction_pi", "test_quantile_p"]
).round(4).to_string(index=False))

print("\n" + "=" * 108)
print("CHECK 2C — CONFIDENCE FLOOR AUDIT")
print(f"Floor definition: raw (1-confidence) <= CONF_EPS={CONF_EPS:g}; pi confidence at p<=0.25 is ignored when either split exceeds 5% floor mass.")
print("=" * 108)
print(floor_df.round(4).to_string(index=False))

print("\n" + "=" * 108)
print("CHECK 2D — CLASS-CONDITIONAL IMPLICIT pi(p), MID-RANK CDF (RAW, NOT CLIPPED)")
print("F_OOF uses (count less than q + 0.5 * count tied at q) / n. Raw pi can be negative or exceed plausible bounds under domain shift.")
print("Confidence pi is ignored at p<=0.25 in any class/split with >5% floor ties.")
print("=" * 108)
print(implicit_df[implicit_df["predicted_class_stratum"] != "ALL"].sort_values(
    ["signal_high_means_OOD", "predicted_class_stratum", "test_quantile_p"]
).round(4).to_string(index=False))

print("\n" + "=" * 108)
print("CHECK 2E — BOOTSTRAP pi(p) 95% INTERVALS BY CLASS")
print(f"OOF: source_group cluster bootstrap; TEST: within-predicted-class row bootstrap; B={N_BOOTSTRAP}.")
print("Intervals quantify sampling uncertainty under this setup; they do not solve domain shift or LOCO mismatch.")
print("=" * 108)
print(pi_boot_df.sort_values(["signal", "predicted_class", "p"]).round(4).to_string(index=False))

print("\n" + "=" * 108)
print("CHECK 2F — N_imp(p) = sum_k n_k * max(0, pi_k(p))")
print("Prioritize kNN at p=0.25 and p=0.50. This is only a mixture-model diagnostic, not a guaranteed lower bound under arbitrary shift.")
print("=" * 108)
print(nimp_df.sort_values(["signal", "p"]).round(3).to_string(index=False))

print("\nClass-conditional summary (distribution diagnostic only):")
print(class_summary.round(4).to_string(index=False))

print("\n" + "=" * 108)
print("CHECK 3 — TEST PROBABILITY FOLD AVERAGING")
print("=" * 108)
print(json.dumps(fold_note, indent=2, ensure_ascii=False))
print("\nOutputs saved:")
for p in [tail_path, bulk_path, implicit_path, floor_path, pi_boot_path, nimp_path, class_summary_path, fig_path, nimp_fig_path, os.path.join(OUT, "triple_v4_fold_averaging_note.json")]:
    print(" -", p)
print("\nNo submission CSV created and nothing was submitted.")


SCHEMATICS NPC DSC 2026 — PRE-SUBMIT SENSITIVITY CHECKS
Offline diagnostics only; no submission is created or sent.

RECONSTRUCTION / ALIGNMENT
  OOF blend shape: (1826, 3)
  Test blend shape: (814, 3)
  OOF confidence max abs difference: 1.110e-16
  Test confidence max abs difference: 4.933e-08
  Test predicted-class agreement with score table: 1.000000
  Reconstructed OOF predicted class from probability arrays; confirm OOF row order via F1/accuracy below.
  Reconstructed OOF Macro F1: 0.942038
  Reconstructed OOF accuracy: 0.984118

CHAMPION RECONSTRUCTION
  File: /kaggle/working/submission_triple_v4_conf_knn50_u245.csv
  Champion Unknown count: 245
  Overlap with reconstructed 245: 245/245
  Exact set match: True

CHECK 1 — COMPONENT TAILS BY PREDICTED CLASS (LOG-CONFIDENCE AND kNN)
Scores are oriented higher = more OOD. Excess is diagnostic, not a true-Unknown estimate.
predicted_class_stratum                    component  OOF_reference_n  test_n  OOF_quantile  OOF_threshold  test

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

# Uji jumlah tetangga; k=5 adalah baseline awal.
KNN_VALUES = [3, 5, 10, 15, 20]
sweep_results = []

for k in KNN_VALUES:
    KNN_K = k

    # LOCO memakai fold yang sudah ditetapkan dan embedding BEATs.
    detail_k, _, summary_k = run_loco(
        train_meta, embeddings, label_to_idx
    )

    class_results = []

    for heldout_class in LABELS:
        part = detail_k[
            detail_k["heldout_class"] == heldout_class
        ].copy()

        y_unknown = part["is_pseudo_unknown"].astype(int).to_numpy()

        # Skor lebih tinggi berarti lebih menyerupai Unknown.
        confidence_ood = (
            1.0 - part["max_probability"].to_numpy()
        )
        knn_ood = part["beats_knn_distance"].to_numpy()

        # Rank fusion 50:50, dihitung per skenario LOCO.
        conf_rank = pd.Series(confidence_ood).rank(
            pct=True, method="average"
        ).to_numpy()

        knn_rank = pd.Series(knn_ood).rank(
            pct=True, method="average"
        ).to_numpy()

        fused_score = 0.5 * conf_rank + 0.5 * knn_rank

        auc_knn = roc_auc_score(y_unknown, knn_ood)
        auc_fused = roc_auc_score(y_unknown, fused_score)
        ap_fused = average_precision_score(
            y_unknown, fused_score
        )

        class_results.append({
            "heldout_class": heldout_class,
            "auc_knn": auc_knn,
            "auc_fused_50_50": auc_fused,
            "ap_fused_50_50": ap_fused,
        })

    class_df = pd.DataFrame(class_results)

    row = {
        "k": k,
        "macro_auc_knn": class_df["auc_knn"].mean(),
        "macro_auc_fused_50_50":
            class_df["auc_fused_50_50"].mean(),
        "macro_ap_fused_50_50":
            class_df["ap_fused_50_50"].mean(),
    }

    for _, r in class_df.iterrows():
        cls_key = r["heldout_class"].replace(" ", "_").lower()
        row[f"auc_fused_{cls_key}"] = r["auc_fused_50_50"]

    sweep_results.append(row)

    print(
        f"k={k:2d} | "
        f"Macro AUC kNN={row['macro_auc_knn']:.5f} | "
        f"Macro AUC fusion={row['macro_auc_fused_50_50']:.5f}"
    )

sweep_df = pd.DataFrame(sweep_results).sort_values(
    "macro_auc_fused_50_50", ascending=False
)

sweep_path = "/kaggle/working/beats_knn_k_sweep_loco.csv"
sweep_df.to_csv(sweep_path, index=False)

print("\nHASIL AKHIR — URUTAN TERBAIK")
display(sweep_df.round(5))
print("Tersimpan:", sweep_path)

NameError: name 'run_loco' is not defined

In [7]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, average_precision_score

# ============================================================
# 1. KONFIGURASI
# ============================================================
ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

LABELS = ["Bondol Jawa", "Cendrawasih", "Kutilang"]
K_VALUES = [3, 5, 10, 15, 20]
N_FOLDS = 4
SEED = 16
MAX_K = max(K_VALUES)

# ============================================================
# 2. CARI FILE EMBEDDING DAN FOLD YANG SUDAH ADA
# ============================================================
def find_artifact(filename):
    candidates = [
        os.path.join(OUT, filename),
        os.path.join(
            "/kaggle/input/notebooks/radithiafatir/rill-versi-baru",
            filename
        ),
        os.path.join(
            "/kaggle/input/notebooks/radithiafatir/testttrisan",
            filename
        ),
    ]

    for path in candidates:
        if os.path.isfile(path):
            return path

    for base in [OUT, "/kaggle/input"]:
        hits = glob.glob(
            os.path.join(base, "**", filename),
            recursive=True
        )
        if hits:
            return hits[0]

    raise FileNotFoundError(
        f"File tidak ditemukan: {filename}"
    )


train_meta_path = os.path.join(ROOT, "train", "metadata.csv")
fold_path = find_artifact("bird_group_folds_seed16.csv")
emb_path = find_artifact(
    "beats_iter3plus_as2m_8s_train_embeddings.npy"
)

assert os.path.isfile(train_meta_path), train_meta_path

train_meta = pd.read_csv(train_meta_path)
fold_df = pd.read_csv(fold_path)
embeddings = np.load(emb_path, mmap_mode="r")

train_meta["file_id"] = train_meta["file_id"].astype(str)
fold_df["file_id"] = fold_df["file_id"].astype(str)

# Pasangkan fold berdasarkan file_id, bukan asumsi urutan.
train_meta = train_meta.merge(
    fold_df[["file_id", "fold"]],
    on="file_id",
    how="left",
    validate="one_to_one"
)

if train_meta["fold"].isna().any():
    raise ValueError("Ada sampel tanpa fold.")

train_meta["fold"] = train_meta["fold"].astype(int)

y = train_meta["label"].map(
    {label: i for i, label in enumerate(LABELS)}
).to_numpy(dtype=int)

if np.isnan(y).any():
    raise ValueError("Ada label yang tidak cocok dengan LABELS.")

if embeddings.shape[0] != len(train_meta):
    raise ValueError(
        f"Urutan/jumlah embedding tidak cocok: "
        f"{embeddings.shape[0]} vs {len(train_meta)}"
    )

folds = train_meta["fold"].to_numpy()

# Pastikan source_group tidak menyeberangi fold.
group_leakage = (
    train_meta.groupby("source_group")["fold"].nunique() > 1
).sum()

if group_leakage:
    raise ValueError(
        f"Terdeteksi {group_leakage} source_group bocor antar-fold."
    )

print("Train:", len(train_meta))
print("Embedding:", embeddings.shape)
print("Fold counts:", train_meta["fold"].value_counts().sort_index().to_dict())
print("Source-group leakage:", int(group_leakage))
print("Embedding file:", emb_path)

# ============================================================
# 3. LOCO: UJI NILAI K TANPA RETRAINING MODEL AUDIO
# ============================================================
x = np.asarray(embeddings, dtype=np.float32)
records_by_k = {k: [] for k in K_VALUES}

for heldout in range(len(LABELS)):
    known_classes = [
        c for c in range(len(LABELS)) if c != heldout
    ]

    print(f"\nLOCO pseudo-Unknown: {LABELS[heldout]}")

    for fold in range(N_FOLDS):
        # Fold validasi tidak ikut training.
        # Kelas yang sedang ditahan juga tidak ikut training.
        tr_idx = np.flatnonzero(
            (folds != fold) & (y != heldout)
        )
        va_idx = np.flatnonzero(folds == fold)

        if set(np.unique(y[tr_idx])) != set(known_classes):
            raise ValueError(
                f"Fold {fold}, heldout={LABELS[heldout]}: "
                "kelas training tidak lengkap."
            )

        xtr = x[tr_idx]
        xva = x[va_idx]
        ytr = y[tr_idx]

        # Confidence baseline: logistic regression dua kelas.
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.03,
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED
            )
        )
        clf.fit(xtr, ytr)
        pva = clf.predict_proba(xva)
        confidence_ood = 1.0 - pva.max(axis=1)

        # Cari tetangga sampai MAX_K sekali saja.
        xtr_norm = normalize(xtr, norm="l2")
        xva_norm = normalize(xva, norm="l2")

        knn = NearestNeighbors(
            n_neighbors=min(MAX_K, len(tr_idx)),
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
        )
        knn.fit(xtr_norm)
        distances, _ = knn.kneighbors(xva_norm)

        y_unknown = (y[va_idx] == heldout).astype(int)

        for k in K_VALUES:
            knn_distance = distances[:, :k].mean(axis=1)

            for j in range(len(va_idx)):
                records_by_k[k].append({
                    "heldout_class": LABELS[heldout],
                    "fold": fold,
                    "is_pseudo_unknown": int(y_unknown[j]),
                    "confidence_ood": float(confidence_ood[j]),
                    "knn_distance": float(knn_distance[j]),
                })

        print(
            f"  Fold {fold}: train={len(tr_idx)}, "
            f"validasi={len(va_idx)}"
        )

# ============================================================
# 4. EVALUASI RANK FUSION 50:50
# ============================================================
results = []

for k in K_VALUES:
    detail = pd.DataFrame(records_by_k[k])
    class_rows = []

    for class_name in LABELS:
        part = detail[
            detail["heldout_class"] == class_name
        ].copy()

        target = part["is_pseudo_unknown"].to_numpy()

        # Ranking dihitung terpisah untuk setiap skenario LOCO.
        conf_rank = part["confidence_ood"].rank(
            pct=True, method="average"
        ).to_numpy()

        knn_rank = part["knn_distance"].rank(
            pct=True, method="average"
        ).to_numpy()

        fused = 0.5 * conf_rank + 0.5 * knn_rank

        class_rows.append({
            "class": class_name,
            "auc_knn": roc_auc_score(
                target, part["knn_distance"]
            ),
            "auc_fusion": roc_auc_score(target, fused),
            "ap_fusion": average_precision_score(
                target, fused
            ),
        })

    per_class = pd.DataFrame(class_rows)

    row = {
        "k": k,
        "macro_auc_knn": per_class["auc_knn"].mean(),
        "macro_auc_fusion": per_class["auc_fusion"].mean(),
        "macro_ap_fusion": per_class["ap_fusion"].mean(),
    }

    for _, r in per_class.iterrows():
        key = r["class"].replace(" ", "_").lower()
        row[f"auc_fusion_{key}"] = r["auc_fusion"]

    results.append(row)

sweep_df = pd.DataFrame(results).sort_values(
    "macro_auc_fusion", ascending=False
).reset_index(drop=True)

save_path = os.path.join(
    OUT, "beats_knn_k_sweep_loco.csv"
)
sweep_df.to_csv(save_path, index=False)

print("\n========== HASIL KNN K SWEEP ==========")
display(sweep_df.round(5))
print("Hasil tersimpan:", save_path)

Train: 1826
Embedding: (1826, 768)
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}
Source-group leakage: 0
Embedding file: /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy

LOCO pseudo-Unknown: Bondol Jawa
  Fold 0: train=535, validasi=284
  Fold 1: train=521, validasi=495
  Fold 2: train=543, validasi=553
  Fold 3: train=507, validasi=494

LOCO pseudo-Unknown: Cendrawasih
  Fold 0: train=1494, validasi=284
  Fold 1: train=1251, validasi=495
  Fold 2: train=1206, validasi=553
  Fold 3: train=1278, validasi=494

LOCO pseudo-Unknown: Kutilang
  Fold 0: train=1055, validasi=284
  Fold 1: train=890, validasi=495
  Fold 2: train=797, validasi=553
  Fold 3: train=879, validasi=494

========== HASIL KNN K SWEEP ==========


,k,macro_auc_knn,macro_auc_fusion,macro_ap_fusion,auc_fusion_bondol_jawa,auc_fusion_cendrawasih,auc_fusion_kutilang
0,3,0.72236,0.78556,0.53097,0.78187,0.87230,0.70250
1,5,0.71845,0.78356,0.52697,0.77834,0.87506,0.69727
2,10,0.71287,0.78011,0.52247,0.77226,0.87780,0.69027
3,15,0.70933,0.77793,0.51946,0.76813,0.87841,0.68727
4,20,0.70740,0.77676,0.51777,0.76571,0.87925,0.68532


Hasil tersimpan: /kaggle/working/beats_knn_k_sweep_loco.csv


In [8]:
# ============================================================
# TRIPLE V4 + BEATs kNN K=3 — KANDIDAT SUBMISSION
# Tetap: confidence/kNN = 50:50, jumlah Unknown = 245
# Champion lama tidak akan ditimpa.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# ============================================================
# 1. KONFIGURASI
# ============================================================

DATA_ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

CLASS_NAMES = [
    "Bondol Jawa",
    "Cendrawasih",
    "Kutilang"
]

N_UNKNOWN = 245
K_BASELINE = 5
K_CANDIDATE = 3

CHAMPION_NAME = "submission_triple_v4_conf_knn50_u245.csv"
CANDIDATE_NAME = "submission_triple_v4_conf_knn50_k3_u245.csv"

# ============================================================
# 2. MENCARI FILE YANG SUDAH TERSIMPAN
# ============================================================

def find_file(filename, required=True):
    # Prioritaskan working agar memakai artefak yang sudah disalin.
    preferred = [
        os.path.join(OUT, filename),
        os.path.join(
            "/kaggle/input/notebooks/radithiafatir/testttrisan",
            filename
        ),
        os.path.join(
            "/kaggle/input/notebooks/radithiafatir/rill-versi-baru",
            filename
        ),
        os.path.join(DATA_ROOT, filename),
    ]

    for path in preferred:
        if os.path.isfile(path):
            return path

    # Fallback untuk direktori artefak lain.
    for root in [OUT, "/kaggle/input"]:
        hits = glob.glob(
            os.path.join(root, "**", filename),
            recursive=True
        )
        if hits:
            hits.sort(key=lambda p: (len(p), p))
            return hits[0]

    if required:
        raise FileNotFoundError(
            f"File tidak ditemukan: {filename}\n"
            "Pastikan dataset dan notebook artefak sudah terpasang."
        )

    return None


# ============================================================
# 3. MEMUAT SAMPLE SUBMISSION, PROBABILITAS, DAN EMBEDDING
# ============================================================

sample_path = find_file("sample_submission.csv")

prob_path = find_file(
    "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"
)

train_emb_path = find_file(
    "beats_iter3plus_as2m_8s_train_embeddings.npy"
)

test_emb_path = find_file(
    "beats_iter3plus_as2m_8s_test_embeddings.npy"
)

champion_path = find_file(CHAMPION_NAME, required=False)

sample_sub = pd.read_csv(sample_path)

# Deteksi nama kolom submission.
if "file_id" in sample_sub.columns:
    ID_COL = "file_id"
else:
    ID_COL = sample_sub.columns[0]

if "label" in sample_sub.columns:
    LABEL_COL = "label"
else:
    other_cols = [
        c for c in sample_sub.columns if c != ID_COL
    ]
    if len(other_cols) != 1:
        raise ValueError(
            f"Tidak dapat menentukan kolom label: "
            f"{sample_sub.columns.tolist()}"
        )
    LABEL_COL = other_cols[0]

# Muat array yang sudah ada. Tidak ada training ulang.
test_probs = np.load(prob_path).astype(np.float32)
train_emb = np.load(train_emb_path).astype(np.float32)
test_emb = np.load(test_emb_path).astype(np.float32)

# ============================================================
# 4. VALIDASI DIMENSI DAN INPUT
# ============================================================

print("=" * 65)
print("PEMERIKSAAN INPUT")
print("=" * 65)

print("Sample submission :", sample_path)
print("Triple V4 probs   :", prob_path)
print("Train embeddings  :", train_emb_path)
print("Test embeddings   :", test_emb_path)
print("Champion ditemukan:", champion_path is not None)

print("\nShape sample :", sample_sub.shape)
print("Shape probs  :", test_probs.shape)
print("Shape train  :", train_emb.shape)
print("Shape test   :", test_emb.shape)

assert len(sample_sub) == 814, (
    f"Jumlah sample tidak sesuai: {len(sample_sub)}"
)

assert sample_sub[ID_COL].is_unique, (
    "Ada ID duplikat pada sample submission."
)

assert test_probs.shape == (
    len(sample_sub), len(CLASS_NAMES)
), "Shape probabilitas tidak sesuai."

assert train_emb.ndim == 2
assert test_emb.ndim == 2

assert train_emb.shape[1] == test_emb.shape[1], (
    "Dimensi train dan test embedding berbeda."
)

assert test_emb.shape[0] == len(sample_sub), (
    "Jumlah test embedding tidak sama dengan sample submission."
)

assert train_emb.shape[0] == 1826, (
    f"Jumlah train embedding tidak sesuai: {train_emb.shape[0]}"
)

assert np.isfinite(test_probs).all()
assert np.isfinite(train_emb).all()
assert np.isfinite(test_emb).all()

assert np.all(test_probs >= -1e-5), (
    "Ada probabilitas negatif."
)

assert np.allclose(
    test_probs.sum(axis=1), 1.0, atol=1e-3
), "Probabilitas per baris tidak berjumlah sekitar 1."

print("\nInput lolos pemeriksaan dimensi.")

# ============================================================
# 5. HITUNG JARAK KE 5 TETANGGA TERDEKAT SEKALI SAJA
# ============================================================
# Normalisasi L2 dan cosine distance, konsisten dengan
# pendekatan kNN yang digunakan pada eksperimen LOCO.
#
# Jarak 3 tetangga pertama dipakai untuk kandidat k=3.
# Jarak 5 tetangga pertama dipakai sebagai pemeriksaan baseline.

print("\n" + "=" * 65)
print("MENGHITUNG BEATS KNN")
print("=" * 65)

train_norm = normalize(train_emb, norm="l2")
test_norm = normalize(test_emb, norm="l2")

knn = NearestNeighbors(
    n_neighbors=K_BASELINE,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

knn.fit(train_norm)

distances, neighbor_indices = knn.kneighbors(test_norm)

assert distances.shape == (len(sample_sub), K_BASELINE)

# Jarak rata-rata untuk k=3 dan k=5.
knn_dist_k3 = distances[:, :K_CANDIDATE].mean(axis=1)
knn_dist_k5 = distances[:, :K_BASELINE].mean(axis=1)

print("Jarak k=3 selesai.")
print("Jarak k=5 selesai.")
print("Jumlah sampel test:", len(knn_dist_k3))

# ============================================================
# 6. RANK FUSION 50:50
# ============================================================
# Semakin rendah confidence classifier, semakin besar skor OOD.
# Semakin besar jarak kNN, semakin besar skor OOD.
#
# Keduanya dibuat menjadi percentile rank di antara test,
# kemudian digabung dengan bobot 50:50.

confidence_ood = 1.0 - test_probs.max(axis=1)

confidence_rank = pd.Series(
    confidence_ood
).rank(
    pct=True,
    method="average"
).to_numpy()

def make_submission_for_k(knn_dist, k_value):
    knn_rank = pd.Series(
        knn_dist
    ).rank(
        pct=True,
        method="average"
    ).to_numpy()

    fusion_score = (
        0.5 * confidence_rank
        + 0.5 * knn_rank
    )

    # Pilih tepat 245 skor OOD tertinggi.
    unknown_idx = np.argsort(
        -fusion_score,
        kind="stable"
    )[:N_UNKNOWN]

    # Klasifikasi awal berdasarkan Triple V4.
    pred_idx = test_probs.argmax(axis=1)
    pred_labels = np.array(CLASS_NAMES, dtype=object)[pred_idx]

    # Override 245 sampel dengan skor OOD tertinggi.
    pred_labels[unknown_idx] = "Unknown"

    result = sample_sub.copy()
    result[LABEL_COL] = pred_labels

    unknown_ids = set(
        result.iloc[unknown_idx][ID_COL].astype(str)
    )

    return result, fusion_score, unknown_ids


# ============================================================
# 7. REKONSTRUKSI BASELINE K=5 DAN KANDIDAT K=3
# ============================================================

baseline_k5, score_k5, unknown_k5 = make_submission_for_k(
    knn_dist_k5, K_BASELINE
)

candidate_k3, score_k3, unknown_k3 = make_submission_for_k(
    knn_dist_k3, K_CANDIDATE
)

# ============================================================
# 8. PERIKSA APAKAH K=5 SAMA DENGAN CHAMPION ASLI
# ============================================================

print("\n" + "=" * 65)
print("VALIDASI TERHADAP CHAMPION LAMA")
print("=" * 65)

baseline_match = None

if champion_path is not None:
    champion = pd.read_csv(champion_path)

    if ID_COL not in champion.columns:
        raise ValueError(
            "Kolom ID champion tidak cocok: "
            f"{champion.columns.tolist()}"
        )

    if LABEL_COL not in champion.columns:
        raise ValueError(
            "Kolom label champion tidak cocok: "
            f"{champion.columns.tolist()}"
        )

    assert len(champion) == len(sample_sub), (
        "Jumlah baris champion tidak cocok."
    )

    champion_ids = set(champion[ID_COL].astype(str))
    assert champion_ids == set(sample_sub[ID_COL].astype(str)), (
        "ID champion tidak cocok dengan sample submission."
    )

    champion_unknown = set(
        champion.loc[
            champion[LABEL_COL] == "Unknown", ID_COL
        ].astype(str)
    )

    overlap = len(champion_unknown & unknown_k5)

    champion_map = (
        champion.assign(
            __id=champion[ID_COL].astype(str)
        )
        .set_index("__id")[LABEL_COL]
    )

    baseline_map = pd.Series(
        baseline_k5[LABEL_COL].to_numpy(),
        index=baseline_k5[ID_COL].astype(str)
    )

    champion_map = champion_map.reindex(baseline_map.index)

    n_label_mismatch = int(
        (champion_map.to_numpy() != baseline_map.to_numpy()).sum()
    )

    baseline_match = (n_label_mismatch == 0)

    print("Unknown champion           :", len(champion_unknown))
    print("Unknown hasil rekonstruksi :", len(unknown_k5))
    print("ID Unknown yang sama       :", overlap, "/", N_UNKNOWN)
    print("Perbedaan seluruh label k=5:", n_label_mismatch, "/", len(sample_sub))

    if baseline_match:
        print("\nVERIFIKASI LOLOS: k=5 mereproduksi champion.")
    else:
        print(
            "\nPERINGATAN: k=5 belum sama persis dengan champion."
        )
        print(
            "Jangan submit kandidat dulu. Kirim output pemeriksaan "
            "ini agar perbedaan pipeline bisa ditelusuri."
        )
else:
    print("File champion lama tidak ditemukan di lokasi yang dicari.")
    print(
        "Kandidat akan dibuat, tetapi baseline k=5 belum dapat "
        "diverifikasi terhadap file champion."
    )

# ============================================================
# 9. SIMPAN KANDIDAT K=3 — TIDAK MENIMPA CHAMPION
# ============================================================

candidate_path = os.path.join(OUT, CANDIDATE_NAME)

candidate_k3.to_csv(candidate_path, index=False)

print("\n" + "=" * 65)
print("KANDIDAT K=3")
print("=" * 65)

print("File kandidat:", candidate_path)
print("File tersimpan:", os.path.exists(candidate_path))
print("Shape:", candidate_k3.shape)

print("\nDistribusi label kandidat:")
print(candidate_k3[LABEL_COL].value_counts())

assert len(candidate_k3) == 814
assert candidate_k3[ID_COL].is_unique
assert (candidate_k3[LABEL_COL] == "Unknown").sum() == N_UNKNOWN

print("\nJumlah Unknown:", int(
    (candidate_k3[LABEL_COL] == "Unknown").sum()
))

if baseline_match is True:
    print(
        "\nSTATUS: Baseline k=5 cocok dengan champion. "
        "Kandidat k=3 siap diperiksa sebelum submission."
    )
else:
    print(
        "\nSTATUS: Kandidat tersimpan, tetapi jangan submit dulu "
        "sampai validasi k=5 dipastikan cocok."
    )

PEMERIKSAAN INPUT
Sample submission : /kaggle/input/competitions/schematics-2026/sample_submission.csv
Triple V4 probs   : /kaggle/input/notebooks/radithiafatir/testttrisan/test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy
Train embeddings  : /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy
Test embeddings   : /kaggle/working/beats_iter3plus_as2m_8s_test_embeddings.npy
Champion ditemukan: True

Shape sample : (814, 2)
Shape probs  : (814, 3)
Shape train  : (1826, 768)
Shape test   : (814, 768)

Input lolos pemeriksaan dimensi.

MENGHITUNG BEATS KNN
Jarak k=3 selesai.
Jarak k=5 selesai.
Jumlah sampel test: 814

VALIDASI TERHADAP CHAMPION LAMA
Unknown champion           : 245
Unknown hasil rekonstruksi : 245
ID Unknown yang sama       : 241 / 245
Perbedaan seluruh label k=5: 8 / 814

PERINGATAN: k=5 belum sama persis dengan champion.
Jangan submit kandidat dulu. Kirim output pemeriksaan ini agar perbedaan pipeline bisa ditelusuri.

KANDIDAT K=3
File kandi

In [9]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 1. PERIKSA 8 BARIS YANG LABELNYA BERUBAH
# ============================================================

champion = pd.read_csv(champion_path)
baseline_k5 = pd.read_csv(
    "/kaggle/working/submission_triple_v4_conf_knn50_u245.csv"
)

ID_COL = "file_id"
LABEL_COL = "label"

champion["file_id"] = champion["file_id"].astype(str)
baseline_k5["file_id"] = baseline_k5["file_id"].astype(str)

champion_map = champion.set_index("file_id")["label"]
baseline_map = baseline_k5.set_index("file_id")["label"]

changed_ids = sorted(
    set(champion.loc[
        champion["label"] == "Unknown", "file_id"
    ])
    ^
    set(baseline_k5.loc[
        baseline_k5["label"] == "Unknown", "file_id"
    ])
)

# Urutan sampel sesuai sample_submission yang dipakai untuk prediksi
id_to_pos = {
    str(file_id): i
    for i, file_id in enumerate(sample_sub["file_id"])
}

rows = []

for file_id in changed_ids:
    i = id_to_pos[file_id]

    rows.append({
        "file_id": file_id,
        "champion_label": champion_map[file_id],
        "reconstructed_k5_label": baseline_map[file_id],
        "triple_v4_confidence": float(test_probs[i].max()),
        "confidence_ood_rank": float(confidence_rank[i]),
        "knn_distance_k5": float(knn_dist_k5[i]),
        "knn_ood_rank_k5": float(
            pd.Series(knn_dist_k5).rank(
                pct=True, method="average"
            ).iloc[i]
        ),
        "fusion_score_k5": float(score_k5[i]),
        "fusion_score_k3": float(score_k3[i]),
    })

diff_df = pd.DataFrame(rows)

print("=" * 75)
print("PERBEDAAN LABEL: CHAMPION VS REKONSTRUKSI K=5")
print("=" * 75)

display(diff_df)

print("\nJumlah ID yang berbeda:", len(changed_ids))

# ============================================================
# 2. CARI ARTEFAK KNN YANG SUDAH ADA
# ============================================================

search_dirs = [
    "/kaggle/working",
    "/kaggle/input/notebooks/radithiafatir/testttrisan",
    "/kaggle/input/notebooks/radithiafatir/rill-versi-baru",
]

keywords = [
    "knn", "distance", "distances", "ood",
    "fusion", "score", "rank", "unknown"
]

found_files = []

for root in search_dirs:
    if not os.path.isdir(root):
        continue

    for current_dir, _, filenames in os.walk(root):
        for filename in filenames:
            if any(word in filename.lower() for word in keywords):
                found_files.append(
                    os.path.join(current_dir, filename)
                )

print("\n" + "=" * 75)
print("FILE TERKAIT KNN / OOD / SCORE YANG DITEMUKAN")
print("=" * 75)

for path in sorted(set(found_files)):
    print(path)

if not found_files:
    print("Tidak ditemukan artefak dengan nama terkait.")

PERBEDAAN LABEL: CHAMPION VS REKONSTRUKSI K=5


""



Jumlah ID yang berbeda: 0

FILE TERKAIT KNN / OOD / SCORE YANG DITEMUKAN
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/submission_triple_v4_conf_knn50_u245.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_conf_knn50_selected_245.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_loco_maxprob_knn_weight_search.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_nonbird_ood_count_sensitivity.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_nonbird_ood_oof_known_scores.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_nonbird_ood_test_scores.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_nonbird_ood_top245_overlap.csv
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_ood_audio_audit.zip
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_ood_audio_audit/04_low_ood_control_4d4d2abf9ccb.wav
/kaggle/input/notebooks/radithiafatir/rill-versi-baru/trip

In [10]:
import os
import numpy as np
import pandas as pd

# ============================================================
# 1. REKONSTRUKSI K=5 DARI SKOR YANG BENAR-BENAR DIHITUNG
# ============================================================

assert "score_k5" in globals(), "Variabel score_k5 tidak tersedia."
assert "test_probs" in globals(), "Variabel test_probs tidak tersedia."

reconstructed = sample_sub.copy()

pred_labels = np.array(CLASS_NAMES, dtype=object)[
    test_probs.argmax(axis=1)
]

selected_idx = np.argsort(
    -score_k5, kind="stable"
)[:N_UNKNOWN]

pred_labels[selected_idx] = "Unknown"
reconstructed[LABEL_COL] = pred_labels

print("Unknown hasil rekonstruksi:", 
      int((reconstructed[LABEL_COL] == "Unknown").sum()))

# ============================================================
# 2. BANDINGKAN DENGAN SEMUA SALINAN CHAMPION
# ============================================================

champion_paths = [
    "/kaggle/working/submission_triple_v4_conf_knn50_u245.csv",
    "/kaggle/input/notebooks/radithiafatir/rill-versi-baru/"
    "submission_triple_v4_conf_knn50_u245.csv",
]

for path in champion_paths:
    if not os.path.isfile(path):
        print("\nFile tidak ditemukan:", path)
        continue

    champion_df = pd.read_csv(path)

    champion_df[ID_COL] = champion_df[ID_COL].astype(str)
    reconstructed_cmp = reconstructed.copy()
    reconstructed_cmp[ID_COL] = reconstructed_cmp[ID_COL].astype(str)

    merged = champion_df[[ID_COL, LABEL_COL]].merge(
        reconstructed_cmp[[ID_COL, LABEL_COL]],
        on=ID_COL,
        how="outer",
        suffixes=("_champion", "_reconstructed"),
        indicator=True,
        validate="one_to_one"
    )

    if not (merged["_merge"] == "both").all():
        print("\nPERINGATAN: ID tidak sama pada", path)
        print(merged["_merge"].value_counts())
        continue

    label_a = merged[f"{LABEL_COL}_champion"]
    label_b = merged[f"{LABEL_COL}_reconstructed"]

    mismatch = merged[label_a != label_b].copy()

    unknown_a = set(
        merged.loc[label_a == "Unknown", ID_COL]
    )
    unknown_b = set(
        merged.loc[label_b == "Unknown", ID_COL]
    )

    print("\n" + "=" * 70)
    print("CHAMPION:", path)
    print("Jumlah baris berbeda:", len(mismatch), "/", len(merged))
    print("ID Unknown sama:", len(unknown_a & unknown_b), "/", N_UNKNOWN)
    print("ID Unknown berbeda:", len(unknown_a ^ unknown_b))

    if len(mismatch):
        display(mismatch[
            [ID_COL, f"{LABEL_COL}_champion", 
             f"{LABEL_COL}_reconstructed"]
        ].head(20))
    else:
        print("Semua label cocok.")

# ============================================================
# 3. PERIKSA ARTEFAK YANG MUNGKIN MENYIMPAN PILIHAN UNKNOWN ASLI
# ============================================================

artifact_paths = [
    "/kaggle/working/triple_v4_conf_knn50_selected_245.csv",
    "/kaggle/input/notebooks/radithiafatir/rill-versi-baru/"
    "triple_v4_conf_knn50_selected_245.csv",
    "/kaggle/input/notebooks/radithiafatir/testttrisan/"
    "ood_rank_diagnostics_test.csv",
]

for path in artifact_paths:
    if os.path.isfile(path):
        df = pd.read_csv(path)

        print("\n" + "=" * 70)
        print("ARTEFAK:", path)
        print("Shape:", df.shape)
        print("Kolom:", df.columns.tolist())
        display(df.head(5))

Unknown hasil rekonstruksi: 245

CHAMPION: /kaggle/working/submission_triple_v4_conf_knn50_u245.csv
Jumlah baris berbeda: 8 / 814
ID Unknown sama: 241 / 245
ID Unknown berbeda: 8


,file_id,label_champion,label_reconstructed
40,0e33ee0e52dc415ed1517469e2162ea4e93473be5f21a8...,Cendrawasih,Unknown
75,19ffa6c8dfba3186fbc2d7c3c3f7102d090d3608e943fc...,Kutilang,Unknown
203,416f6747349b59a2766c35616746ef8b769c24c11abeda...,Unknown,Cendrawasih
216,4673d6e064bd8023cc534309b4d7d7632dece79ce83120...,Bondol Jawa,Unknown
345,6e49e9edf3397cd897382dc9114cced5d980c11b03691a...,Unknown,Cendrawasih
415,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,Unknown,Kutilang
479,9b0ea79b56474aaee940141977a99d82466217a8f32eea...,Unknown,Bondol Jawa
726,e7dea10fa9099887d0aa9170240e5c23c7acc70daf3579...,Bondol Jawa,Unknown



CHAMPION: /kaggle/input/notebooks/radithiafatir/rill-versi-baru/submission_triple_v4_conf_knn50_u245.csv
Jumlah baris berbeda: 8 / 814
ID Unknown sama: 241 / 245
ID Unknown berbeda: 8


,file_id,label_champion,label_reconstructed
40,0e33ee0e52dc415ed1517469e2162ea4e93473be5f21a8...,Cendrawasih,Unknown
75,19ffa6c8dfba3186fbc2d7c3c3f7102d090d3608e943fc...,Kutilang,Unknown
203,416f6747349b59a2766c35616746ef8b769c24c11abeda...,Unknown,Cendrawasih
216,4673d6e064bd8023cc534309b4d7d7632dece79ce83120...,Bondol Jawa,Unknown
345,6e49e9edf3397cd897382dc9114cced5d980c11b03691a...,Unknown,Cendrawasih
415,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,Unknown,Kutilang
479,9b0ea79b56474aaee940141977a99d82466217a8f32eea...,Unknown,Bondol Jawa
726,e7dea10fa9099887d0aa9170240e5c23c7acc70daf3579...,Bondol Jawa,Unknown



ARTEFAK: /kaggle/working/triple_v4_conf_knn50_selected_245.csv
Shape: (245, 41)
Kolom: ['file_id', 'triple_v4_pred_known_class', 'triple_v4_confidence', 'triple_v4_uncertainty', 'triple_v4_entropy_norm', 'triple_v4_disagreement_tv', 'beats_cosine_proto_distance', 'beats_knn_cosine_distance', 'beats_diag_mahalanobis', 'rms', 'rms_db', 'zero_crossing_rate', 'spectral_flatness_median', 'spectral_flatness_mean', 'spectral_entropy_mean', 'spectral_centroid_hz', 'low_band_fraction_lt500hz', 'high_band_fraction_ge2500hz', 'spectral_flatness_p90', 'rank_uncertainty', 'rank_entropy', 'rank_disagreement_tv', 'rank_margin_low_is_ood', 'rank_beats_cosine_proto_distance', 'rank_beats_knn_cosine_distance', 'rank_beats_diag_mahalanobis', 'rank_acoustic_rms', 'rank_acoustic_rms_db', 'rank_acoustic_zero_crossing_rate', 'rank_acoustic_spectral_flatness_median', 'rank_acoustic_spectral_flatness_mean', 'rank_acoustic_spectral_entropy_mean', 'rank_acoustic_spectral_centroid_hz', 'rank_acoustic_low_band_fr

,file_id,triple_v4_pred_known_class,triple_v4_confidence,triple_v4_uncertainty,triple_v4_entropy_norm,triple_v4_disagreement_tv,beats_cosine_proto_distance,beats_knn_cosine_distance,beats_diag_mahalanobis,rms,rms_db,zero_crossing_rate,spectral_flatness_median,spectral_flatness_mean,spectral_entropy_mean,spectral_centroid_hz,low_band_fraction_lt500hz,high_band_fraction_ge2500hz,spectral_flatness_p90,rank_uncertainty,rank_entropy,rank_disagreement_tv,rank_margin_low_is_ood,rank_beats_cosine_proto_distance,rank_beats_knn_cosine_distance,rank_beats_diag_mahalanobis,rank_acoustic_rms,rank_acoustic_rms_db,rank_acoustic_zero_crossing_rate,rank_acoustic_spectral_flatness_median,rank_acoustic_spectral_flatness_mean,rank_acoustic_spectral_entropy_mean,rank_acoustic_spectral_centroid_hz,rank_acoustic_low_band_fraction_lt500hz,rank_acoustic_high_band_fraction_ge2500hz,rank_acoustic_spectral_flatness_p90,ood_rank_score_triple_v4_uncertainty_disagreement_60_40,ood_rank_score_beats_embedding_only,ood_rank_score_triple_plus_beats_embedding,ood_rank_score_all_signals_exploratory,ood_score_conf_knn50
0,204fc6bb00c341216cbe2d9d8dcb07d85dc6892348e220...,Bondol Jawa,0.452098,0.547902,0.894954,0.455306,0.233269,0.229809,1.328793,0.048585,-26.269981,0.177759,0.073469,0.077052,0.668010,1001.5526,0.675698,0.111318,0.105889,0.996714,0.990142,0.915663,0.996714,0.929901,0.995071,0.974261,0.068456,0.068456,0.052026,0.271632,0.271084,0.149507,0.026835,0.975904,0.011501,0.186747,0.964294,0.966411,0.967990,0.813921,0.962531
1,a6b4b00ba92712f8f21d9418146f7562861ac63119ec59...,Bondol Jawa,0.372956,0.627044,0.996313,0.295911,0.196001,0.193067,1.105634,0.115649,-18.737175,0.425347,0.145700,0.140619,0.786536,3161.8674,0.179166,0.699476,0.196787,1.000000,1.000000,0.869113,0.995071,0.805586,0.991238,0.826396,0.871851,0.871851,0.662103,0.687842,0.638007,0.719058,0.581051,0.568456,0.626506,0.588171,0.947645,0.874407,0.925986,0.870110,0.955774
2,345aa833b707506b706b7d53dc5e95ca34771e52f0eae1...,Cendrawasih,0.452175,0.547825,0.953999,0.549799,0.196306,0.217682,1.012364,0.048994,-26.197105,0.288979,0.091151,0.107750,0.662423,1915.9478,0.210693,0.283600,0.205694,0.996714,0.996166,0.935378,0.987952,0.806134,0.994524,0.586528,0.071194,0.071194,0.320372,0.395400,0.451260,0.136364,0.207558,0.629792,0.135816,0.622125,0.972180,0.795728,0.914513,0.783116,0.954545
3,07093600145543b77969e71bc5a53780b3e2b97e50846a...,Kutilang,0.435488,0.564512,0.952268,0.521986,0.236694,0.204333,1.103284,0.094058,-20.532078,0.275487,0.092626,0.159745,0.791116,2796.9949,0.121701,0.432118,0.500001,0.997262,0.996166,0.930449,0.994524,0.937568,0.993428,0.820372,0.680723,0.680723,0.285323,0.407448,0.730559,0.740416,0.484118,0.435926,0.335706,1.000000,0.970537,0.917123,0.956681,0.908702,0.953931
4,14619b5e0a8e61898c76b4b314832d6942112049a9ebf1...,Kutilang,0.388741,0.611259,0.983676,0.454557,0.189178,0.189255,1.140270,0.097526,-20.217588,0.471097,0.179583,0.182011,0.807028,3714.6000,0.056593,0.816833,0.239597,0.999452,0.998357,0.915663,0.997262,0.772180,0.989595,0.877327,0.710843,0.710843,0.789704,0.819825,0.823658,0.814348,0.726177,0.256298,0.783133,0.716320,0.965936,0.879701,0.934912,0.906462,0.949631



ARTEFAK: /kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_conf_knn50_selected_245.csv
Shape: (245, 41)
Kolom: ['file_id', 'triple_v4_pred_known_class', 'triple_v4_confidence', 'triple_v4_uncertainty', 'triple_v4_entropy_norm', 'triple_v4_disagreement_tv', 'beats_cosine_proto_distance', 'beats_knn_cosine_distance', 'beats_diag_mahalanobis', 'rms', 'rms_db', 'zero_crossing_rate', 'spectral_flatness_median', 'spectral_flatness_mean', 'spectral_entropy_mean', 'spectral_centroid_hz', 'low_band_fraction_lt500hz', 'high_band_fraction_ge2500hz', 'spectral_flatness_p90', 'rank_uncertainty', 'rank_entropy', 'rank_disagreement_tv', 'rank_margin_low_is_ood', 'rank_beats_cosine_proto_distance', 'rank_beats_knn_cosine_distance', 'rank_beats_diag_mahalanobis', 'rank_acoustic_rms', 'rank_acoustic_rms_db', 'rank_acoustic_zero_crossing_rate', 'rank_acoustic_spectral_flatness_median', 'rank_acoustic_spectral_flatness_mean', 'rank_acoustic_spectral_entropy_mean', 'rank_acoustic_spectral_ce

,file_id,triple_v4_pred_known_class,triple_v4_confidence,triple_v4_uncertainty,triple_v4_entropy_norm,triple_v4_disagreement_tv,beats_cosine_proto_distance,beats_knn_cosine_distance,beats_diag_mahalanobis,rms,rms_db,zero_crossing_rate,spectral_flatness_median,spectral_flatness_mean,spectral_entropy_mean,spectral_centroid_hz,low_band_fraction_lt500hz,high_band_fraction_ge2500hz,spectral_flatness_p90,rank_uncertainty,rank_entropy,rank_disagreement_tv,rank_margin_low_is_ood,rank_beats_cosine_proto_distance,rank_beats_knn_cosine_distance,rank_beats_diag_mahalanobis,rank_acoustic_rms,rank_acoustic_rms_db,rank_acoustic_zero_crossing_rate,rank_acoustic_spectral_flatness_median,rank_acoustic_spectral_flatness_mean,rank_acoustic_spectral_entropy_mean,rank_acoustic_spectral_centroid_hz,rank_acoustic_low_band_fraction_lt500hz,rank_acoustic_high_band_fraction_ge2500hz,rank_acoustic_spectral_flatness_p90,ood_rank_score_triple_v4_uncertainty_disagreement_60_40,ood_rank_score_beats_embedding_only,ood_rank_score_triple_plus_beats_embedding,ood_rank_score_all_signals_exploratory,ood_score_conf_knn50
0,204fc6bb00c341216cbe2d9d8dcb07d85dc6892348e220...,Bondol Jawa,0.452098,0.547902,0.894954,0.455306,0.233269,0.229809,1.328793,0.048585,-26.269981,0.177759,0.073469,0.077052,0.668010,1001.5526,0.675698,0.111318,0.105889,0.996714,0.990142,0.915663,0.996714,0.929901,0.995071,0.974261,0.068456,0.068456,0.052026,0.271632,0.271084,0.149507,0.026835,0.975904,0.011501,0.186747,0.964294,0.966411,0.967990,0.813921,0.962531
1,a6b4b00ba92712f8f21d9418146f7562861ac63119ec59...,Bondol Jawa,0.372956,0.627044,0.996313,0.295911,0.196001,0.193067,1.105634,0.115649,-18.737175,0.425347,0.145700,0.140619,0.786536,3161.8674,0.179166,0.699476,0.196787,1.000000,1.000000,0.869113,0.995071,0.805586,0.991238,0.826396,0.871851,0.871851,0.662103,0.687842,0.638007,0.719058,0.581051,0.568456,0.626506,0.588171,0.947645,0.874407,0.925986,0.870110,0.955774
2,345aa833b707506b706b7d53dc5e95ca34771e52f0eae1...,Cendrawasih,0.452175,0.547825,0.953999,0.549799,0.196306,0.217682,1.012364,0.048994,-26.197105,0.288979,0.091151,0.107750,0.662423,1915.9478,0.210693,0.283600,0.205694,0.996714,0.996166,0.935378,0.987952,0.806134,0.994524,0.586528,0.071194,0.071194,0.320372,0.395400,0.451260,0.136364,0.207558,0.629792,0.135816,0.622125,0.972180,0.795728,0.914513,0.783116,0.954545
3,07093600145543b77969e71bc5a53780b3e2b97e50846a...,Kutilang,0.435488,0.564512,0.952268,0.521986,0.236694,0.204333,1.103284,0.094058,-20.532078,0.275487,0.092626,0.159745,0.791116,2796.9949,0.121701,0.432118,0.500001,0.997262,0.996166,0.930449,0.994524,0.937568,0.993428,0.820372,0.680723,0.680723,0.285323,0.407448,0.730559,0.740416,0.484118,0.435926,0.335706,1.000000,0.970537,0.917123,0.956681,0.908702,0.953931
4,14619b5e0a8e61898c76b4b314832d6942112049a9ebf1...,Kutilang,0.388741,0.611259,0.983676,0.454557,0.189178,0.189255,1.140270,0.097526,-20.217588,0.471097,0.179583,0.182011,0.807028,3714.6000,0.056593,0.816833,0.239597,0.999452,0.998357,0.915663,0.997262,0.772180,0.989595,0.877327,0.710843,0.710843,0.789704,0.819825,0.823658,0.814348,0.726177,0.256298,0.783133,0.716320,0.965936,0.879701,0.934912,0.906462,0.949631



ARTEFAK: /kaggle/input/notebooks/radithiafatir/testttrisan/ood_rank_diagnostics_test.csv
Shape: (814, 16)
Kolom: ['file_id', 'raw_uncertainty', 'rank_uncertainty', 'raw_entropy', 'rank_entropy', 'raw_disagreement', 'rank_disagreement', 'raw_centroid_distance', 'rank_centroid_distance', 'raw_knn_distance', 'rank_knn_distance', 'ood_score_confidence_only', 'ood_score_confidence_entropy', 'ood_score_confidence_disagreement', 'ood_score_embedding_distance', 'ood_score_fused_all_signals']


,file_id,raw_uncertainty,rank_uncertainty,raw_entropy,rank_entropy,raw_disagreement,rank_disagreement,raw_centroid_distance,rank_centroid_distance,raw_knn_distance,rank_knn_distance,ood_score_confidence_only,ood_score_confidence_entropy,ood_score_confidence_disagreement,ood_score_embedding_distance,ood_score_fused_all_signals
0,7e05954cc1e90e75330575982ab9dac4717fad2fb70599...,0.135527,0.811610,0.374970,0.802848,0.092046,0.762870,0.009400,0.300110,0.012484,0.990142,0.811610,0.808105,0.797234,0.645126,0.750164
1,ac5e9f292b26bb1e0557beb235b9d6bcb066cc10834d98...,0.231918,0.857065,0.634564,0.931544,0.316708,0.891566,0.010630,0.408543,0.009932,0.952903,0.857065,0.886857,0.884310,0.680723,0.825958
2,e8ace030a12a1b16784f6b64a47d79793e3fdeb432d292...,0.064440,0.749726,0.224905,0.740964,0.031758,0.650602,0.008480,0.201533,0.009716,0.946878,0.749726,0.746221,0.722755,0.574206,0.675493
3,337e8971e3cec5155ec0ea5887c7d5cb360379574ab9e6...,0.083464,0.776561,0.293922,0.774918,0.045878,0.689485,0.008200,0.174699,0.007927,0.871303,0.776561,0.775904,0.754381,0.523001,0.682749
4,e720974ec6c3c5a9c4c43739ac4f93037bc5d4470e92ab...,0.585965,1.000000,0.978097,1.000000,0.205645,0.859255,0.013789,0.651698,0.004939,0.485761,1.000000,1.000000,0.964814,0.568729,0.842470


In [11]:
import os
import numpy as np
import pandas as pd

# ============================================================
# 1. MEMUAT ARTEFAK PIPELINE CHAMPION
# ============================================================

artifact_path = find_file(
    "triple_v4_conf_knn50_selected_245.csv"
)

selected = pd.read_csv(artifact_path)

selected["file_id"] = selected["file_id"].astype(str)
sample_ids = sample_sub["file_id"].astype(str).to_numpy()

id_to_pos = {
    file_id: i for i, file_id in enumerate(sample_ids)
}

assert selected["file_id"].isin(id_to_pos).all(), (
    "Ada file_id dari artefak yang tidak ada di sample submission."
)

pos = np.array([
    id_to_pos[file_id]
    for file_id in selected["file_id"]
])

# ============================================================
# 2. COCOKKAN PREDIKSI TRIPLE V4
# ============================================================

saved_conf = selected["triple_v4_confidence"].to_numpy()
current_conf = test_probs[pos].max(axis=1)

saved_pred = selected["triple_v4_pred_known_class"].astype(str).to_numpy()
current_pred = np.array(CLASS_NAMES)[test_probs[pos].argmax(axis=1)]

print("=" * 70)
print("A. VALIDASI PROBABILITAS TRIPLE V4")
print("=" * 70)

print(
    "Confidence MAE:",
    round(float(np.mean(np.abs(saved_conf - current_conf))), 8)
)
print(
    "Predicted class cocok:",
    int(np.sum(saved_pred == current_pred)),
    "/",
    len(selected)
)

# ============================================================
# 3. COCOKKAN JARAK KNN LAMA DENGAN HITUNGAN SEKARANG
# ============================================================

saved_dist = selected["beats_knn_cosine_distance"].to_numpy()

def distance_report(name, current_dist):
    current_selected = np.asarray(current_dist)[pos]

    mae = np.mean(np.abs(saved_dist - current_selected))
    corr = np.corrcoef(saved_dist, current_selected)[0, 1]
    spearman = pd.Series(saved_dist).corr(
        pd.Series(current_selected),
        method="spearman"
    )

    print(f"\n{name}")
    print("  MAE              :", round(float(mae), 8))
    print("  Pearson correlation:", round(float(corr), 6))
    print("  Spearman correlation:", round(float(spearman), 6))

print("\n" + "=" * 70)
print("B. VALIDASI JARAK BEATS KNN")
print("=" * 70)

distance_report("Jarak k=3 sekarang", knn_dist_k3)
distance_report("Jarak k=5 sekarang", knn_dist_k5)

# ============================================================
# 4. BANDINGKAN SKOR FUSION LAMA DAN SEKARANG
# ============================================================

saved_fusion = selected["ood_score_conf_knn50"].to_numpy()

print("\n" + "=" * 70)
print("C. VALIDASI SKOR FUSION")
print("=" * 70)

for name, current_score in [
    ("Fusion k=3", score_k3),
    ("Fusion k=5", score_k5),
]:
    current_selected = np.asarray(current_score)[pos]

    mae = np.mean(np.abs(saved_fusion - current_selected))
    corr = np.corrcoef(saved_fusion, current_selected)[0, 1]
    spearman = pd.Series(saved_fusion).corr(
        pd.Series(current_selected),
        method="spearman"
    )

    print(f"\n{name}")
    print("  MAE                :", round(float(mae), 8))
    print("  Pearson correlation:", round(float(corr), 6))
    print("  Spearman correlation:", round(float(spearman), 6))

# ============================================================
# 5. PASTIKAN 245 ID ARTEFAK MEMANG SAMA DENGAN CHAMPION
# ============================================================

champion_path = find_file(
    "submission_triple_v4_conf_knn50_u245.csv"
)
champion = pd.read_csv(champion_path)

champion_unknown = set(
    champion.loc[
        champion["label"] == "Unknown", "file_id"
    ].astype(str)
)
artifact_unknown = set(selected["file_id"])

print("\n" + "=" * 70)
print("D. VALIDASI DAFTAR UNKNOWN CHAMPION")
print("=" * 70)

print("Unknown champion:", len(champion_unknown))
print("Baris artefak:", len(artifact_unknown))
print(
    "ID yang cocok:",
    len(champion_unknown & artifact_unknown),
    "/",
    len(champion_unknown)
)
print("Artefak:", artifact_path)

A. VALIDASI PROBABILITAS TRIPLE V4
Confidence MAE: 2e-08
Predicted class cocok: 245 / 245

B. VALIDASI JARAK BEATS KNN

Jarak k=3 sekarang
  MAE              : 0.00854897
  Pearson correlation: 0.99547
  Spearman correlation: 0.993729

Jarak k=5 sekarang
  MAE              : 0.00364423
  Pearson correlation: 0.999234
  Spearman correlation: 0.998512

C. VALIDASI SKOR FUSION

Fusion k=3
  MAE                : 0.00700496
  Pearson correlation: 0.99197
  Spearman correlation: 0.992453

Fusion k=5
  MAE                : 0.002981
  Pearson correlation: 0.998656
  Spearman correlation: 0.998256

D. VALIDASI DAFTAR UNKNOWN CHAMPION
Unknown champion: 245
Baris artefak: 245
ID yang cocok: 245 / 245
Artefak: /kaggle/working/triple_v4_conf_knn50_selected_245.csv


In [12]:
import os
import numpy as np
import pandas as pd

# ============================================================
# BANDINGKAN KNN HASIL HITUNG ULANG DENGAN ARTEFAK LAMA
# ============================================================

diag_path = (
    "/kaggle/input/notebooks/radithiafatir/"
    "testttrisan/ood_rank_diagnostics_test.csv"
)

assert os.path.isfile(diag_path), (
    f"File diagnostik tidak ditemukan: {diag_path}"
)

diag = pd.read_csv(diag_path)
diag["file_id"] = diag["file_id"].astype(str)

current = pd.DataFrame({
    "file_id": sample_sub["file_id"].astype(str),
    "confidence_ood_current": 1.0 - test_probs.max(axis=1),
    "knn_k3_current": knn_dist_k3,
    "knn_k5_current": knn_dist_k5,
})

# Gabungkan berdasarkan ID agar tidak bergantung pada urutan baris.
merged = diag.merge(
    current,
    on="file_id",
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert (merged["_merge"] == "both").all(), (
    "ID pada file diagnostik dan test tidak cocok."
)

# ============================================================
# FUNGSI RINGKAS PERBANDINGAN
# ============================================================

def compare_values(name, saved, current_values):
    saved = np.asarray(saved, dtype=float)
    current_values = np.asarray(current_values, dtype=float)

    valid = np.isfinite(saved) & np.isfinite(current_values)
    saved = saved[valid]
    current_values = current_values[valid]

    print(f"\n{name}")
    print("Jumlah dibandingkan:", len(saved))

    if len(saved) == 0:
        print("Tidak ada nilai valid.")
        return

    print(
        "MAE:",
        round(float(np.mean(np.abs(saved - current_values))), 8)
    )

    if len(saved) > 1:
        print(
            "Pearson:",
            round(float(np.corrcoef(saved, current_values)[0, 1]), 6)
        )
        print(
            "Spearman:",
            round(
                float(
                    pd.Series(saved).corr(
                        pd.Series(current_values),
                        method="spearman"
                    )
                ),
                6
            )
        )

# ============================================================
# 1. RAW DISTANCE DAN RANK KNN
# ============================================================

compare_values(
    "Raw kNN distance lama vs k=5 sekarang",
    merged["raw_knn_distance"],
    merged["knn_k5_current"]
)

current_knn_rank = merged["knn_k5_current"].rank(
    pct=True, method="average"
)

compare_values(
    "Rank kNN lama vs rank k=5 sekarang",
    merged["rank_knn_distance"],
    current_knn_rank
)

# ============================================================
# 2. CONFIDENCE / UNCERTAINTY
# ============================================================

compare_values(
    "Raw uncertainty lama vs confidence OOD sekarang",
    merged["raw_uncertainty"],
    merged["confidence_ood_current"]
)

current_conf_rank = merged["confidence_ood_current"].rank(
    pct=True, method="average"
)

compare_values(
    "Rank uncertainty lama vs rank confidence sekarang",
    merged["rank_uncertainty"],
    current_conf_rank
)

# ============================================================
# 3. CEK ID DI SEKITAR AMBANG UNKNOWN
# ============================================================

merged["current_fusion_k5"] = (
    0.5 * current_conf_rank
    + 0.5 * current_knn_rank
)

merged["current_fusion_k3"] = (
    0.5 * current_conf_rank
    + 0.5 * merged["knn_k3_current"].rank(
        pct=True, method="average"
    )
)

print("\n" + "=" * 70)
print("TOP 15 BERDASARKAN FUSION k=5 SAAT INI")
print("=" * 70)

display(
    merged.sort_values(
        "current_fusion_k5", ascending=False
    )[
        [
            "file_id",
            "raw_knn_distance",
            "knn_k5_current",
            "rank_knn_distance",
            "current_fusion_k5",
            "current_fusion_k3",
        ]
    ].head(15)
)

print("\nFile diagnostik:", diag_path)


Raw kNN distance lama vs k=5 sekarang
Jumlah dibandingkan: 814
MAE: 0.10631959
Pearson: 0.740968
Spearman: 0.744712

Rank kNN lama vs rank k=5 sekarang
Jumlah dibandingkan: 814
MAE: 0.20190487
Pearson: 0.728837
Spearman: 0.744739

Raw uncertainty lama vs confidence OOD sekarang
Jumlah dibandingkan: 814
MAE: 0.08109769
Pearson: 0.840518
Spearman: 0.853612

Rank uncertainty lama vs rank confidence sekarang
Jumlah dibandingkan: 814
MAE: 0.29126156
Pearson: 0.746971
Spearman: 0.853662

TOP 15 BERDASARKAN FUSION k=5 SAAT INI


,file_id,raw_knn_distance,knn_k5_current,rank_knn_distance,current_fusion_k5,current_fusion_k3
95,204fc6bb00c341216cbe2d9d8dcb07d85dc6892348e220...,0.008423,0.227297,0.898686,0.962531,0.963759
522,a6b4b00ba92712f8f21d9418146f7562861ac63119ec59...,0.012447,0.189315,0.990142,0.957002,0.953317
165,345aa833b707506b706b7d53dc5e95ca34771e52f0eae1...,0.008936,0.214508,0.924425,0.956388,0.953317
21,07093600145543b77969e71bc5a53780b3e2b97e50846a...,0.014667,0.199890,0.998357,0.952703,0.951474
174,38e431c47dfbdfcae8379af891a8206c63009920a107f3...,0.012095,0.187263,0.985761,0.945946,0.945946
55,14619b5e0a8e61898c76b4b314832d6942112049a9ebf1...,0.009572,0.181466,0.943593,0.942875,0.945332
524,a72432f005e7a35a575a15117324348784ef3c3b60f61e...,0.010860,0.258461,0.969332,0.942260,0.941646
407,81890d827fb44ea6f1579fbd5ce1b371063d6013c663e2...,0.010350,0.172113,0.963855,0.929975,0.936118
620,c4e56871b234a1747249ca3fcd62777da79971b329cba4...,0.009652,0.187176,0.944140,0.928133,0.933047
753,ecddfa37b924a13bb58c2f7cfd5cfc1216d7aa88d685c9...,0.011858,0.188364,0.983571,0.922604,0.918919



File diagnostik: /kaggle/input/notebooks/radithiafatir/testttrisan/ood_rank_diagnostics_test.csv


In [13]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# ============================================================
# 1. MEMUAT ARTEFAK CHAMPION
# ============================================================

artifact_path = find_file(
    "triple_v4_conf_knn50_selected_245.csv"
)
selected = pd.read_csv(artifact_path)
selected["file_id"] = selected["file_id"].astype(str)

sample_ids = sample_sub["file_id"].astype(str).to_numpy()

id_to_pos = {
    file_id: i for i, file_id in enumerate(sample_ids)
}

assert selected["file_id"].isin(id_to_pos).all()

pos = np.array([
    id_to_pos[file_id] for file_id in selected["file_id"]
])

saved_distance = selected[
    "beats_knn_cosine_distance"
].to_numpy(dtype=float)

saved_fusion = selected[
    "ood_score_conf_knn50"
].to_numpy(dtype=float)

# Confidence yang sebelumnya sudah terbukti cocok.
confidence_ood = 1.0 - test_probs.max(axis=1)

confidence_rank = pd.Series(
    confidence_ood
).rank(pct=True, method="average").to_numpy()

# ============================================================
# 2. HITUNG JARAK SAMPAI 50 TETANGGA SEKALI SAJA
# ============================================================

K_VALUES = [1, 2, 3, 4, 5, 7, 10, 12, 15, 20, 30, 40, 50]

train_norm = normalize(
    np.asarray(train_emb, dtype=np.float32),
    norm="l2"
)
test_norm = normalize(
    np.asarray(test_emb, dtype=np.float32),
    norm="l2"
)

knn = NearestNeighbors(
    n_neighbors=max(K_VALUES),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)
knn.fit(train_norm)

distances, _ = knn.kneighbors(test_norm)

# ============================================================
# 3. BANDINKAN JARAK DAN SKOR FUSION
# ============================================================

results = []

for k in K_VALUES:
    d = distances[:, :k].mean(axis=1)

    # Bandingkan jarak yang disimpan dengan jarak hasil hitung ulang.
    d_saved = saved_distance
    d_current = d[pos]

    distance_mae = np.mean(
        np.abs(d_saved - d_current)
    )
    distance_spearman = pd.Series(d_saved).corr(
        pd.Series(d_current), method="spearman"
    )

    # Rekonstruksi fusion 50:50 pada seluruh 814 sampel.
    knn_rank = pd.Series(d).rank(
        pct=True, method="average"
    ).to_numpy()

    fusion = 0.5 * confidence_rank + 0.5 * knn_rank
    fusion_current = fusion[pos]

    fusion_mae = np.mean(
        np.abs(saved_fusion - fusion_current)
    )
    fusion_spearman = pd.Series(saved_fusion).corr(
        pd.Series(fusion_current), method="spearman"
    )

    results.append({
        "k": k,
        "distance_MAE": distance_mae,
        "distance_Spearman": distance_spearman,
        "fusion_MAE": fusion_mae,
        "fusion_Spearman": fusion_spearman
    })

result_df = pd.DataFrame(results)

print("=== URUTAN KECocokan JARAK ===")
display(
    result_df.sort_values("distance_MAE").round(6)
)

print("=== URUTAN KECocokan SKOR FUSION ===")
display(
    result_df.sort_values("fusion_MAE").round(6)
)

=== URUTAN KECocokan JARAK ===


,k,distance_MAE,distance_Spearman,fusion_MAE,fusion_Spearman
5,7,0.000412,0.999739,0.001186,0.999382
4,5,0.003644,0.998512,0.002981,0.998256
6,10,0.004238,0.998606,0.003156,0.997522
3,4,0.005837,0.996901,0.004668,0.996406
7,12,0.006566,0.997361,0.004540,0.995362
2,3,0.008549,0.993729,0.007005,0.992453
8,15,0.009567,0.995170,0.006215,0.992866
1,2,0.012040,0.987073,0.010400,0.984496
9,20,0.013686,0.992122,0.008409,0.988169
0,1,0.016927,0.972175,0.016229,0.966615


=== URUTAN KECocokan SKOR FUSION ===


,k,distance_MAE,distance_Spearman,fusion_MAE,fusion_Spearman
5,7,0.000412,0.999739,0.001186,0.999382
4,5,0.003644,0.998512,0.002981,0.998256
6,10,0.004238,0.998606,0.003156,0.997522
7,12,0.006566,0.997361,0.004540,0.995362
3,4,0.005837,0.996901,0.004668,0.996406
8,15,0.009567,0.995170,0.006215,0.992866
2,3,0.008549,0.993729,0.007005,0.992453
9,20,0.013686,0.992122,0.008409,0.988169
1,2,0.012040,0.987073,0.010400,0.984496
10,30,0.019976,0.985844,0.011525,0.979921


In [14]:
# ============================================================
# VALIDASI CHAMPION K=7 + KANDIDAT K=3
# Triple V4 confidence / BEATs kNN = 50:50
# Jumlah Unknown = 245
# Champion lama tidak ditimpa.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# ------------------------------------------------------------
# 1. KONFIGURASI
# ------------------------------------------------------------

K_BASELINE = 7
K_CANDIDATE = 3
N_UNKNOWN = 245

CLASS_NAMES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]

CHAMPION_PATH = (
    "/kaggle/working/"
    "submission_triple_v4_conf_knn50_u245.csv"
)

CANDIDATE_PATH = (
    "/kaggle/working/"
    "submission_triple_v4_conf_knn50_k3_u245.csv"
)

assert os.path.isfile(CHAMPION_PATH), (
    "File champion tidak ditemukan. Jangan lanjut sebelum "
    "file submission dengan skor 0.68085 tersedia."
)

# Menggunakan variabel dari cell sebelumnya.
assert "sample_sub" in globals()
assert "test_probs" in globals()
assert "train_emb" in globals()
assert "test_emb" in globals()

champion = pd.read_csv(CHAMPION_PATH)
sample = sample_sub.copy()

ID_COL = "file_id"
LABEL_COL = "label"

assert len(sample) == 814
assert len(champion) == 814
assert sample[ID_COL].is_unique
assert champion[ID_COL].is_unique
assert set(sample[ID_COL].astype(str)) == set(
    champion[ID_COL].astype(str)
)

assert test_probs.shape == (len(sample), 3)
assert train_emb.shape[0] == 1826
assert test_emb.shape[0] == len(sample)

# ------------------------------------------------------------
# 2. HITUNG JARAK KNN K=7 DAN K=3
# ------------------------------------------------------------

print("=" * 65)
print("MENGHITUNG JARAK BEATS")
print("=" * 65)

train_norm = normalize(
    np.asarray(train_emb, dtype=np.float32),
    norm="l2"
)
test_norm = normalize(
    np.asarray(test_emb, dtype=np.float32),
    norm="l2"
)

nn = NearestNeighbors(
    n_neighbors=K_BASELINE,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nn.fit(train_norm)
distances, _ = nn.kneighbors(test_norm)

distance_k3 = distances[:, :K_CANDIDATE].mean(axis=1)
distance_k7 = distances[:, :K_BASELINE].mean(axis=1)

print("Shape jarak:", distances.shape)

# ------------------------------------------------------------
# 3. FUNGSI PEMBUATAN SUBMISSION
# ------------------------------------------------------------

# Confidence OOD = 1 - confidence Triple V4.
confidence_ood = 1.0 - test_probs.max(axis=1)

confidence_rank = pd.Series(
    confidence_ood
).rank(
    pct=True,
    method="average"
).to_numpy()


def build_submission(knn_distance):
    knn_rank = pd.Series(
        knn_distance
    ).rank(
        pct=True,
        method="average"
    ).to_numpy()

    fusion_score = (
        0.5 * confidence_rank
        + 0.5 * knn_rank
    )

    selected_idx = np.argsort(
        -fusion_score,
        kind="stable"
    )[:N_UNKNOWN]

    pred = np.array(
        CLASS_NAMES, dtype=object
    )[test_probs.argmax(axis=1)]

    pred[selected_idx] = "Unknown"

    output = sample.copy()
    output[LABEL_COL] = pred

    return output, fusion_score


# ------------------------------------------------------------
# 4. REKONSTRUKSI K=7 DAN BANDINGKAN DENGAN CHAMPION
# ------------------------------------------------------------

reconstructed_k7, score_k7 = build_submission(distance_k7)

comparison = champion[
    [ID_COL, LABEL_COL]
].copy()

comparison[ID_COL] = comparison[ID_COL].astype(str)

reconstructed_compare = reconstructed_k7[
    [ID_COL, LABEL_COL]
].copy()

reconstructed_compare[ID_COL] = (
    reconstructed_compare[ID_COL].astype(str)
)

comparison = comparison.merge(
    reconstructed_compare,
    on=ID_COL,
    how="inner",
    validate="one_to_one",
    suffixes=("_champion", "_reconstructed")
)

mismatch = comparison[
    comparison[f"{LABEL_COL}_champion"]
    != comparison[f"{LABEL_COL}_reconstructed"]
].copy()

champion_unknown = set(
    champion.loc[
        champion[LABEL_COL] == "Unknown", ID_COL
    ].astype(str)
)

reconstructed_unknown = set(
    reconstructed_k7.loc[
        reconstructed_k7[LABEL_COL] == "Unknown", ID_COL
    ].astype(str)
)

print("\n" + "=" * 65)
print("VALIDASI BASELINE K=7")
print("=" * 65)

print("Unknown champion:", len(champion_unknown))
print("Unknown rekonstruksi:", len(reconstructed_unknown))
print(
    "ID Unknown cocok:",
    len(champion_unknown & reconstructed_unknown),
    "/",
    N_UNKNOWN
)
print("Label berbeda:", len(mismatch), "/", len(sample))

if len(mismatch):
    print("\nBaris yang berbeda:")
    display(mismatch.head(20))

# ------------------------------------------------------------
# 5. BUAT KANDIDAT K=3 HANYA JIKA K=7 COCOK PERSIS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("KEPUTUSAN")
print("=" * 65)

if len(mismatch) != 0:
    print(
        "BASELINE BELUM COCOK. Kandidat k=3 tidak dibuat ulang.\n"
        "Jangan submit dulu. Kita perlu menyamakan pipeline k=7."
    )

else:
    candidate_k3, score_k3 = build_submission(distance_k3)

    assert len(candidate_k3) == 814
    assert candidate_k3[ID_COL].is_unique
    assert (
        candidate_k3[LABEL_COL] == "Unknown"
    ).sum() == N_UNKNOWN

    # Simpan ke nama file baru; champion tetap utuh.
    candidate_k3.to_csv(CANDIDATE_PATH, index=False)

    candidate_unknown = set(
        candidate_k3.loc[
            candidate_k3[LABEL_COL] == "Unknown", ID_COL
        ].astype(str)
    )

    print("BASELINE K=7 COCOK PERSIS.")
    print("Kandidat k=3 tersimpan:", os.path.exists(CANDIDATE_PATH))
    print("Lokasi:", CANDIDATE_PATH)
    print(
        "ID Unknown yang sama dengan champion:",
        len(candidate_unknown & champion_unknown),
        "/",
        N_UNKNOWN
    )
    print(
        "ID Unknown yang berubah:",
        len(candidate_unknown ^ champion_unknown)
    )

    print("\nDistribusi kelas kandidat:")
    print(candidate_k3[LABEL_COL].value_counts())

    print("\nChampion lama tetap:", CHAMPION_PATH)

MENGHITUNG JARAK BEATS
Shape jarak: (814, 7)

VALIDASI BASELINE K=7
Unknown champion: 245
Unknown rekonstruksi: 245
ID Unknown cocok: 244 / 245
Label berbeda: 2 / 814

Baris yang berbeda:


,file_id,label_champion,label_reconstructed
379,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,Unknown,Kutilang
757,c1ed2bbd36f984adb173317a631a37a2001f97689a2099...,Kutilang,Unknown



KEPUTUSAN
BASELINE BELUM COCOK. Kandidat k=3 tidak dibuat ulang.
Jangan submit dulu. Kita perlu menyamakan pipeline k=7.


In [15]:
import os
import numpy as np
import pandas as pd

# ============================================================
# 1. TEMUKAN DUA ID YANG TERTUKAR
# ============================================================

champion_path = (
    "/kaggle/working/"
    "submission_triple_v4_conf_knn50_u245.csv"
)

champion = pd.read_csv(champion_path)
champion["file_id"] = champion["file_id"].astype(str)

reconstructed = reconstructed_k7.copy()
reconstructed["file_id"] = reconstructed["file_id"].astype(str)

cmp = champion[["file_id", "label"]].merge(
    reconstructed[["file_id", "label"]],
    on="file_id",
    how="inner",
    suffixes=("_champion", "_k7")
)

diff = cmp[
    cmp["label_champion"] != cmp["label_k7"]
].copy()

print("Jumlah label berbeda:", len(diff))
display(diff)

diff_ids = diff["file_id"].tolist()

# ============================================================
# 2. PERIKSA SKOR DI SEKITAR AMBANG UNKNOWN
# ============================================================

sample_ids = sample_sub["file_id"].astype(str).to_numpy()
id_to_pos = {fid: i for i, fid in enumerate(sample_ids)}

knn_rank_k7 = pd.Series(distance_k7).rank(
    pct=True, method="average"
).to_numpy()

knn_rank_k3 = pd.Series(distance_k3).rank(
    pct=True, method="average"
).to_numpy()

confidence_ood = 1.0 - test_probs.max(axis=1)
confidence_rank = pd.Series(confidence_ood).rank(
    pct=True, method="average"
).to_numpy()

threshold_k7 = np.sort(score_k7)[-N_UNKNOWN]
threshold_k3 = np.sort(
    0.5 * confidence_rank + 0.5 * knn_rank_k3
)[-N_UNKNOWN]

rows = []

for fid in diff_ids:
    i = id_to_pos[fid]

    rows.append({
        "file_id": fid,
        "champion_label": champion.set_index(
            "file_id"
        ).loc[fid, "label"],
        "reconstructed_k7_label": reconstructed.set_index(
            "file_id"
        ).loc[fid, "label"],
        "confidence": float(test_probs[i].max()),
        "confidence_rank": float(confidence_rank[i]),
        "knn_distance_k7": float(distance_k7[i]),
        "knn_rank_k7": float(knn_rank_k7[i]),
        "fusion_k7": float(score_k7[i]),
        "distance_to_cutoff_k7": float(
            score_k7[i] - threshold_k7
        ),
        "fusion_k3": float(
            0.5 * confidence_rank[i] + 0.5 * knn_rank_k3[i]
        ),
        "distance_to_cutoff_k3": float(
            0.5 * confidence_rank[i]
            + 0.5 * knn_rank_k3[i]
            - threshold_k3
        ),
    })

print("\n=== DETAIL DUA ID ===")
display(pd.DataFrame(rows).round(8))

# ============================================================
# 3. CARI SCORE FILE LAMA UNTUK KEDUA ID
# ============================================================

search_paths = [
    "/kaggle/working/triple_v4_unknown_count_rank_scores.csv",
    "/kaggle/working/triple_v4_unknown_count_selection_manifest.csv",
    "/kaggle/working/triple_v4_conf_knn50_selected_245.csv",
    "/kaggle/input/notebooks/radithiafatir/rill-versi-baru/"
    "triple_v4_conf_knn50_selected_245.csv",
    "/kaggle/input/notebooks/radithiafatir/testttrisan/"
    "ood_rank_diagnostics_test.csv",
]

print("\n=== ARTEFAK SKOR LAMA ===")

for path in search_paths:
    if not os.path.isfile(path):
        continue

    try:
        df = pd.read_csv(path)
    except Exception as exc:
        print("Gagal membaca:", path, exc)
        continue

    if "file_id" not in df.columns:
        continue

    df["file_id"] = df["file_id"].astype(str)
    found = df[df["file_id"].isin(diff_ids)]

    print(f"\nFile: {path}")
    print("Shape:", df.shape)
    print("Kolom:", df.columns.tolist())

    if len(found):
        useful_cols = [
            c for c in found.columns
            if c == "file_id"
            or any(term in c.lower() for term in [
                "rank", "score", "distance",
                "confidence", "uncertainty", "margin"
            ])
        ]
        display(found[useful_cols])
    else:
        print("Tidak ditemukan kedua ID pada file ini.")

Jumlah label berbeda: 2


,file_id,label_champion,label_k7
379,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,Unknown,Kutilang
757,c1ed2bbd36f984adb173317a631a37a2001f97689a2099...,Kutilang,Unknown



=== DETAIL DUA ID ===


,file_id,champion_label,reconstructed_k7_label,confidence,confidence_rank,knn_distance_k7,knn_rank_k7,fusion_k7,distance_to_cutoff_k7,fusion_k3,distance_to_cutoff_k3
0,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,Unknown,Kutilang,0.654189,0.606880,0.131655,0.667076,0.636978,-0.000614,0.634521,-0.004914
1,c1ed2bbd36f984adb173317a631a37a2001f97689a2099...,Kutilang,Unknown,0.435958,0.964373,0.087186,0.310811,0.637592,0.000000,0.634521,-0.004914



=== ARTEFAK SKOR LAMA ===

File: /kaggle/working/triple_v4_unknown_count_rank_scores.csv
Shape: (814, 43)
Kolom: ['file_id', 'triple_v4_pred_known_class', 'triple_v4_confidence', 'triple_v4_uncertainty', 'triple_v4_entropy_norm', 'triple_v4_disagreement_tv', 'beats_cosine_proto_distance', 'beats_knn_cosine_distance', 'beats_diag_mahalanobis', 'rms', 'rms_db', 'zero_crossing_rate', 'spectral_flatness_median', 'spectral_flatness_mean', 'spectral_entropy_mean', 'spectral_centroid_hz', 'low_band_fraction_lt500hz', 'high_band_fraction_ge2500hz', 'spectral_flatness_p90', 'rank_uncertainty', 'rank_entropy', 'rank_disagreement_tv', 'rank_margin_low_is_ood', 'rank_beats_cosine_proto_distance', 'rank_beats_knn_cosine_distance', 'rank_beats_diag_mahalanobis', 'rank_acoustic_rms', 'rank_acoustic_rms_db', 'rank_acoustic_zero_crossing_rate', 'rank_acoustic_spectral_flatness_median', 'rank_acoustic_spectral_flatness_mean', 'rank_acoustic_spectral_entropy_mean', 'rank_acoustic_spectral_centroid_hz', 

,file_id,triple_v4_confidence,triple_v4_uncertainty,beats_cosine_proto_distance,beats_knn_cosine_distance,rank_uncertainty,rank_entropy,rank_disagreement_tv,rank_margin_low_is_ood,rank_beats_cosine_proto_distance,rank_beats_knn_cosine_distance,rank_beats_diag_mahalanobis,rank_acoustic_rms,rank_acoustic_rms_db,rank_acoustic_zero_crossing_rate,rank_acoustic_spectral_flatness_median,rank_acoustic_spectral_flatness_mean,rank_acoustic_spectral_entropy_mean,rank_acoustic_spectral_centroid_hz,rank_acoustic_low_band_fraction_lt500hz,rank_acoustic_high_band_fraction_ge2500hz,rank_acoustic_spectral_flatness_p90,ood_rank_score_triple_v4_uncertainty_disagreement_60_40,ood_rank_score_beats_embedding_only,ood_rank_score_triple_plus_beats_embedding,ood_rank_score_all_signals_exploratory,confidence_uncertainty_rank,knn_distance_rank,combined_rank_score_50_50
379,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,0.654189,0.345811,0.134151,0.131563,0.899233,0.964403,0.779847,0.892114,0.301752,0.909639,0.384447,0.336254,0.336254,0.437021,0.638554,0.705367,0.491785,0.370208,0.800657,0.384447,0.711939,0.851479,0.531946,0.735816,0.694962,0.606880,0.669533,0.638206
757,c1ed2bbd36f984adb173317a631a37a2001f97689a2099...,0.435958,0.564042,0.181354,0.086807,0.997262,0.998357,0.924425,0.985214,0.725082,0.564622,0.886637,0.891566,0.891566,0.117196,0.631435,0.567360,0.457284,0.068456,0.864184,0.043264,0.662103,0.968127,0.725447,0.865909,0.785581,0.964373,0.308354,0.636364


... (4 outputs left out)


File: /kaggle/input/notebooks/radithiafatir/rill-versi-baru/triple_v4_conf_knn50_selected_245.csv
Shape: (245, 41)
Kolom: ['file_id', 'triple_v4_pred_known_class', 'triple_v4_confidence', 'triple_v4_uncertainty', 'triple_v4_entropy_norm', 'triple_v4_disagreement_tv', 'beats_cosine_proto_distance', 'beats_knn_cosine_distance', 'beats_diag_mahalanobis', 'rms', 'rms_db', 'zero_crossing_rate', 'spectral_flatness_median', 'spectral_flatness_mean', 'spectral_entropy_mean', 'spectral_centroid_hz', 'low_band_fraction_lt500hz', 'high_band_fraction_ge2500hz', 'spectral_flatness_p90', 'rank_uncertainty', 'rank_entropy', 'rank_disagreement_tv', 'rank_margin_low_is_ood', 'rank_beats_cosine_proto_distance', 'rank_beats_knn_cosine_distance', 'rank_beats_diag_mahalanobis', 'rank_acoustic_rms', 'rank_acoustic_rms_db', 'rank_acoustic_zero_crossing_rate', 'rank_acoustic_spectral_flatness_median', 'rank_acoustic_spectral_flatness_mean', 'rank_acoustic_spectral_entropy_mean', 'rank_acoustic_spectral_centr

,file_id,triple_v4_confidence,triple_v4_uncertainty,beats_cosine_proto_distance,beats_knn_cosine_distance,rank_uncertainty,rank_entropy,rank_disagreement_tv,rank_margin_low_is_ood,rank_beats_cosine_proto_distance,rank_beats_knn_cosine_distance,rank_beats_diag_mahalanobis,rank_acoustic_rms,rank_acoustic_rms_db,rank_acoustic_zero_crossing_rate,rank_acoustic_spectral_flatness_median,rank_acoustic_spectral_flatness_mean,rank_acoustic_spectral_entropy_mean,rank_acoustic_spectral_centroid_hz,rank_acoustic_low_band_fraction_lt500hz,rank_acoustic_high_band_fraction_ge2500hz,rank_acoustic_spectral_flatness_p90,ood_rank_score_triple_v4_uncertainty_disagreement_60_40,ood_rank_score_beats_embedding_only,ood_rank_score_triple_plus_beats_embedding,ood_rank_score_all_signals_exploratory,ood_score_conf_knn50
244,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,0.654189,0.345811,0.134151,0.131563,0.899233,0.964403,0.779847,0.892114,0.301752,0.909639,0.384447,0.336254,0.336254,0.437021,0.638554,0.705367,0.491785,0.370208,0.800657,0.384447,0.711939,0.851479,0.531946,0.735816,0.694962,0.638206



File: /kaggle/input/notebooks/radithiafatir/testttrisan/ood_rank_diagnostics_test.csv
Shape: (814, 16)
Kolom: ['file_id', 'raw_uncertainty', 'rank_uncertainty', 'raw_entropy', 'rank_entropy', 'raw_disagreement', 'rank_disagreement', 'raw_centroid_distance', 'rank_centroid_distance', 'raw_knn_distance', 'rank_knn_distance', 'ood_score_confidence_only', 'ood_score_confidence_entropy', 'ood_score_confidence_disagreement', 'ood_score_embedding_distance', 'ood_score_fused_all_signals']


,file_id,raw_uncertainty,rank_uncertainty,rank_entropy,rank_disagreement,raw_centroid_distance,rank_centroid_distance,raw_knn_distance,rank_knn_distance,ood_score_confidence_only,ood_score_confidence_entropy,ood_score_confidence_disagreement,ood_score_embedding_distance,ood_score_fused_all_signals
379,86a95e87e2dd275b57b98ebb650463008abe0597a5b8ff...,0.526272,0.993976,0.978642,0.871851,0.008519,0.205367,0.009217,0.935926,0.993976,0.987842,0.959611,0.570646,0.839485
757,c1ed2bbd36f984adb173317a631a37a2001f97689a2099...,0.553870,0.996714,0.997262,0.957284,0.008855,0.234392,0.004975,0.494524,0.996714,0.996933,0.986993,0.364458,0.799261


In [16]:
# ============================================================
# REPRODUKSI CHAMPION DARI ARTEFAK LAMA + KANDIDAT K=3
# Confidence rank lama + BEATs kNN distance rank k=3
# Fusion 50:50, tepat 245 Unknown
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# ------------------------------------------------------------
# 1. PATH
# ------------------------------------------------------------

DATA_ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

SAMPLE_PATH = os.path.join(DATA_ROOT, "sample_submission.csv")
PROB_PATH = (
    "/kaggle/input/notebooks/radithiafatir/testttrisan/"
    "test_resnet_effnet_b2_convnext_pretrained_v4_55_5_40_probabilities.npy"
)
TRAIN_EMB_PATH = os.path.join(
    OUT, "beats_iter3plus_as2m_8s_train_embeddings.npy"
)
TEST_EMB_PATH = os.path.join(
    OUT, "beats_iter3plus_as2m_8s_test_embeddings.npy"
)
RANK_PATH = os.path.join(
    OUT, "triple_v4_unknown_count_rank_scores.csv"
)
SELECTED_PATH = os.path.join(
    OUT, "triple_v4_conf_knn50_selected_245.csv"
)
CHAMPION_PATH = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_u245.csv"
)
CANDIDATE_PATH = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_k3_u245.csv"
)

N_UNKNOWN = 245
K_CANDIDATE = 3

# ------------------------------------------------------------
# 2. PASTIKAN SEMUA ARTEFAK YANG DIPERLUKAN ADA
# ------------------------------------------------------------

for path in [
    SAMPLE_PATH, PROB_PATH, TRAIN_EMB_PATH, TEST_EMB_PATH,
    RANK_PATH, SELECTED_PATH, CHAMPION_PATH
]:
    assert os.path.isfile(path), f"File tidak ditemukan: {path}"

sample = pd.read_csv(SAMPLE_PATH)
rank_df = pd.read_csv(RANK_PATH)
selected_df = pd.read_csv(SELECTED_PATH)
champion = pd.read_csv(CHAMPION_PATH)

# Gunakan file_id untuk penyelarasan, bukan asumsi urutan.
for df in [sample, rank_df, selected_df, champion]:
    df["file_id"] = df["file_id"].astype(str)

assert len(sample) == 814
assert len(rank_df) == 814
assert rank_df["file_id"].is_unique

assert len(selected_df) == N_UNKNOWN
assert selected_df["file_id"].is_unique

assert len(champion) == 814
assert champion["file_id"].is_unique

# Pastikan kolom artefak yang digunakan tersedia.
required = [
    "triple_v4_pred_known_class",
    "confidence_uncertainty_rank",
    "combined_rank_score_50_50",
]
missing = [c for c in required if c not in rank_df.columns]
assert not missing, f"Kolom artefak tidak ditemukan: {missing}"

# ------------------------------------------------------------
# 3. REKONSTRUKSI CHAMPION DARI DAFTAR UNKNOWN ASLI
# ------------------------------------------------------------

# Prediksi kelas dasar dan ranking confidence diambil dari artefak lama.
base = sample[["file_id"]].merge(
    rank_df[
        [
            "file_id",
            "triple_v4_pred_known_class",
            "confidence_uncertainty_rank",
            "combined_rank_score_50_50",
        ]
    ],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert base["triple_v4_pred_known_class"].notna().all()
assert base["confidence_uncertainty_rank"].notna().all()

legacy_unknown = set(selected_df["file_id"])
champion_unknown = set(
    champion.loc[champion["label"] == "Unknown", "file_id"]
)

assert len(legacy_unknown) == N_UNKNOWN
assert legacy_unknown == champion_unknown, (
    "Daftar Unknown artefak lama tidak cocok dengan champion. "
    "Hentikan proses dan periksa file input."
)

reconstructed = base[["file_id"]].copy()
reconstructed["label"] = (
    base["triple_v4_pred_known_class"].to_numpy()
)
reconstructed.loc[
    reconstructed["file_id"].isin(legacy_unknown), "label"
] = "Unknown"

check = champion[["file_id", "label"]].merge(
    reconstructed,
    on="file_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_champion", "_reconstructed"),
)

mismatch = check[
    check["label_champion"] != check["label_reconstructed"]
]

print("=" * 65)
print("VALIDASI REKONSTRUKSI CHAMPION")
print("=" * 65)
print("Jumlah Unknown cocok:", len(legacy_unknown), "/", N_UNKNOWN)
print("Label berbeda:", len(mismatch), "/", len(champion))

if len(mismatch):
    display(mismatch.head(20))
    raise RuntimeError(
        "Rekonstruksi champion gagal. Kandidat k=3 tidak dibuat."
    )

print("LOLOS: seluruh 814 label cocok dengan champion.")

# ------------------------------------------------------------
# 4. HITUNG DISTANCE RANK DENGAN K=3
# ------------------------------------------------------------

print("\nMenghitung BEATs kNN k=3...")

train_emb = np.load(TRAIN_EMB_PATH).astype(np.float32)
test_emb = np.load(TEST_EMB_PATH).astype(np.float32)

assert train_emb.shape == (1826, 768)
assert test_emb.shape == (814, 768)

train_norm = normalize(train_emb, norm="l2")
test_norm = normalize(test_emb, norm="l2")

knn = NearestNeighbors(
    n_neighbors=K_CANDIDATE,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
knn.fit(train_norm)

distances, _ = knn.kneighbors(test_norm)
distance_k3 = distances.mean(axis=1)

# Susun jarak berdasarkan file_id agar baris selalu tepat.
distance_df = pd.DataFrame({
    "file_id": sample["file_id"].to_numpy(),
    "knn_distance_k3": distance_k3,
})

base = base.merge(
    distance_df,
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert base["knn_distance_k3"].notna().all()

# Ranking confidence tetap dari artefak lama.
# Hanya komponen ranking kNN yang diganti.
base["knn_rank_k3"] = base["knn_distance_k3"].rank(
    pct=True,
    method="average",
)

base["fusion_score_k3"] = (
    0.5 * base["confidence_uncertainty_rank"]
    + 0.5 * base["knn_rank_k3"]
)

# ------------------------------------------------------------
# 5. PILIH 245 UNKNOWN UNTUK KANDIDAT K=3
# ------------------------------------------------------------

# Stable sorting untuk konsistensi saat skor seri.
ordered = base.sort_values(
    "fusion_score_k3",
    ascending=False,
    kind="mergesort",
)

candidate_unknown = set(
    ordered.head(N_UNKNOWN)["file_id"]
)

candidate = base[["file_id", "triple_v4_pred_known_class"]].copy()
candidate = candidate.rename(
    columns={"triple_v4_pred_known_class": "label"}
)

candidate.loc[
    candidate["file_id"].isin(candidate_unknown), "label"
] = "Unknown"

assert len(candidate) == 814
assert candidate["file_id"].is_unique
assert (candidate["label"] == "Unknown").sum() == N_UNKNOWN

# Pastikan urutan dan ID sesuai template submission.
candidate = sample[["file_id"]].merge(
    candidate,
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert candidate["label"].notna().all()

# Simpan ke nama baru, jangan pernah timpa champion.
candidate.to_csv(CANDIDATE_PATH, index=False)

# ------------------------------------------------------------
# 6. LAPORAN PERBANDINGAN
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("KANDIDAT K=3 BERHASIL DIBUAT")
print("=" * 65)

print("Lokasi:", CANDIDATE_PATH)
print("File ada:", os.path.isfile(CANDIDATE_PATH))
print("Jumlah baris:", len(candidate))
print(
    "Jumlah Unknown:",
    int((candidate["label"] == "Unknown").sum())
)
print(
    "ID Unknown yang tetap sama:",
    len(champion_unknown & candidate_unknown),
    "/", N_UNKNOWN
)
print(
    "ID Unknown yang berubah:",
    len(champion_unknown ^ candidate_unknown)
)

print("\nDistribusi kelas kandidat:")
print(candidate["label"].value_counts())

print("\nChampion tetap tersimpan:", os.path.isfile(CHAMPION_PATH))

VALIDASI REKONSTRUKSI CHAMPION
Jumlah Unknown cocok: 245 / 245
Label berbeda: 0 / 814
LOLOS: seluruh 814 label cocok dengan champion.

Menghitung BEATs kNN k=3...

KANDIDAT K=3 BERHASIL DIBUAT
Lokasi: /kaggle/working/submission_triple_v4_conf_knn50_k3_u245.csv
File ada: True
Jumlah baris: 814
Jumlah Unknown: 245
ID Unknown yang tetap sama: 239 / 245
ID Unknown yang berubah: 12

Distribusi kelas kandidat:
label
Kutilang       262
Unknown        245
Bondol Jawa    230
Cendrawasih     77
Name: count, dtype: int64

Champion tetap tersimpan: True


In [18]:
# ============================================================
# FINAL LOCO COMPARISON: BEATS KNN K=3, K=5, K=7
# Kompetisi: SCHEMATICS NPC DSC 2026
#
# Tujuan:
#   Membandingkan kualitas ranking pseudo-Unknown
#   menggunakan confidence OOD dan BEATs kNN 50:50.
#
# Tidak melatih ulang model audio.
# Tidak mengubah file submission champion.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, average_precision_score

# ============================================================
# 1. KONFIGURASI
# ============================================================

DATA_ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

CLASS_NAMES = [
    "Bondol Jawa",
    "Cendrawasih",
    "Kutilang"
]

K_VALUES = [3, 5, 7]
MAX_K = max(K_VALUES)
SEED = 16

# ============================================================
# 2. CARI FILE OTOMATIS
# ============================================================

def locate_file(filename, preferred_paths=None, search_roots=None):
    """
    Mencari file dari beberapa lokasi yang mungkin.
    Mengembalikan path yang ditemukan atau memberi error.
    """

    preferred_paths = preferred_paths or []
    search_roots = search_roots or [
        "/kaggle/working",
        "/kaggle/input"
    ]

    # Periksa lokasi yang sudah diketahui terlebih dahulu.
    for path in preferred_paths:
        if path and os.path.isfile(path):
            return path

    # Cari otomatis di direktori yang tersedia.
    for root in search_roots:
        if not os.path.isdir(root):
            continue

        matches = glob.glob(
            os.path.join(root, "**", filename),
            recursive=True
        )

        if matches:
            matches.sort(key=lambda p: (len(p), p))
            return matches[0]

    raise FileNotFoundError(
        f"File tidak ditemukan: {filename}\n"
        f"Lokasi yang diperiksa: {preferred_paths}"
    )


metadata_path = os.path.join(
    DATA_ROOT, "train", "metadata.csv"
)

if not os.path.isfile(metadata_path):
    metadata_path = locate_file(
        "metadata.csv",
        search_roots=[DATA_ROOT]
    )

fold_path = locate_file(
    "bird_group_folds_seed16.csv",
    preferred_paths=[
        os.path.join(OUT, "bird_group_folds_seed16.csv"),
        "/kaggle/input/notebooks/radithiafatir/"
        "testttrisan/bird_group_folds_seed16.csv",
        "/kaggle/input/notebooks/radithiafatir/"
        "rill-versi-baru/bird_group_folds_seed16.csv",
    ]
)

embedding_path = locate_file(
    "beats_iter3plus_as2m_8s_train_embeddings.npy",
    preferred_paths=[
        os.path.join(
            OUT,
            "beats_iter3plus_as2m_8s_train_embeddings.npy"
        ),
        "/kaggle/input/notebooks/radithiafatir/"
        "rill-versi-baru/beats_iter3plus_as2m_8s_train_embeddings.npy",
    ]
)

print("=" * 70)
print("FILE INPUT YANG DIGUNAKAN")
print("=" * 70)
print("Metadata :", metadata_path)
print("Fold CSV :", fold_path)
print("Embedding:", embedding_path)

# ============================================================
# 3. BACA METADATA, FOLD, DAN EMBEDDING
# ============================================================

meta = pd.read_csv(metadata_path)
fold_df = pd.read_csv(fold_path)
emb = np.load(embedding_path).astype(np.float32)

required_meta_cols = [
    "file_id",
    "label",
    "source_group"
]

missing_meta_cols = [
    c for c in required_meta_cols if c not in meta.columns
]

if missing_meta_cols:
    raise ValueError(
        f"Kolom metadata tidak ditemukan: {missing_meta_cols}"
    )

if not {"file_id", "fold"}.issubset(fold_df.columns):
    raise ValueError(
        "File fold harus berisi kolom 'file_id' dan 'fold'. "
        f"Kolom ditemukan: {fold_df.columns.tolist()}"
    )

meta["file_id"] = meta["file_id"].astype(str)
fold_df["file_id"] = fold_df["file_id"].astype(str)

if not meta["file_id"].is_unique:
    raise ValueError("Ada file_id duplikat pada metadata.")

if not fold_df["file_id"].is_unique:
    raise ValueError("Ada file_id duplikat pada file fold.")

# Pasangkan fold berdasarkan file_id, bukan urutan baris.
meta = meta.merge(
    fold_df[["file_id", "fold"]],
    on="file_id",
    how="left",
    validate="one_to_one"
)

if meta["fold"].isna().any():
    missing_ids = meta.loc[
        meta["fold"].isna(), "file_id"
    ].head(10).tolist()

    raise ValueError(
        "Ada sampel yang tidak memiliki fold. "
        f"Contoh ID: {missing_ids}"
    )

meta["fold"] = meta["fold"].astype(int)

label_to_idx = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

label_series = meta["label"].map(label_to_idx)

if label_series.isna().any():
    bad_labels = meta.loc[
        label_series.isna(), "label"
    ].unique().tolist()

    raise ValueError(
        f"Label tidak sesuai CLASS_NAMES: {bad_labels}"
    )

y = label_series.to_numpy(dtype=int)
folds = meta["fold"].to_numpy()

if emb.ndim != 2:
    raise ValueError(
        f"Embedding harus dua dimensi, diperoleh {emb.shape}"
    )

if emb.shape != (len(meta), 768):
    raise ValueError(
        f"Shape embedding tidak sesuai: {emb.shape}; "
        f"metadata memiliki {len(meta)} baris."
    )

if not np.isfinite(emb).all():
    raise ValueError("Embedding memiliki NaN atau nilai tak hingga.")

# ============================================================
# 4. VALIDASI SOURCE_GROUP DAN FOLD
# ============================================================

group_fold_counts = (
    meta.groupby("source_group")["fold"].nunique()
)

group_leakage = int((group_fold_counts > 1).sum())

if group_leakage:
    raise ValueError(
        f"Terdapat {group_leakage} source_group yang muncul "
        "di lebih dari satu fold."
    )

fold_values = sorted(meta["fold"].unique())

if len(fold_values) < 2:
    raise ValueError("Jumlah fold tidak mencukupi.")

print("\n" + "=" * 70)
print("VALIDASI DATA")
print("=" * 70)
print("Jumlah train:", len(meta))
print("Shape embedding:", emb.shape)
print("Fold counts:", meta["fold"].value_counts().sort_index().to_dict())
print("Distribusi kelas:", meta["label"].value_counts().to_dict())
print("Jumlah fold:", len(fold_values))
print("Source-group leakage:", group_leakage)

# ============================================================
# 5. SIMULASI LOCO
# ============================================================
# Untuk setiap kelas:
#   - Tahan satu kelas sebagai pseudo-Unknown.
#   - Latih classifier hanya pada dua kelas yang lain.
#   - Validasi pada fold yang tidak dipakai untuk training.
#   - Ukur apakah jarak kNN dan confidence dapat meranking
#     sampel pseudo-Unknown di atas sampel known.
#
# Classifier hanya dipakai sebagai alat evaluasi.
# Tidak ada retraining model audio/embedding.

records = []

for heldout in range(len(CLASS_NAMES)):

    heldout_name = CLASS_NAMES[heldout]

    known_classes = [
        c for c in range(len(CLASS_NAMES))
        if c != heldout
    ]

    print("\n" + "=" * 70)
    print("LOCO PSEUDO-UNKNOWN:", heldout_name)
    print("=" * 70)

    for fold in fold_values:

        # Fold validasi tidak boleh masuk ke training.
        # Sampel kelas heldout juga dikeluarkan dari training.
        train_idx = np.flatnonzero(
            (folds != fold) & (y != heldout)
        )

        valid_idx = np.flatnonzero(
            folds == fold
        )

        if len(train_idx) == 0 or len(valid_idx) == 0:
            raise ValueError(
                f"Training/validasi kosong pada fold {fold}."
            )

        actual_train_classes = set(
            np.unique(y[train_idx]).tolist()
        )

        if actual_train_classes != set(known_classes):
            raise ValueError(
                f"Kelas training tidak lengkap. "
                f"Fold={fold}, heldout={heldout_name}, "
                f"kelas training={actual_train_classes}"
            )

        x_train = emb[train_idx]
        x_valid = emb[valid_idx]
        y_train = y[train_idx]

        # ----------------------------------------------------
        # 5A. CONFIDENCE OOD
        # ----------------------------------------------------

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.03,
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED
            )
        )

        clf.fit(x_train, y_train)
        probabilities = clf.predict_proba(x_valid)

        # Confidence rendah => skor OOD lebih tinggi.
        confidence_ood = (
            1.0 - probabilities.max(axis=1)
        )

        # ----------------------------------------------------
        # 5B. DISTANCE KNN
        # ----------------------------------------------------

        x_train_norm = normalize(
            x_train, norm="l2"
        )

        x_valid_norm = normalize(
            x_valid, norm="l2"
        )

        n_neighbors = min(MAX_K, len(train_idx))

        knn = NearestNeighbors(
            n_neighbors=n_neighbors,
            metric="cosine",
            algorithm="brute",
            n_jobs=-1
        )

        knn.fit(x_train_norm)

        distances, _ = knn.kneighbors(
            x_valid_norm
        )

        # Target positif berarti sampel berasal dari kelas
        # yang sedang ditahan, sehingga dianggap pseudo-Unknown.
        is_pseudo_unknown = (
            y[valid_idx] == heldout
        ).astype(int)

        # ----------------------------------------------------
        # 5C. CATAT SEMUA NILAI UNTUK SETIAP K
        # ----------------------------------------------------

        for j, sample_idx in enumerate(valid_idx):

            for k in K_VALUES:

                knn_distance = float(
                    distances[j, :k].mean()
                )

                records.append({
                    "heldout_class": heldout_name,
                    "fold": int(fold),
                    "sample_idx": int(sample_idx),
                    "is_pseudo_unknown": int(
                        is_pseudo_unknown[j]
                    ),
                    "confidence_ood": float(
                        confidence_ood[j]
                    ),
                    "knn_distance": knn_distance,
                    "k": int(k)
                })

        print(
            f"Fold {fold}: "
            f"train={len(train_idx)}, "
            f"validasi={len(valid_idx)}, "
            f"tetangga dihitung={n_neighbors}"
        )

detail = pd.DataFrame(records)

if detail.empty:
    raise RuntimeError("Simulasi LOCO tidak menghasilkan data.")

# ============================================================
# 6. EVALUASI KNN DAN FUSION 50:50
# ============================================================

results = []
per_class_results = []

for k in K_VALUES:

    detail_k = detail[
        detail["k"] == k
    ]

    class_metrics = []

    for class_name in CLASS_NAMES:

        part = detail_k[
            detail_k["heldout_class"] == class_name
        ].copy()

        target = part[
            "is_pseudo_unknown"
        ].to_numpy(dtype=int)

        if len(np.unique(target)) < 2:
            raise ValueError(
                f"Target tidak memiliki kedua kelas untuk "
                f"evaluasi {class_name}, k={k}."
            )

        # Rank dihitung pada sampel validasi LOCO untuk
        # kelas heldout yang sama, menggabungkan seluruh fold.
        confidence_rank = part[
            "confidence_ood"
        ].rank(
            pct=True,
            method="average"
        ).to_numpy()

        knn_rank = part[
            "knn_distance"
        ].rank(
            pct=True,
            method="average"
        ).to_numpy()

        fusion_score = (
            0.5 * confidence_rank
            + 0.5 * knn_rank
        )

        auc_knn = roc_auc_score(
            target,
            part["knn_distance"].to_numpy()
        )

        auc_fusion = roc_auc_score(
            target,
            fusion_score
        )

        ap_fusion = average_precision_score(
            target,
            fusion_score
        )

        class_metrics.append({
            "heldout_class": class_name,
            "auc_knn": auc_knn,
            "auc_fusion": auc_fusion,
            "ap_fusion": ap_fusion
        })

        per_class_results.append({
            "k": k,
            "heldout_class": class_name,
            "auc_knn": auc_knn,
            "auc_fusion": auc_fusion,
            "ap_fusion": ap_fusion
        })

    cm = pd.DataFrame(class_metrics)

    row = {
        "k": k,
        "macro_auc_knn": cm["auc_knn"].mean(),
        "macro_auc_fusion": cm["auc_fusion"].mean(),
        "macro_ap_fusion": cm["ap_fusion"].mean()
    }

    for _, class_row in cm.iterrows():
        suffix = (
            class_row["heldout_class"]
            .replace(" ", "_")
            .lower()
        )

        row[f"auc_fusion_{suffix}"] = (
            class_row["auc_fusion"]
        )

    results.append(row)

# ============================================================
# 7. SIMPAN DAN TAMPILKAN HASIL
# ============================================================

result_df = pd.DataFrame(results).sort_values(
    "macro_auc_fusion",
    ascending=False
).reset_index(drop=True)

per_class_df = pd.DataFrame(
    per_class_results
)

summary_path = os.path.join(
    OUT,
    "beats_knn_k3_k5_k7_loco_comparison.csv"
)

per_class_path = os.path.join(
    OUT,
    "beats_knn_k3_k5_k7_loco_per_class.csv"
)

result_df.to_csv(summary_path, index=False)
per_class_df.to_csv(per_class_path, index=False)

print("\n" + "=" * 70)
print("HASIL AKHIR: K=3 VS K=5 VS K=7")
print("=" * 70)

display(result_df.round(5))

print("\n" + "=" * 70)
print("HASIL TERPERINCI PER KELAS")
print("=" * 70)

display(
    per_class_df.sort_values(
        ["heldout_class", "k"]
    ).round(5)
)

print("\nFile ringkasan:", summary_path)
print("File per kelas:", per_class_path)
print("\nEksperimen selesai. Tidak ada submission yang diubah.")

FILE INPUT YANG DIGUNAKAN
Metadata : /kaggle/input/competitions/schematics-2026/train/metadata.csv
Fold CSV : /kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
Embedding: /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy

VALIDASI DATA
Jumlah train: 1826
Shape embedding: (1826, 768)
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}
Distribusi kelas: {'Bondol Jawa': 1124, 'Kutilang': 619, 'Cendrawasih': 83}
Jumlah fold: 4
Source-group leakage: 0

LOCO PSEUDO-UNKNOWN: Bondol Jawa
Fold 0: train=535, validasi=284, tetangga dihitung=7
Fold 1: train=521, validasi=495, tetangga dihitung=7
Fold 2: train=543, validasi=553, tetangga dihitung=7
Fold 3: train=507, validasi=494, tetangga dihitung=7

LOCO PSEUDO-UNKNOWN: Cendrawasih
Fold 0: train=1494, validasi=284, tetangga dihitung=7
Fold 1: train=1251, validasi=495, tetangga dihitung=7
Fold 2: train=1206, validasi=553, tetangga dihitung=7
Fold 3: train=1278, validasi=494, tetangga dihitung=7

LOCO PSEUDO-UNKNOWN

,k,macro_auc_knn,macro_auc_fusion,macro_ap_fusion,auc_fusion_bondol_jawa,auc_fusion_cendrawasih,auc_fusion_kutilang
0,3,0.72236,0.78556,0.53097,0.78187,0.87230,0.70250
1,5,0.71845,0.78356,0.52697,0.77834,0.87506,0.69727
2,7,0.71592,0.78211,0.52605,0.77569,0.87676,0.69388



HASIL TERPERINCI PER KELAS


,k,heldout_class,auc_knn,auc_fusion,ap_fusion
0,3,Bondol Jawa,0.65615,0.78187,0.82657
3,5,Bondol Jawa,0.65201,0.77834,0.82380
6,7,Bondol Jawa,0.64886,0.77569,0.82181
1,3,Cendrawasih,0.89380,0.87230,0.27067
4,5,Cendrawasih,0.89526,0.87506,0.26800
7,7,Cendrawasih,0.89627,0.87676,0.27130
2,3,Kutilang,0.61713,0.70250,0.49568
5,5,Kutilang,0.60808,0.69727,0.48911
8,7,Kutilang,0.60262,0.69388,0.48504



File ringkasan: /kaggle/working/beats_knn_k3_k5_k7_loco_comparison.csv
File per kelas: /kaggle/working/beats_knn_k3_k5_k7_loco_per_class.csv

Eksperimen selesai. Tidak ada submission yang diubah.


In [19]:
import os
import pandas as pd

candidate_path = (
    "/kaggle/working/"
    "submission_triple_v4_conf_knn50_k3_u245.csv"
)

champion_path = (
    "/kaggle/working/"
    "submission_triple_v4_conf_knn50_u245.csv"
)

sample_path = (
    "/kaggle/input/competitions/"
    "schematics-2026/sample_submission.csv"
)

assert os.path.isfile(candidate_path), "Kandidat tidak ditemukan."
assert os.path.isfile(champion_path), "Champion tidak ditemukan."

candidate = pd.read_csv(candidate_path)
champion = pd.read_csv(champion_path)
sample = pd.read_csv(sample_path)

# Validasi format dan urutan ID.
assert list(candidate.columns) == list(sample.columns)
assert len(candidate) == len(sample) == 814
assert candidate["file_id"].is_unique
assert candidate["file_id"].astype(str).tolist() == (
    sample["file_id"].astype(str).tolist()
)

# Validasi kelas dan jumlah Unknown.
allowed_labels = {
    "Bondol Jawa", "Cendrawasih", "Kutilang", "Unknown"
}

assert set(candidate["label"].unique()).issubset(allowed_labels)
assert (candidate["label"] == "Unknown").sum() == 245

old_unknown = set(
    champion.loc[champion["label"] == "Unknown", "file_id"]
)
new_unknown = set(
    candidate.loc[candidate["label"] == "Unknown", "file_id"]
)

print("FORMAT VALID")
print("Jumlah baris:", len(candidate))
print("\nDistribusi label:")
print(candidate["label"].value_counts())
print("\nUnknown tetap sama:", len(old_unknown & new_unknown), "/ 245")
print("Unknown berubah:", len(old_unknown ^ new_unknown))
print("\nFile siap diunggah:", candidate_path)

FORMAT VALID
Jumlah baris: 814

Distribusi label:
label
Kutilang       262
Unknown        245
Bondol Jawa    230
Cendrawasih     77
Name: count, dtype: int64

Unknown tetap sama: 239 / 245
Unknown berubah: 12

File siap diunggah: /kaggle/working/submission_triple_v4_conf_knn50_k3_u245.csv


In [20]:
# ============================================================
# LOCO: GROUP BOOTSTRAP + FUSION WEIGHT SWEEP
#
# Dibandingkan:
#   k = 3, 5, 7
#   Bobot confidence = 0.3, 0.4, 0.5, 0.6, 0.7
#   Bobot BEATs kNN = 1 - bobot confidence
#
# Bootstrap berpasangan berdasarkan source_group:
#   Membandingkan Macro AUC k=3 dan k=7 pada bobot 50:50.
#
# Tidak melatih ulang model audio.
# Tidak mengubah submission.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score

# ============================================================
# 1. CEK DATA DARI EKSPERIMEN LOCO SEBELUMNYA
# ============================================================

required_detail = {
    "heldout_class",
    "sample_idx",
    "is_pseudo_unknown",
    "confidence_ood",
    "knn_distance",
    "k",
}

if "detail" not in globals() or "meta" not in globals():
    raise RuntimeError(
        "Variabel detail/meta tidak ditemukan. "
        "Jalankan cell LOCO k=3, 5, 7 terlebih dahulu."
    )

missing = required_detail - set(detail.columns)

if missing:
    raise ValueError(f"Kolom detail tidak lengkap: {missing}")

if "source_group" not in meta.columns:
    raise ValueError("Kolom source_group tidak ditemukan di meta.")

K_VALUES = [3, 5, 7]
CONF_WEIGHTS = [0.3, 0.4, 0.5, 0.6, 0.7]
CLASS_NAMES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 2026
OUT = "/kaggle/working"

detail = detail.copy()
meta = meta.reset_index(drop=True).copy()

available_k = set(detail["k"].astype(int).unique())

if not set(K_VALUES).issubset(available_k):
    raise ValueError(
        f"Nilai k tidak lengkap: {available_k}"
    )

# Hubungkan setiap sampel ke source_group berdasarkan sample_idx.
group_lookup = dict(
    enumerate(meta["source_group"].astype(str).to_numpy())
)

detail["sample_idx"] = detail["sample_idx"].astype(int)
detail["source_group"] = detail["sample_idx"].map(group_lookup)

if detail["source_group"].isna().any():
    raise ValueError("Ada sampel LOCO tanpa source_group.")

detail["k"] = detail["k"].astype(int)
detail["is_pseudo_unknown"] = (
    detail["is_pseudo_unknown"].astype(int)
)

print("=" * 70)
print("VALIDASI")
print("=" * 70)
print("Jumlah baris detail:", len(detail))
print("Jumlah source_group:", detail["source_group"].nunique())
print("Nilai k:", sorted(available_k))
print("Bobot confidence:", CONF_WEIGHTS)
print("Bootstrap:", N_BOOTSTRAP)

# ============================================================
# 2. HITUNG RANK DAN EVALUASI GRID BOBOT
# ============================================================

score_parts = {}
summary_rows = []
class_rows = []

for k in K_VALUES:

    detail_k = detail[detail["k"] == k]

    for class_name in CLASS_NAMES:

        part = detail_k[
            detail_k["heldout_class"] == class_name
        ].copy()

        target = part["is_pseudo_unknown"].to_numpy(dtype=int)

        if len(np.unique(target)) != 2:
            raise ValueError(
                f"Target LOCO tidak memiliki dua kelas: {class_name}"
            )

        # Ranking per skenario heldout class, sama seperti LOCO sebelumnya.
        conf_rank = part["confidence_ood"].rank(
            pct=True,
            method="average"
        ).to_numpy()

        knn_rank = part["knn_distance"].rank(
            pct=True,
            method="average"
        ).to_numpy()

        for weight_conf in CONF_WEIGHTS:

            weight_knn = 1.0 - weight_conf

            fusion = (
                weight_conf * conf_rank
                + weight_knn * knn_rank
            )

            auc = roc_auc_score(target, fusion)
            ap = average_precision_score(target, fusion)

            class_rows.append({
                "k": k,
                "weight_confidence": weight_conf,
                "weight_knn": weight_knn,
                "heldout_class": class_name,
                "auc_fusion": auc,
                "ap_fusion": ap,
            })

            key = (k, weight_conf)

            if key not in score_parts:
                score_parts[key] = []

            score_parts[key].append(
                pd.DataFrame({
                    "heldout_class": class_name,
                    "source_group": part["source_group"].to_numpy(),
                    "is_pseudo_unknown": target,
                    "fusion_score": fusion,
                })
            )

class_df = pd.DataFrame(class_rows)

for (k, weight_conf), frames in score_parts.items():

    per_combo = class_df[
        (class_df["k"] == k)
        & (class_df["weight_confidence"] == weight_conf)
    ]

    summary_rows.append({
        "k": k,
        "weight_confidence": weight_conf,
        "weight_knn": 1.0 - weight_conf,
        "macro_auc_fusion": per_combo["auc_fusion"].mean(),
        "macro_ap_fusion": per_combo["ap_fusion"].mean(),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    "macro_auc_fusion",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("RANKING KOMBINASI K DAN BOBOT")
print("=" * 70)
display(summary_df.round(5))

print("\n" + "=" * 70)
print("HASIL PER KELAS")
print("=" * 70)

display(
    class_df.sort_values(
        ["k", "weight_confidence", "heldout_class"]
    ).round(5)
)

# ============================================================
# 3. PAIRED SOURCE-GROUP BOOTSTRAP: k=3 VS k=7
#    BOBOT CONFIDENCE 50%, KNN 50%
# ============================================================

print("\n" + "=" * 70)
print("PAIRED SOURCE-GROUP BOOTSTRAP")
print("=" * 70)

WEIGHT_FOR_BOOTSTRAP = 0.5

df_k3 = pd.concat(
    score_parts[(3, WEIGHT_FOR_BOOTSTRAP)],
    ignore_index=True
)

df_k7 = pd.concat(
    score_parts[(7, WEIGHT_FOR_BOOTSTRAP)],
    ignore_index=True
)

# Gunakan himpunan source_group yang sama untuk dua konfigurasi.
all_groups = np.array(
    sorted(detail["source_group"].unique()),
    dtype=object
)

rng = np.random.default_rng(BOOTSTRAP_SEED)

def weighted_macro_auc(frame, group_multiplicity):

    aucs = []

    for class_name in CLASS_NAMES:

        part = frame[
            frame["heldout_class"] == class_name
        ]

        target = part["is_pseudo_unknown"].to_numpy(dtype=int)
        score = part["fusion_score"].to_numpy(dtype=float)

        weights = (
            part["source_group"]
            .map(group_multiplicity)
            .fillna(0)
            .to_numpy(dtype=float)
        )

        keep = weights > 0

        target = target[keep]
        score = score[keep]
        weights = weights[keep]

        if len(np.unique(target)) < 2:
            return np.nan

        auc = roc_auc_score(
            target,
            score,
            sample_weight=weights
        )

        aucs.append(auc)

    return float(np.mean(aucs))


# Estimasi titik pada keseluruhan data LOCO.
full_counts = {g: 1 for g in all_groups}

auc3_full = weighted_macro_auc(df_k3, full_counts)
auc7_full = weighted_macro_auc(df_k7, full_counts)
delta_full = auc3_full - auc7_full

delta_boot = []

for b in range(N_BOOTSTRAP):

    # Ambil source_group dengan pengembalian.
    sampled_groups = rng.choice(
        all_groups,
        size=len(all_groups),
        replace=True
    )

    # Banyaknya kemunculan setiap grup menjadi sample weight.
    counts = (
        pd.Series(sampled_groups)
        .value_counts()
        .to_dict()
    )

    # Pasangan k=3 dan k=7 menggunakan grup bootstrap yang sama.
    auc3 = weighted_macro_auc(df_k3, counts)
    auc7 = weighted_macro_auc(df_k7, counts)

    if np.isfinite(auc3) and np.isfinite(auc7):
        delta_boot.append(auc3 - auc7)

delta_boot = np.asarray(delta_boot, dtype=float)

if len(delta_boot) < 100:
    raise RuntimeError(
        f"Hanya {len(delta_boot)} bootstrap valid. "
        "Periksa jumlah source_group dan distribusi kelas."
    )

ci_low, ci_high = np.percentile(
    delta_boot, [2.5, 97.5]
)

print(f"Macro AUC k=3 : {auc3_full:.6f}")
print(f"Macro AUC k=7 : {auc7_full:.6f}")
print(f"Delta (k3-k7): {delta_full:+.6f}")
print(f"Bootstrap valid: {len(delta_boot)} / {N_BOOTSTRAP}")
print(f"95% CI delta  : [{ci_low:+.6f}, {ci_high:+.6f}]")
print(
    "Proporsi bootstrap delta > 0:",
    round(float(np.mean(delta_boot > 0)), 4)
)

if ci_low <= 0 <= ci_high:
    print(
        "\nINTERPRETASI: CI mencakup nol. "
        "Belum ada bukti kuat bahwa k=3 lebih baik daripada k=7 "
        "pada evaluasi LOCO ini."
    )
else:
    print(
        "\nINTERPRETASI: CI tidak mencakup nol pada bootstrap ini. "
        "Perbedaan LOCO relatif konsisten, tetapi belum menjamin "
        "hasil pada Unknown asli di leaderboard."
    )

# ============================================================
# 4. SIMPAN HASIL KE CSV
# ============================================================

summary_path = os.path.join(
    OUT, "beats_knn_weight_grid_loco.csv"
)

class_path = os.path.join(
    OUT, "beats_knn_weight_grid_loco_per_class.csv"
)

bootstrap_path = os.path.join(
    OUT, "beats_knn_k3_vs_k7_group_bootstrap.csv"
)

summary_df.to_csv(summary_path, index=False)
class_df.to_csv(class_path, index=False)

pd.DataFrame({
    "delta_auc_macro_k3_minus_k7": delta_boot
}).to_csv(bootstrap_path, index=False)

print("\nFile tersimpan:")
print(summary_path)
print(class_path)
print(bootstrap_path)

VALIDASI
Jumlah baris detail: 16434
Jumlah source_group: 15
Nilai k: [np.int64(3), np.int64(5), np.int64(7)]
Bobot confidence: [0.3, 0.4, 0.5, 0.6, 0.7]
Bootstrap: 1000

RANKING KOMBINASI K DAN BOBOT


,k,weight_confidence,weight_knn,macro_auc_fusion,macro_ap_fusion
0,3,0.5,0.5,0.78556,0.53097
1,5,0.5,0.5,0.78356,0.52697
2,7,0.5,0.5,0.78211,0.52605
3,3,0.4,0.6,0.78152,0.53204
4,5,0.4,0.6,0.77903,0.52957
5,7,0.4,0.6,0.77718,0.52683
6,3,0.6,0.4,0.77703,0.52405
7,5,0.6,0.4,0.77550,0.52195
8,7,0.6,0.4,0.77453,0.52077
9,3,0.3,0.7,0.77048,0.53380



HASIL PER KELAS


,k,weight_confidence,weight_knn,heldout_class,auc_fusion,ap_fusion
0,3,0.3,0.7,Bondol Jawa,0.73497,0.78606
5,3,0.3,0.7,Cendrawasih,0.90527,0.34091
10,3,0.3,0.7,Kutilang,0.67121,0.47444
1,3,0.4,0.6,Bondol Jawa,0.76060,0.80809
6,3,0.4,0.6,Cendrawasih,0.89538,0.30154
11,3,0.4,0.6,Kutilang,0.68859,0.48650
2,3,0.5,0.5,Bondol Jawa,0.78187,0.82657
7,3,0.5,0.5,Cendrawasih,0.87230,0.27067
12,3,0.5,0.5,Kutilang,0.70250,0.49568
3,3,0.6,0.4,Bondol Jawa,0.79516,0.83863



PAIRED SOURCE-GROUP BOOTSTRAP
Macro AUC k=3 : 0.785556
Macro AUC k=7 : 0.782112
Delta (k3-k7): +0.003444
Bootstrap valid: 980 / 1000
95% CI delta  : [-0.000568, +0.005953]
Proporsi bootstrap delta > 0: 0.9622

INTERPRETASI: CI mencakup nol. Belum ada bukti kuat bahwa k=3 lebih baik daripada k=7 pada evaluasi LOCO ini.

File tersimpan:
/kaggle/working/beats_knn_weight_grid_loco.csv
/kaggle/working/beats_knn_weight_grid_loco_per_class.csv
/kaggle/working/beats_knn_k3_vs_k7_group_bootstrap.csv


In [21]:
# ============================================================
# EVALUASI PSEUDO-UNKNOWN PADA PROPORSI SELEKSI 30.1%
#
# Menguji:
#   k = 3, 5, 7
#   Bobot confidence = 30%, 40%, 50%, 60%, 70%
#   Bobot kNN = 1 - bobot confidence
#
# Metrik:
#   Precision, Recall, F1 untuk deteksi pseudo-Unknown
#   pada kuota seleksi yang mendekati 245/814.
#
# Offline saja — tidak mengubah submission.
# ============================================================

import numpy as np
import pandas as pd
import os

# ============================================================
# 1. CEK INPUT
# ============================================================

required_cols = {
    "k",
    "heldout_class",
    "sample_idx",
    "is_pseudo_unknown",
    "confidence_ood",
    "knn_distance",
}

if "detail" not in globals():
    raise RuntimeError(
        "Variabel detail tidak ditemukan. "
        "Jalankan eksperimen LOCO terlebih dahulu."
    )

missing = required_cols - set(detail.columns)

if missing:
    raise ValueError(f"Kolom belum lengkap: {missing}")

K_VALUES = [3, 5, 7]
CONF_WEIGHTS = [0.3, 0.4, 0.5, 0.6, 0.7]
CLASS_NAMES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]

TEST_UNKNOWN_FRACTION = 245 / 814

# ============================================================
# 2. HITUNG METRIK PADA KUOTA 30.1%
# ============================================================

class_results = []

for k in K_VALUES:

    detail_k = detail[detail["k"] == k]

    for heldout_class in CLASS_NAMES:

        part = detail_k[
            detail_k["heldout_class"] == heldout_class
        ].copy()

        # Satu prediksi OOF per sampel dalam skenario heldout ini.
        if part["sample_idx"].duplicated().any():
            raise ValueError(
                f"Ada sampel duplikat: k={k}, "
                f"heldout={heldout_class}"
            )

        y_true = part["is_pseudo_unknown"].to_numpy(
            dtype=int
        )

        conf_rank = part["confidence_ood"].rank(
            pct=True,
            method="average"
        ).to_numpy()

        knn_rank = part["knn_distance"].rank(
            pct=True,
            method="average"
        ).to_numpy()

        # Jumlah sampel yang dipilih mendekati 245/814.
        n_select = int(
            np.ceil(TEST_UNKNOWN_FRACTION * len(part))
        )

        n_select = min(n_select, len(part))

        for w_conf in CONF_WEIGHTS:

            w_knn = 1.0 - w_conf

            fusion = (
                w_conf * conf_rank
                + w_knn * knn_rank
            )

            selected_idx = np.argsort(
                -fusion,
                kind="stable"
            )[:n_select]

            predicted_unknown = np.zeros(
                len(part), dtype=bool
            )
            predicted_unknown[selected_idx] = True

            tp = int(
                np.sum(predicted_unknown & (y_true == 1))
            )
            fp = int(
                np.sum(predicted_unknown & (y_true == 0))
            )
            fn = int(
                np.sum((~predicted_unknown) & (y_true == 1))
            )

            precision = tp / max(tp + fp, 1)
            recall = tp / max(tp + fn, 1)

            f1 = (
                2 * precision * recall
                / max(precision + recall, 1e-12)
            )

            class_results.append({
                "k": k,
                "weight_confidence": w_conf,
                "weight_knn": w_knn,
                "heldout_class": heldout_class,
                "n_samples": len(part),
                "n_selected": n_select,
                "true_unknown_rate": y_true.mean(),
                "precision_at_30pct": precision,
                "recall_at_30pct": recall,
                "f1_at_30pct": f1,
            })

class_df = pd.DataFrame(class_results)

# ============================================================
# 3. AGREGASI MACRO ANTAR KELAS HELDOUT
# ============================================================

summary_df = (
    class_df.groupby(
        ["k", "weight_confidence", "weight_knn"],
        as_index=False
    )
    .agg(
        macro_precision_at_30pct=(
            "precision_at_30pct", "mean"
        ),
        macro_recall_at_30pct=(
            "recall_at_30pct", "mean"
        ),
        macro_f1_at_30pct=(
            "f1_at_30pct", "mean"
        ),
    )
    .sort_values(
        "macro_f1_at_30pct",
        ascending=False
    )
    .reset_index(drop=True)
)

# ============================================================
# 4. TAMPILKAN PERBANDINGAN
# ============================================================

print("=" * 75)
print("RANKING BOBOT BERDASARKAN MACRO F1 PSEUDO-UNKNOWN @ 30.1%")
print("=" * 75)

display(summary_df.round(5))

print("\n" + "=" * 75)
print("METRIK PER KELAS HELDOUT")
print("=" * 75)

display(
    class_df.sort_values(
        ["k", "weight_confidence", "heldout_class"]
    ).round(5)
)

# ============================================================
# 5. SIMPAN HASIL
# ============================================================

summary_path = (
    "/kaggle/working/"
    "beats_knn_fixed_rejection_30pct_summary.csv"
)

class_path = (
    "/kaggle/working/"
    "beats_knn_fixed_rejection_30pct_per_class.csv"
)

summary_df.to_csv(summary_path, index=False)
class_df.to_csv(class_path, index=False)

print("\nRingkasan disimpan:", summary_path)
print("Rincian disimpan:", class_path)
print(
    "\nCatatan: ini simulasi pseudo-Unknown, "
    "bukan estimasi pasti Macro F1 leaderboard."
)

RANKING BOBOT BERDASARKAN MACRO F1 PSEUDO-UNKNOWN @ 30.1%


,k,weight_confidence,weight_knn,macro_precision_at_30pct,macro_recall_at_30pct,macro_f1_at_30pct
0,3,0.4,0.6,0.49515,0.59021,0.42352
1,3,0.5,0.5,0.49818,0.57681,0.42289
2,5,0.4,0.6,0.49273,0.59525,0.42238
3,5,0.5,0.5,0.49758,0.57579,0.42198
4,7,0.4,0.6,0.49152,0.59045,0.42058
5,7,0.5,0.5,0.49515,0.57735,0.42035
6,5,0.6,0.4,0.50000,0.54843,0.41919
7,3,0.6,0.4,0.49879,0.54735,0.41805
8,7,0.6,0.4,0.49636,0.54916,0.41660
9,3,0.7,0.3,0.50242,0.52131,0.41658



METRIK PER KELAS HELDOUT


,k,weight_confidence,weight_knn,heldout_class,n_samples,n_selected,true_unknown_rate,precision_at_30pct,recall_at_30pct,f1_at_30pct
0,3,0.3,0.7,Bondol Jawa,1826,550,0.61555,0.79455,0.38879,0.52210
5,3,0.3,0.7,Cendrawasih,1826,550,0.04545,0.14000,0.92771,0.24329
10,3,0.3,0.7,Kutilang,1826,550,0.33899,0.50545,0.44911,0.47562
1,3,0.4,0.6,Bondol Jawa,1826,550,0.61555,0.83091,0.40658,0.54600
6,3,0.4,0.6,Cendrawasih,1826,550,0.04545,0.13636,0.90361,0.23697
11,3,0.4,0.6,Kutilang,1826,550,0.33899,0.51818,0.46042,0.48760
2,3,0.5,0.5,Bondol Jawa,1826,550,0.61555,0.84727,0.41459,0.55675
7,3,0.5,0.5,Cendrawasih,1826,550,0.04545,0.12909,0.85542,0.22433
12,3,0.5,0.5,Kutilang,1826,550,0.33899,0.51818,0.46042,0.48760
3,3,0.6,0.4,Bondol Jawa,1826,550,0.61555,0.86364,0.42260,0.56750



Ringkasan disimpan: /kaggle/working/beats_knn_fixed_rejection_30pct_summary.csv
Rincian disimpan: /kaggle/working/beats_knn_fixed_rejection_30pct_per_class.csv

Catatan: ini simulasi pseudo-Unknown, bukan estimasi pasti Macro F1 leaderboard.


In [22]:
import os
import pandas as pd

path = (
    "/kaggle/working/"
    "triple_v4_implicit_unknown_count_by_knn_and_confidence.csv"
)

assert os.path.isfile(path), (
    f"File tidak ditemukan: {path}"
)

df_2f = pd.read_csv(path)

print("Shape:", df_2f.shape)
print("Kolom:", df_2f.columns.tolist())

# Cari baris yang secara eksplisit menyebut confidence
# jika tabel menyimpan metode dalam bentuk baris.
text_mask = df_2f.astype(str).apply(
    lambda col: col.str.contains(
        "confidence|conf-only|confidence-only",
        case=False,
        regex=True,
        na=False
    )
).any(axis=1)

if text_mask.any():
    print("\nBaris terkait confidence:")
    display(df_2f.loc[text_mask])
else:
    # Jika confidence disimpan dalam kolom tersendiri,
    # tampilkan tabel lengkap agar struktur aslinya terlihat.
    print(
        "\nTidak ditemukan nama metode confidence pada nilai baris. "
        "Tampilkan tabel lengkap untuk memeriksa formatnya:"
    )
    display(df_2f)

Shape: (8, 11)
Kolom: ['signal', 'p', 'N_imp_raw_pi_positive_parts', 'N_imp_bootstrap_ci95_low', 'N_imp_bootstrap_ci95_high', 'bootstrap_reps_valid', 'all_class_pi_available', 'ignored_due_to_confidence_floor_rule', 'N_imp_for_decision', 'test_predicted_class_counts', 'interpretation_note']

Baris terkait confidence:


,signal,p,N_imp_raw_pi_positive_parts,N_imp_bootstrap_ci95_low,N_imp_bootstrap_ci95_high,bootstrap_reps_valid,all_class_pi_available,ignored_due_to_confidence_floor_rule,N_imp_for_decision,test_predicted_class_counts,interpretation_note
0,log10_inverse_confidence_ood,0.05,703.080119,502.578277,732.969506,430,True,False,703.080119,"{""Bondol Jawa"": 321, ""Cendrawasih"": 182, ""Kuti...",Diagnostic only; mixture assumptions and OOF/t...
1,log10_inverse_confidence_ood,0.10,646.139756,473.253056,680.482529,430,True,False,646.139756,"{""Bondol Jawa"": 321, ""Cendrawasih"": 182, ""Kuti...",Diagnostic only; mixture assumptions and OOF/t...
2,log10_inverse_confidence_ood,0.25,527.327771,383.742069,560.654782,481,True,False,527.327771,"{""Bondol Jawa"": 321, ""Cendrawasih"": 182, ""Kuti...",Diagnostic only; mixture assumptions and OOF/t...
3,log10_inverse_confidence_ood,0.50,327.984971,209.073929,371.083162,499,True,False,327.984971,"{""Bondol Jawa"": 321, ""Cendrawasih"": 182, ""Kuti...",Diagnostic only; mixture assumptions and OOF/t...


In [24]:
# ============================================================
# PREVALENCE-MATCHED R-PRECISION + WEIGHT GATE
#
# Tidak melatih ulang model audio.
# Tidak membuat atau mengubah submission.
#
# Syarat gerbang:
# 1. Peningkatan R-precision > 0.03 pada minimal 2 kelas.
# 2. Peningkatan tersebut konsisten di k=3 dan k=7.
# 3. Tidak ada kelas yang turun lebih dari 0.03 pada k mana pun.
# ============================================================

import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# 1. KONFIGURASI DAN VALIDASI
# ------------------------------------------------------------

if "detail" not in globals():
    raise RuntimeError(
        "Variabel detail tidak ditemukan. "
        "Jalankan eksperimen LOCO k=3, 5, 7 terlebih dahulu."
    )

required_cols = {
    "k",
    "heldout_class",
    "sample_idx",
    "is_pseudo_unknown",
    "confidence_ood",
    "knn_distance",
}

missing = required_cols - set(detail.columns)
if missing:
    raise ValueError(f"Kolom detail tidak lengkap: {missing}")

CLASS_NAMES = [
    "Bondol Jawa",
    "Cendrawasih",
    "Kutilang",
]

K_VALUES = [3, 7]
CONF_WEIGHTS = [0.3, 0.4, 0.5, 0.6, 0.7]

N_DRAWS = 200
RANDOM_SEED = 2026
DELTA_THRESHOLD = 0.03

# Pool buatan untuk meniru proporsi 245 Unknown : 569 known.
TARGET_POSITIVES = 245
TARGET_NEGATIVES = 569

detail = detail.copy()
detail["k"] = detail["k"].astype(int)
detail["sample_idx"] = detail["sample_idx"].astype(int)
detail["is_pseudo_unknown"] = (
    detail["is_pseudo_unknown"].astype(int)
)

available_k = set(detail["k"].unique())
if not set(K_VALUES).issubset(available_k):
    raise ValueError(f"Nilai k yang tersedia tidak lengkap: {available_k}")

rng = np.random.default_rng(RANDOM_SEED)
draw_records = []

# ------------------------------------------------------------
# 2. BUAT POOL PREVALENCE-MATCHED
# ------------------------------------------------------------

for heldout in CLASS_NAMES:

    base = detail[
        (detail["k"] == 3)
        & (detail["heldout_class"] == heldout)
    ].copy()

    if base.empty:
        raise ValueError(f"Tidak ada data LOCO untuk {heldout}")

    if base["sample_idx"].duplicated().any():
        raise ValueError(
            f"Ada sample_idx duplikat untuk kelas {heldout}"
        )

    positive_ids = base.loc[
        base["is_pseudo_unknown"] == 1,
        "sample_idx",
    ].to_numpy()

    negative_ids = base.loc[
        base["is_pseudo_unknown"] == 0,
        "sample_idx",
    ].to_numpy()

    # Cendrawasih memiliki 83 positif, sehingga digunakan semua
    # positif dan jumlah negatif dikurangi secara proporsional.
    n_pos = min(TARGET_POSITIVES, len(positive_ids))

    if n_pos == TARGET_POSITIVES:
        n_neg = TARGET_NEGATIVES
    else:
        n_neg = int(round(
            n_pos * TARGET_NEGATIVES / TARGET_POSITIVES
        ))

    if len(negative_ids) < n_neg:
        raise ValueError(
            f"Negatif tidak cukup untuk {heldout}: "
            f"{len(negative_ids)} tersedia, {n_neg} dibutuhkan"
        )

    print(
        f"{heldout}: "
        f"positif pool={n_pos}, negatif pool={n_neg}, "
        f"total={n_pos+n_neg}, "
        f"prevalensi={n_pos/(n_pos+n_neg):.4f}"
    )

    # Siapkan tabel per k; semua konfigurasi memakai sampel
    # pool yang sama di setiap draw.
    data_by_k = {}

    for k in K_VALUES:
        part = detail[
            (detail["k"] == k)
            & (detail["heldout_class"] == heldout)
        ].copy()

        if part["sample_idx"].duplicated().any():
            raise ValueError(
                f"Sampel duplikat di k={k}, kelas={heldout}"
            )

        data_by_k[k] = part.set_index(
            "sample_idx", drop=False
        )

    # --------------------------------------------------------
    # 3. ULANGI PENGAMBILAN POOL
    # --------------------------------------------------------

    for draw in range(N_DRAWS):

        # Kedua set dipilih tanpa pengembalian dalam setiap draw.
        pos_sample = rng.choice(
            positive_ids,
            size=n_pos,
            replace=False,
        )

        neg_sample = rng.choice(
            negative_ids,
            size=n_neg,
            replace=False,
        )

        pool_ids = np.concatenate([
            pos_sample, neg_sample
        ])

        # Urutan pool diacak agar pemilihan tidak tergantung
        # pada urutan metadata.
        rng.shuffle(pool_ids)

        for k in K_VALUES:

            pool = data_by_k[k].loc[pool_ids].copy()

            target = pool[
                "is_pseudo_unknown"
            ].to_numpy(dtype=int)

            confidence = pool[
                "confidence_ood"
            ].to_numpy(dtype=float)

            knn_distance = pool[
                "knn_distance"
            ].to_numpy(dtype=float)

            if (
                not np.isfinite(confidence).all()
                or not np.isfinite(knn_distance).all()
            ):
                raise ValueError(
                    f"Nilai tidak valid pada k={k}, {heldout}"
                )

            # Ranking dihitung ulang di dalam pool, karena
            # pool inilah yang digunakan untuk seleksi.
            conf_rank = pd.Series(confidence).rank(
                pct=True, method="average"
            ).to_numpy()

            knn_rank = pd.Series(knn_distance).rank(
                pct=True, method="average"
            ).to_numpy()

            # R-precision: jumlah prediksi teratas sama dengan
            # jumlah pseudo-Unknown positif yang sebenarnya.
            R = int(target.sum())

            for w_conf in CONF_WEIGHTS:

                w_knn = 1.0 - w_conf

                score = (
                    w_conf * conf_rank
                    + w_knn * knn_rank
                )

                # Tie-breaking deterministik menggunakan sample_idx.
                order = np.lexsort((
                    pool["sample_idx"].to_numpy(),
                    -score,
                ))

                top = order[:R]
                hits = int(target[top].sum())

                draw_records.append({
                    "draw": draw,
                    "k": k,
                    "heldout": heldout,
                    "w_conf": w_conf,
                    "w_knn": w_knn,
                    "R": R,
                    "r_precision": hits / R,
                })

draw_df = pd.DataFrame(draw_records)

# ------------------------------------------------------------
# 4. AGREGASIKAN HASIL POOL
# ------------------------------------------------------------

res = (
    draw_df.groupby(
        ["k", "heldout", "w_conf", "w_knn"],
        as_index=False,
    )
    .agg(
        r_precision=("r_precision", "mean"),
        sd_between_draws=("r_precision", "std"),
    )
)

# Selisih terhadap baseline 50:50 untuk kelas dan k yang sama.
base_rp = (
    res[res.w_conf == 0.5]
    .set_index(["k", "heldout"])["r_precision"]
)

res["delta"] = [
    row.r_precision - base_rp[(row.k, row.heldout)]
    for row in res.itertuples()
]

print("\n" + "=" * 70)
print("R-PRECISION PREVALENCE-MATCHED")
print("=" * 70)

display(
    res.sort_values(
        ["w_conf", "heldout", "k"]
    ).round(5)
)

# ------------------------------------------------------------
# 5. GERBANG PERUBAHAN BOBOT
# ------------------------------------------------------------

gate_rows = []

for w in CONF_WEIGHTS:
    if w == 0.5:
        continue

    sub = res[
        (res.w_conf == w)
        & res.k.isin([3, 7])
    ]

    p = sub.pivot(
        index="heldout",
        columns="k",
        values="delta",
    )

    # Kelas harus membaik lebih dari 0.03 pada k=3 DAN k=7.
    up = [
        cls for cls in p.index
        if (
            p.loc[cls, [3, 7]].notna().all()
            and (p.loc[cls, [3, 7]] > DELTA_THRESHOLD).all()
        )
    ]

    worst = float(p.min().min())

    gate_rows.append({
        "w_conf": w,
        "w_knn": 1.0 - w,
        "classes_up_gt_003_both_k": len(up),
        "up": ", ".join(up),
        "worst_class_delta": worst,
        "passes": len(up) >= 2 and worst > -DELTA_THRESHOLD,
    })

gate_df = pd.DataFrame(gate_rows)

print("\n" + "=" * 70)
print("GERBANG BOBOT — PEMBANDING 50:50")
print("=" * 70)

display(gate_df.round(5))

# ------------------------------------------------------------
# 6. SIMPAN HASIL
# ------------------------------------------------------------

out_dir = "/kaggle/working"

res_path = os.path.join(
    out_dir,
    "beats_knn_r_precision_prevalence_matched.csv",
)

draw_path = os.path.join(
    out_dir,
    "beats_knn_r_precision_prevalence_matched_draws.csv",
)

gate_path = os.path.join(
    out_dir,
    "beats_knn_r_precision_weight_gate.csv",
)

res.to_csv(res_path, index=False)
draw_df.to_csv(draw_path, index=False)
gate_df.to_csv(gate_path, index=False)

print("\nFile hasil:")
print(res_path)
print(draw_path)
print(gate_path)

if gate_df["passes"].any():
    print(
        "\nAda kandidat yang lolos gerbang awal. "
        "Ini alasan untuk analisis lanjutan, bukan jaminan LB naik."
    )
else:
    print(
        "\nTidak ada bobot yang lolos gerbang. "
        "Pertahankan champion 50:50 dan jangan submit kandidat bobot."
    )

print(
    "\nCatatan: variasi antar-draw mengukur sensitivitas terhadap "
    "pemilihan pool sampel, bukan confidence interval berdasarkan "
    "source_group."
)

Bondol Jawa: positif pool=245, negatif pool=569, total=814, prevalensi=0.3010
Cendrawasih: positif pool=83, negatif pool=193, total=276, prevalensi=0.3007
Kutilang: positif pool=245, negatif pool=569, total=814, prevalensi=0.3010

R-PRECISION PREVALENCE-MATCHED


,k,heldout,w_conf,w_knn,r_precision,sd_between_draws,delta
0,3,Bondol Jawa,0.3,0.7,0.51171,0.01880,-0.05704
15,7,Bondol Jawa,0.3,0.7,0.49618,0.01817,-0.05484
5,3,Cendrawasih,0.3,0.7,0.77319,0.02426,0.09211
20,7,Cendrawasih,0.3,0.7,0.77620,0.02222,0.08886
10,3,Kutilang,0.3,0.7,0.45788,0.01833,-0.01543
25,7,Kutilang,0.3,0.7,0.44553,0.01921,-0.01641
1,3,Bondol Jawa,0.4,0.6,0.53547,0.01530,-0.03329
16,7,Bondol Jawa,0.4,0.6,0.52641,0.01780,-0.02461
6,3,Cendrawasih,0.4,0.6,0.75398,0.02962,0.07289
21,7,Cendrawasih,0.4,0.6,0.75169,0.02858,0.06434



GERBANG BOBOT — PEMBANDING 50:50


,w_conf,w_knn,classes_up_gt_003_both_k,up,worst_class_delta,passes
0,0.3,0.7,1,Cendrawasih,-0.05704,False
1,0.4,0.6,1,Cendrawasih,-0.03329,False
2,0.6,0.4,0,,-0.05259,False
3,0.7,0.3,0,,-0.13518,False



File hasil:
/kaggle/working/beats_knn_r_precision_prevalence_matched.csv
/kaggle/working/beats_knn_r_precision_prevalence_matched_draws.csv
/kaggle/working/beats_knn_r_precision_weight_gate.csv

Tidak ada bobot yang lolos gerbang. Pertahankan champion 50:50 dan jangan submit kandidat bobot.

Catatan: variasi antar-draw mengukur sensitivitas terhadap pemilihan pool sampel, bukan confidence interval berdasarkan source_group.


In [25]:
# ============================================================
# LOF RELATIVE-DENSITY — REDUNDANCY PREFLIGHT
#
# Champion kerja:
#   k=3, confidence 50%, kNN 50%, Unknown=245
#
# Tujuan:
#   1. Membuat fitur OOF LOF dengan validasi group-fold.
#   2. Mengukur Spearman antara LOF dan fusion 50:50 pada LOCO.
#   3. Mengukur Spearman antara LOF dan fusion 50:50 pada test.
#
# Belum membuat submission.
# Tidak mengubah file champion.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.preprocessing import normalize

# ------------------------------------------------------------
# 1. KONFIGURASI
# ------------------------------------------------------------

DATA_ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

CLASS_NAMES = ["Bondol Jawa", "Cendrawasih", "Kutilang"]

LOF_K = 10
BASE_K_VALUES = [3, 7]

# Ambang pemeriksaan redundansi.
CORRELATION_THRESHOLD = 0.90

# ------------------------------------------------------------
# 2. PENCARIAN FILE
# ------------------------------------------------------------

def locate_file(filename, preferred_paths=None):
    preferred_paths = preferred_paths or []

    for path in preferred_paths:
        if path and os.path.isfile(path):
            return path

    for root in [OUT, "/kaggle/input"]:
        if not os.path.isdir(root):
            continue

        matches = glob.glob(
            os.path.join(root, "**", filename),
            recursive=True
        )

        if matches:
            matches.sort(key=lambda p: (len(p), p))
            return matches[0]

    raise FileNotFoundError(
        f"File tidak ditemukan: {filename}"
    )


metadata_path = os.path.join(
    DATA_ROOT, "train", "metadata.csv"
)

if not os.path.isfile(metadata_path):
    metadata_path = locate_file("metadata.csv")

fold_path = locate_file(
    "bird_group_folds_seed16.csv",
    [
        "/kaggle/input/notebooks/radithiafatir/"
        "testttrisan/bird_group_folds_seed16.csv",
        os.path.join(OUT, "bird_group_folds_seed16.csv"),
    ],
)

train_emb_path = locate_file(
    "beats_iter3plus_as2m_8s_train_embeddings.npy"
)

test_emb_path = locate_file(
    "beats_iter3plus_as2m_8s_test_embeddings.npy"
)

rank_path = locate_file(
    "triple_v4_unknown_count_rank_scores.csv"
)

sample_path = os.path.join(
    DATA_ROOT, "sample_submission.csv"
)

print("=" * 70)
print("INPUT")
print("=" * 70)
print("Metadata :", metadata_path)
print("Fold CSV :", fold_path)
print("Train emb:", train_emb_path)
print("Test emb :", test_emb_path)
print("Rank CSV :", rank_path)
print("Sample   :", sample_path)

# ------------------------------------------------------------
# 3. LOAD DAN VALIDASI DATA
# ------------------------------------------------------------

meta = pd.read_csv(metadata_path)
fold_df = pd.read_csv(fold_path)
sample = pd.read_csv(sample_path)
rank_df = pd.read_csv(rank_path)

meta["file_id"] = meta["file_id"].astype(str)
fold_df["file_id"] = fold_df["file_id"].astype(str)
sample["file_id"] = sample["file_id"].astype(str)
rank_df["file_id"] = rank_df["file_id"].astype(str)

assert meta["file_id"].is_unique
assert fold_df["file_id"].is_unique
assert sample["file_id"].is_unique
assert rank_df["file_id"].is_unique

meta = meta.merge(
    fold_df[["file_id", "fold"]],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert meta["fold"].notna().all()
meta["fold"] = meta["fold"].astype(int)

label_to_idx = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

y = meta["label"].map(label_to_idx).to_numpy(dtype=int)

assert not np.isnan(y).any()

folds = meta["fold"].to_numpy()
fold_values = sorted(np.unique(folds))

train_emb = np.load(train_emb_path).astype(np.float32)
test_emb = np.load(test_emb_path).astype(np.float32)

assert train_emb.shape == (len(meta), 768)
assert test_emb.shape == (len(sample), 768)

assert len(sample) == 814
assert np.isfinite(train_emb).all()
assert np.isfinite(test_emb).all()

group_leakage = (
    meta.groupby("source_group")["fold"].nunique() > 1
).sum()

assert group_leakage == 0, (
    "Ada source_group yang bocor antar-fold."
)

print("\nJumlah train:", len(meta))
print("Jumlah test:", len(sample))
print("Jumlah source_group:", meta["source_group"].nunique())
print("Fold counts:", meta["fold"].value_counts().sort_index().to_dict())
print("Source-group leakage:", int(group_leakage))

# ------------------------------------------------------------
# 4. GENERATE OOF LOF PER HELDOUT CLASS DAN FOLD
# ------------------------------------------------------------
# Higher LOF OOD score = lebih anomalous.
#
# Untuk setiap fold:
#   training = semua fold lain, tanpa heldout class
#   validasi = seluruh sampel pada fold tersebut
#
# Tidak ada sampel validasi yang ikut membangun bank LOF.

print("\n" + "=" * 70)
print("GENERATE OOF LOF")
print("=" * 70)

X_norm = normalize(train_emb, norm="l2")
X_test_norm = normalize(test_emb, norm="l2")

oof_records = []

for heldout_idx, heldout_name in enumerate(CLASS_NAMES):

    print(f"\nHeldout class: {heldout_name}")

    for fold in fold_values:

        tr_idx = np.flatnonzero(
            (folds != fold) & (y != heldout_idx)
        )

        va_idx = np.flatnonzero(
            folds == fold
        )

        if len(tr_idx) <= LOF_K:
            raise ValueError(
                f"Training terlalu sedikit untuk LOF pada fold {fold}"
            )

        lof = LocalOutlierFactor(
            n_neighbors=min(LOF_K, len(tr_idx) - 1),
            metric="cosine",
            novelty=True,
            contamination="auto",
            n_jobs=-1,
        )

        lof.fit(X_norm[tr_idx])

        # score_samples makin rendah = makin anomali.
        # Kita balik tandanya agar skor tinggi berarti OOD.
        ood_score = -lof.score_samples(X_norm[va_idx])

        for j, sample_idx in enumerate(va_idx):
            oof_records.append({
                "sample_idx": int(sample_idx),
                "fold": int(fold),
                "heldout_class": heldout_name,
                "source_group": str(
                    meta.iloc[sample_idx]["source_group"]
                ),
                "is_pseudo_unknown": int(
                    y[sample_idx] == heldout_idx
                ),
                "lof_ood_score": float(ood_score[j]),
            })

        print(
            f"  Fold {fold}: train={len(tr_idx)}, "
            f"validasi={len(va_idx)}"
        )

oof_lof = pd.DataFrame(oof_records)

assert len(oof_lof) == len(CLASS_NAMES) * len(meta)

print("\nOOF LOF selesai:", oof_lof.shape)

# Simpan fitur OOF, bukan submission.
oof_lof_path = os.path.join(
    OUT, "beats_lof_relative_density_oof.csv"
)

oof_lof.to_csv(oof_lof_path, index=False)

# ------------------------------------------------------------
# 5. SPEARMAN LOF VS FUSION 50:50 PADA LOCO
# ------------------------------------------------------------
# Gunakan skor confidence dan kNN yang sama dari eksperimen
# LOCO terdahulu agar perbandingan tidak mencampur pipeline.

if "detail" not in globals():
    raise RuntimeError(
        "Variabel detail dari LOCO sebelumnya tidak ada. "
        "Jalankan kembali cell eksperimen LOCO k=3,5,7."
    )

required_detail = {
    "k", "sample_idx", "fold", "heldout_class",
    "confidence_ood", "knn_distance",
}

missing = required_detail - set(detail.columns)

if missing:
    raise ValueError(
        f"Kolom LOCO detail tidak lengkap: {missing}"
    )

detail_compare = detail[
    detail["k"].isin(BASE_K_VALUES)
].copy()

detail_compare["sample_idx"] = (
    detail_compare["sample_idx"].astype(int)
)

detail_compare["fold"] = detail_compare["fold"].astype(int)

oof_compare = detail_compare.merge(
    oof_lof[
        [
            "sample_idx",
            "fold",
            "heldout_class",
            "lof_ood_score",
            "source_group",
        ]
    ],
    on=["sample_idx", "fold", "heldout_class"],
    how="left",
    validate="many_to_one",
)

assert oof_compare["lof_ood_score"].notna().all()

loco_corr_rows = []

for k in BASE_K_VALUES:

    dk = oof_compare[
        oof_compare["k"] == k
    ].copy()

    for cls in CLASS_NAMES:

        part = dk[
            dk["heldout_class"] == cls
        ].copy()

        # Baseline rank dibentuk dalam skenario heldout class
        # yang sama, bukan dicampur antar-kelas heldout.
        conf_rank = part["confidence_ood"].rank(
            pct=True, method="average"
        )

        knn_rank = part["knn_distance"].rank(
            pct=True, method="average"
        )

        baseline_50_50 = 0.5 * conf_rank + 0.5 * knn_rank

        spearman = pd.Series(
            baseline_50_50.to_numpy()
        ).corr(
            pd.Series(part["lof_ood_score"].to_numpy()),
            method="spearman",
        )

        loco_corr_rows.append({
            "k": k,
            "heldout_class": cls,
            "n": len(part),
            "spearman_lof_vs_fusion50_50": spearman,
        })

loco_corr_df = pd.DataFrame(loco_corr_rows)

print("\n" + "=" * 70)
print("SPEARMAN LOCO: LOF VS FUSION 50:50")
print("=" * 70)

display(loco_corr_df.round(5))

# ------------------------------------------------------------
# 6. FIT LOF PADA SELURUH TRAIN, SKOR TEST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATE TEST LOF")
print("=" * 70)

lof_test_model = LocalOutlierFactor(
    n_neighbors=min(LOF_K, len(train_emb) - 1),
    metric="cosine",
    novelty=True,
    contamination="auto",
    n_jobs=-1,
)

lof_test_model.fit(X_norm)

test_lof_score = -lof_test_model.score_samples(
    X_test_norm
)

assert len(test_lof_score) == len(sample)
assert np.isfinite(test_lof_score).all()

test_lof_df = pd.DataFrame({
    "file_id": sample["file_id"].to_numpy(),
    "lof_ood_score": test_lof_score,
})

# ------------------------------------------------------------
# 7. SKOR TEST BASELINE K=3 DAN K=7
# ------------------------------------------------------------
# Ranking confidence memakai artefak lama yang telah digunakan
# untuk merekonstruksi kandidat k=3.
#
# Jarak kNN test dihitung kembali dari embedding train-test.

required_rank_cols = {
    "file_id",
    "confidence_uncertainty_rank",
}

missing_rank_cols = required_rank_cols - set(rank_df.columns)

if missing_rank_cols:
    raise ValueError(
        f"Kolom ranking confidence tidak ada: {missing_rank_cols}"
    )

rank_test = sample[["file_id"]].merge(
    rank_df[
        ["file_id", "confidence_uncertainty_rank"]
    ],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert rank_test[
    "confidence_uncertainty_rank"
].notna().all()

confidence_rank_test = rank_test[
    "confidence_uncertainty_rank"
].to_numpy(dtype=float)

nn_test = NearestNeighbors(
    n_neighbors=max(BASE_K_VALUES),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

nn_test.fit(X_norm)

distances_test, _ = nn_test.kneighbors(X_test_norm)

test_corr_rows = []

test_lof_rank = pd.Series(test_lof_score).rank(
    pct=True,
    method="average",
).to_numpy()

for k in BASE_K_VALUES:

    knn_dist = distances_test[:, :k].mean(axis=1)

    knn_rank = pd.Series(knn_dist).rank(
        pct=True,
        method="average",
    ).to_numpy()

    baseline = (
        0.5 * confidence_rank_test
        + 0.5 * knn_rank
    )

    corr = pd.Series(baseline).corr(
        pd.Series(test_lof_rank),
        method="spearman",
    )

    test_corr_rows.append({
        "k": k,
        "n_test": len(sample),
        "spearman_lof_vs_fusion50_50": corr,
    })

test_corr_df = pd.DataFrame(test_corr_rows)

print("\n" + "=" * 70)
print("SPEARMAN TEST: LOF VS FUSION 50:50")
print("=" * 70)

display(test_corr_df.round(5))

# ------------------------------------------------------------
# 8. GERBANG REDUNDANSI
# ------------------------------------------------------------

loco_mean_abs = (
    loco_corr_df
    .groupby("k")["spearman_lof_vs_fusion50_50"]
    .apply(lambda x: float(np.mean(np.abs(x))))
)

test_corr_values = test_corr_df[
    "spearman_lof_vs_fusion50_50"
].to_numpy(dtype=float)

print("\n" + "=" * 70)
print("GERBANG REDUNDANSI LOF")
print("=" * 70)

for k in BASE_K_VALUES:
    corr_test = float(
        test_corr_df.loc[
            test_corr_df["k"] == k,
            "spearman_lof_vs_fusion50_50",
        ].iloc[0]
    )

    corr_loco = float(loco_mean_abs.loc[k])

    print(
        f"k={k}: mean |Spearman| LOCO={corr_loco:.5f}; "
        f"Spearman test={corr_test:.5f}"
    )

    if (
        corr_loco >= CORRELATION_THRESHOLD
        or abs(corr_test) >= CORRELATION_THRESHOLD
    ):
        print(
            "  -> LOF terlalu redundan menurut gerbang awal."
        )
    else:
        print(
            "  -> LOF cukup berbeda untuk dipertimbangkan."
        )

test_redundant = bool(
    (np.abs(test_corr_values) >= CORRELATION_THRESHOLD).any()
)

loco_redundant = bool(
    (loco_mean_abs >= CORRELATION_THRESHOLD).any()
)

print("\nKeputusan preflight:")

if test_redundant or loco_redundant:
    print(
        "STOP: setidaknya satu pemeriksaan melewati ambang "
        "redundansi 0.90. Jangan lanjutkan ke fusion ketiga."
    )
else:
    print(
        "LANJUT KE EVALUASI OFFLINE: korelasi di bawah 0.90 "
        "untuk kedua konfigurasi k yang diuji."
    )

# ------------------------------------------------------------
# 9. SIMPAN HASIL DIAGNOSTIK
# ------------------------------------------------------------

loco_corr_path = os.path.join(
    OUT, "beats_lof_redundancy_loco.csv"
)

test_corr_path = os.path.join(
    OUT, "beats_lof_redundancy_test.csv"
)

test_lof_path = os.path.join(
    OUT, "beats_lof_relative_density_test.csv"
)

loco_corr_df.to_csv(loco_corr_path, index=False)
test_corr_df.to_csv(test_corr_path, index=False)
test_lof_df.to_csv(test_lof_path, index=False)

print("\nFile tersimpan:")
print(oof_lof_path)
print(loco_corr_path)
print(test_corr_path)
print(test_lof_path)

print("\nTidak ada submission yang dibuat atau diubah.")

INPUT
Metadata : /kaggle/input/competitions/schematics-2026/train/metadata.csv
Fold CSV : /kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
Train emb: /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy
Test emb : /kaggle/working/beats_iter3plus_as2m_8s_test_embeddings.npy
Rank CSV : /kaggle/working/triple_v4_unknown_count_rank_scores.csv
Sample   : /kaggle/input/competitions/schematics-2026/sample_submission.csv

Jumlah train: 1826
Jumlah test: 814
Jumlah source_group: 15
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}
Source-group leakage: 0

GENERATE OOF LOF

Heldout class: Bondol Jawa
  Fold 0: train=535, validasi=284
  Fold 1: train=521, validasi=495
  Fold 2: train=543, validasi=553
  Fold 3: train=507, validasi=494

Heldout class: Cendrawasih
  Fold 0: train=1494, validasi=284
  Fold 1: train=1251, validasi=495
  Fold 2: train=1206, validasi=553
  Fold 3: train=1278, validasi=494

Heldout class: Kutilang
  Fold 0: train=1055, validasi=284
  Fol

,k,heldout_class,n,spearman_lof_vs_fusion50_50
0,3,Bondol Jawa,1826,0.65320
1,3,Cendrawasih,1826,0.51516
2,3,Kutilang,1826,0.70933
3,7,Bondol Jawa,1826,0.65124
4,7,Cendrawasih,1826,0.53163
5,7,Kutilang,1826,0.71465



GENERATE TEST LOF

SPEARMAN TEST: LOF VS FUSION 50:50


,k,n_test,spearman_lof_vs_fusion50_50
0,3,814,0.75715
1,7,814,0.75500



GERBANG REDUNDANSI LOF
k=3: mean |Spearman| LOCO=0.62590; Spearman test=0.75715
  -> LOF cukup berbeda untuk dipertimbangkan.
k=7: mean |Spearman| LOCO=0.63251; Spearman test=0.75500
  -> LOF cukup berbeda untuk dipertimbangkan.

Keputusan preflight:
LANJUT KE EVALUASI OFFLINE: korelasi di bawah 0.90 untuk kedua konfigurasi k yang diuji.

File tersimpan:
/kaggle/working/beats_lof_relative_density_oof.csv
/kaggle/working/beats_lof_redundancy_loco.csv
/kaggle/working/beats_lof_redundancy_test.csv
/kaggle/working/beats_lof_relative_density_test.csv

Tidak ada submission yang dibuat atau diubah.


In [27]:
# ============================================================
# BEATS LOF THIRD-SIGNAL EVALUATION — REVISED
#
# Champion kerja:
#   k=3, confidence 50%, kNN 50%, Unknown=245
#   Public LB 0.68379
#
# Bobot LOF:
#   0.10  -> eksplorasi saja
#   1/3   -> satu-satunya bobot yang boleh dipertimbangkan
#            untuk submission, jika semua gerbang lolos
#
# Evaluasi:
#   - AUC LOF sendiri
#   - Baseline k=3 wajib cocok persis dengan champion
#   - Prevalence-matched R-precision
#   - Paired source-group bootstrap
#   - Dampak per kelas
#   - Pergeseran ID Unknown di test
#
# Tidak membuat atau mengubah submission.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# ============================================================
# 1. KONFIGURASI
# ============================================================

DATA_ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

CLASS_NAMES = [
    "Bondol Jawa",
    "Cendrawasih",
    "Kutilang",
]

K_VALUES = [3, 7]

# 0.10 hanya untuk eksplorasi.
# 1/3 adalah satu-satunya bobot yang boleh dipertimbangkan
# untuk submission setelah melewati seluruh gerbang.
LOF_WEIGHTS = [0.10, 1 / 3]

N_BOOTSTRAP = 500
SEED = 20261020

N_UNKNOWN = 245
N_KNOWN = 569

# Gerbang yang ditetapkan sebelum melihat hasil.
MIN_MACRO_RPRECISION_GAIN = 0.03
MAX_CLASS_DROP = 0.03
MIN_UNKNOWN_ID_CHANGES = 40

# ============================================================
# 2. CARI FILE YANG SUDAH ADA
# ============================================================

def find_file(filename, preferred_paths=None):
    preferred_paths = preferred_paths or []

    for path in preferred_paths:
        if path and os.path.isfile(path):
            return path

    matches = []

    for root in [OUT, "/kaggle/input"]:
        if not os.path.isdir(root):
            continue

        matches.extend(
            glob.glob(
                os.path.join(root, "**", filename),
                recursive=True,
            )
        )

    if matches:
        matches.sort(key=lambda p: (len(p), p))
        return matches[0]

    raise FileNotFoundError(
        f"File tidak ditemukan: {filename}"
    )


SAMPLE_PATH = os.path.join(
    DATA_ROOT, "sample_submission.csv"
)

RANK_PATH = os.path.join(
    OUT, "triple_v4_unknown_count_rank_scores.csv"
)

LOF_OOF_PATH = os.path.join(
    OUT, "beats_lof_relative_density_oof.csv"
)

LOF_TEST_PATH = os.path.join(
    OUT, "beats_lof_relative_density_test.csv"
)

TRAIN_EMB_PATH = os.path.join(
    OUT, "beats_iter3plus_as2m_8s_train_embeddings.npy"
)

TEST_EMB_PATH = os.path.join(
    OUT, "beats_iter3plus_as2m_8s_test_embeddings.npy"
)

K3_CHAMP_PATH = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_k3_u245.csv"
)

K7_CHAMP_PATH = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_u245.csv"
)

if not os.path.isfile(K3_CHAMP_PATH):
    K3_CHAMP_PATH = find_file(
        "submission_triple_v4_conf_knn50_k3_u245.csv"
    )

if not os.path.isfile(K7_CHAMP_PATH):
    K7_CHAMP_PATH = find_file(
        "submission_triple_v4_conf_knn50_u245.csv"
    )

for path in [
    SAMPLE_PATH,
    RANK_PATH,
    LOF_OOF_PATH,
    LOF_TEST_PATH,
    TRAIN_EMB_PATH,
    TEST_EMB_PATH,
    K3_CHAMP_PATH,
    K7_CHAMP_PATH,
]:
    assert os.path.isfile(path), f"File tidak ditemukan: {path}"

if "detail" not in globals():
    raise RuntimeError(
        "Variabel detail dari eksperimen LOCO tidak ditemukan. "
        "Jalankan cell LOCO k=3,5,7 terlebih dahulu."
    )

print("=" * 70)
print("INPUT")
print("=" * 70)
print("Rank scores :", RANK_PATH)
print("LOF OOF     :", LOF_OOF_PATH)
print("LOF test    :", LOF_TEST_PATH)
print("Champion k=3:", K3_CHAMP_PATH)
print("Champion k=7:", K7_CHAMP_PATH)

# ============================================================
# 3. BACA ARTEFAK DAN VALIDASI BASELINE TERLEBIH DAHULU
# ============================================================
# Bagian ini sengaja ditempatkan sebelum bootstrap.
# Jika baseline tidak cocok, evaluasi langsung dihentikan.

sample = pd.read_csv(SAMPLE_PATH)
rank_df = pd.read_csv(RANK_PATH)
selected_k7 = pd.read_csv(LOF_OOF_PATH)  # hanya cek file dapat dibaca
lof_test = pd.read_csv(LOF_TEST_PATH)

champion_k3 = pd.read_csv(K3_CHAMP_PATH)
champion_k7 = pd.read_csv(K7_CHAMP_PATH)

for df in [
    sample,
    rank_df,
    lof_test,
    champion_k3,
    champion_k7,
]:
    if "file_id" not in df.columns:
        raise ValueError(
            f"Kolom file_id tidak ditemukan: {df.columns.tolist()}"
        )

    df["file_id"] = df["file_id"].astype(str)

assert len(sample) == 814
assert sample["file_id"].is_unique
assert len(rank_df) == 814
assert rank_df["file_id"].is_unique
assert len(lof_test) == 814
assert lof_test["file_id"].is_unique
assert len(champion_k3) == 814
assert len(champion_k7) == 814

assert set(sample["file_id"]) == set(
    rank_df["file_id"]
)
assert set(sample["file_id"]) == set(
    lof_test["file_id"]
)
assert set(sample["file_id"]) == set(
    champion_k3["file_id"]
)
assert set(sample["file_id"]) == set(
    champion_k7["file_id"]
)

required_rank_cols = {
    "file_id",
    "confidence_uncertainty_rank",
}

missing_rank_cols = required_rank_cols - set(rank_df.columns)

if missing_rank_cols:
    raise ValueError(
        f"Kolom confidence rank hilang: {missing_rank_cols}"
    )

# Confidence rank asli harus berasal dari artefak champion.
rank_test = sample[["file_id"]].merge(
    rank_df[
        ["file_id", "confidence_uncertainty_rank"]
    ],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert rank_test[
    "confidence_uncertainty_rank"
].notna().all()

confidence_rank_test = rank_test[
    "confidence_uncertainty_rank"
].to_numpy(dtype=float)

# Cek daftar Unknown asli dari kedua champion.
k3_unknown = set(
    champion_k3.loc[
        champion_k3["label"] == "Unknown",
        "file_id",
    ]
)

k7_unknown = set(
    champion_k7.loc[
        champion_k7["label"] == "Unknown",
        "file_id",
    ]
)

assert len(k3_unknown) == N_UNKNOWN
assert len(k7_unknown) == N_UNKNOWN

# selected_245 adalah artefak milik pipeline k=7.
selected_path = find_file(
    "triple_v4_conf_knn50_selected_245.csv"
)

selected_df = pd.read_csv(selected_path)
selected_df["file_id"] = selected_df["file_id"].astype(str)

assert len(selected_df) == N_UNKNOWN
assert set(selected_df["file_id"]) == k7_unknown, (
    "Artefak selected_245 tidak cocok dengan champion k=7."
)

print("\nID Unknown berbeda k=3 vs k=7:",
      len(k3_unknown ^ k7_unknown))

# ------------------------------------------------------------
# 3A. REKONSTRUKSI BASELINE TEST K=3 DAN K=7
# ------------------------------------------------------------

train_emb = np.load(TRAIN_EMB_PATH).astype(np.float32)
test_emb = np.load(TEST_EMB_PATH).astype(np.float32)

assert train_emb.shape == (1826, 768)
assert test_emb.shape == (814, 768)

assert np.isfinite(train_emb).all()
assert np.isfinite(test_emb).all()

train_norm = normalize(train_emb, norm="l2")
test_norm = normalize(test_emb, norm="l2")

nn_test = NearestNeighbors(
    n_neighbors=max(K_VALUES),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

nn_test.fit(train_norm)
distances_test, _ = nn_test.kneighbors(test_norm)

baseline_test_scores = {}
baseline_unknown_ids = {}

for k in K_VALUES:
    distance_k = distances_test[:, :k].mean(axis=1)

    knn_rank_test = pd.Series(
        distance_k
    ).rank(
        pct=True,
        method="average",
    ).to_numpy()

    # Baseline champion: fusion 50:50.
    score = (
        0.5 * confidence_rank_test
        + 0.5 * knn_rank_test
    )

    idx = np.argsort(-score, kind="stable")[:N_UNKNOWN]

    ids = sample["file_id"].astype(str).to_numpy()

    baseline_test_scores[k] = score
    baseline_unknown_ids[k] = set(ids[idx])

# ============================================================
# VALIDASI BASELINE TEST — K3 WAJIB COCOK, K7 DIAGNOSTIK
# ============================================================

print("=" * 70)
print("VALIDASI BASELINE TEST")
print("=" * 70)

for k in K_VALUES:
    target_ids = k3_unknown if k == 3 else k7_unknown
    reproduced_ids = baseline_unknown_ids[k]

    overlap = len(target_ids & reproduced_ids)

    print(
        f"k={k}: ID Unknown cocok "
        f"{overlap}/{N_UNKNOWN}"
    )

# Champion aktif adalah k=3.
# Ini harus cocok persis sebelum evaluasi LOF dilanjutkan.
if baseline_unknown_ids[3] != k3_unknown:
    raise RuntimeError(
        "Baseline k=3 tidak cocok persis dengan champion. "
        "Hentikan evaluasi LOF dan periksa pipeline."
    )

print("LOLOS: baseline k=3 cocok persis.")

# Untuk k=7, rekonstruksi jarak tidak identik dengan
# pipeline lama. Catat ketidakcocokan dan gunakan daftar
# Unknown k=7 asli saat menghitung perubahan ID relatif
# terhadap champion k=7.
if baseline_unknown_ids[7] != k7_unknown:
    print(
        "PERINGATAN: baseline k=7 berbeda dari champion "
        "pada satu atau beberapa ID."
    )
    print(
        "Untuk perbandingan pergeseran terhadap champion, "
        "gunakan daftar Unknown k=7 asli."
    )

    baseline_unknown_ids[7] = k7_unknown
else:
    print("Baseline k=7 juga cocok persis.")

# ------------------------------------------------------------
# 3B. HITUNG PERUBAHAN ID TEST UNTUK SETIAP BOBOT LOF
# ------------------------------------------------------------

lof_test_aligned = sample[["file_id"]].merge(
    lof_test[["file_id", "lof_ood_score"]],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert lof_test_aligned["lof_ood_score"].notna().all()

lof_rank_test = pd.Series(
    lof_test_aligned["lof_ood_score"].to_numpy(dtype=float)
).rank(
    pct=True,
    method="average",
).to_numpy()

test_shift_rows = []

for alpha in LOF_WEIGHTS:
    for k in K_VALUES:

        baseline_score = baseline_test_scores[k]

        candidate_score = (
            (1.0 - alpha) * baseline_score
            + alpha * lof_rank_test
        )

        idx = np.argsort(
            -candidate_score,
            kind="stable",
        )[:N_UNKNOWN]

        ids = sample["file_id"].astype(str).to_numpy()
        candidate_ids = set(ids[idx])

        baseline_ids = baseline_unknown_ids[k]

        out_ids = baseline_ids - candidate_ids
        in_ids = candidate_ids - baseline_ids

        symmetric_difference = len(
            baseline_ids ^ candidate_ids
        )

        test_shift_rows.append({
            "k": k,
            "lof_weight": alpha,
            "unknown_retained": len(
                baseline_ids & candidate_ids
            ),
            "unknown_out": len(out_ids),
            "unknown_in": len(in_ids),
            "symmetric_difference_ids": symmetric_difference,
            "meets_40_id_shift": (
                symmetric_difference >= MIN_UNKNOWN_ID_CHANGES
            ),
        })

test_shift_df = pd.DataFrame(test_shift_rows)

print("\n" + "=" * 70)
print("PERGESERAN ID UNKNOWN TEST")
print("=" * 70)

display(test_shift_df)

# ============================================================
# 4. LOAD DAN VALIDASI LOCO DETAIL
# ============================================================

required_detail = {
    "k",
    "sample_idx",
    "fold",
    "heldout_class",
    "is_pseudo_unknown",
    "confidence_ood",
    "knn_distance",
}

missing_detail = required_detail - set(detail.columns)

if missing_detail:
    raise ValueError(
        f"Kolom detail tidak lengkap: {missing_detail}"
    )

detail_work = detail.copy()

detail_work["k"] = detail_work["k"].astype(int)
detail_work["sample_idx"] = (
    detail_work["sample_idx"].astype(int)
)
detail_work["fold"] = detail_work["fold"].astype(int)
detail_work["heldout_class"] = (
    detail_work["heldout_class"].astype(str)
)

available_k = set(detail_work["k"].unique())

assert set(K_VALUES).issubset(available_k)

oof_lof = pd.read_csv(LOF_OOF_PATH)

required_lof_cols = {
    "sample_idx",
    "fold",
    "heldout_class",
    "source_group",
    "lof_ood_score",
}

missing_lof_cols = required_lof_cols - set(oof_lof.columns)

if missing_lof_cols:
    raise ValueError(
        f"Kolom LOF OOF tidak lengkap: {missing_lof_cols}"
    )

oof_lof["sample_idx"] = oof_lof["sample_idx"].astype(int)
oof_lof["fold"] = oof_lof["fold"].astype(int)
oof_lof["heldout_class"] = (
    oof_lof["heldout_class"].astype(str)
)
oof_lof["source_group"] = (
    oof_lof["source_group"].astype(str)
)

keys = ["sample_idx", "fold", "heldout_class"]

if oof_lof.duplicated(keys).any():
    raise ValueError("Ada kunci LOF OOF duplikat.")

# Gabungkan k=3 dan k=7 dengan satu set LOF OOF.
d3 = detail_work[
    detail_work["k"] == 3
][
    keys + [
        "is_pseudo_unknown",
        "confidence_ood",
        "knn_distance",
    ]
].rename(
    columns={
        "confidence_ood": "confidence_ood_k3",
        "knn_distance": "knn_distance_k3",
    }
)

d7 = detail_work[
    detail_work["k"] == 7
][
    keys + [
        "is_pseudo_unknown",
        "confidence_ood",
        "knn_distance",
    ]
].rename(
    columns={
        "confidence_ood": "confidence_ood_k7",
        "knn_distance": "knn_distance_k7",
        "is_pseudo_unknown": "target_k7",
    }
)

wide = d3.merge(
    d7,
    on=keys,
    how="inner",
    validate="one_to_one",
)

assert (
    wide["is_pseudo_unknown"].to_numpy()
    == wide["target_k7"].to_numpy()
).all()

assert np.allclose(
    wide["confidence_ood_k3"],
    wide["confidence_ood_k7"],
    rtol=1e-6,
    atol=1e-7,
), "Confidence LOCO berbeda antara k=3 dan k=7."

wide = wide.merge(
    oof_lof[
        keys + [
            "source_group",
            "lof_ood_score",
        ]
    ],
    on=keys,
    how="left",
    validate="one_to_one",
)

if wide["lof_ood_score"].isna().any():
    raise ValueError(
        "Ada sampel LOCO yang tidak memiliki skor LOF."
    )

wide["source_group"] = wide["source_group"].astype(str)

assert wide["sample_idx"].nunique() == 1826
assert wide["source_group"].nunique() == 15

print("\nLOCO rows:", len(wide))
print("Source groups:", wide["source_group"].nunique())

# ============================================================
# 5. CEK ARAH DAN AUC LOF SENDIRI
# ============================================================

_auc_rows = []

for cls in CLASS_NAMES:

    part = wide[
        wide["heldout_class"] == cls
    ]

    target = part[
        "is_pseudo_unknown"
    ].to_numpy(dtype=int)

    score = part[
        "lof_ood_score"
    ].to_numpy(dtype=float)

    auc = roc_auc_score(target, score)

    _auc_rows.append({
        "heldout_class": cls,
        "auc_lof_only": auc,
    })

auc_lof_df = pd.DataFrame(_auc_rows)

print("\n" + "=" * 70)
print("AUC LOF SENDIRI")
print("=" * 70)

display(auc_lof_df.round(5))

mean_lof_auc = float(
    auc_lof_df["auc_lof_only"].mean()
)

print("Macro AUC LOF:", round(mean_lof_auc, 5))

assert mean_lof_auc > 0.5, (
    "Arah LOF tidak lebih baik daripada acak secara rata-rata. "
    "Hentikan eksperimen fusion."
)

print(
    "Arah LOF lolos pemeriksaan rata-rata. "
    "Ini belum membuktikan bahwa fusion bertambah baik."
)

# ============================================================
# 6. SIAPKAN POOL PREVALENCE-MATCHED
# ============================================================

TARGET_POS = 245
TARGET_NEG = 569

class_pools = {}

for cls in CLASS_NAMES:

    part = wide[
        wide["heldout_class"] == cls
    ].copy().reset_index(drop=True)

    target = part["is_pseudo_unknown"].to_numpy(dtype=int)

    positive_idx = np.flatnonzero(target == 1)
    negative_idx = np.flatnonzero(target == 0)

    n_pos = min(TARGET_POS, len(positive_idx))
    n_neg = int(round(n_pos * TARGET_NEG / TARGET_POS))

    if n_neg > len(negative_idx):
        raise ValueError(
            f"Negatif tidak cukup untuk kelas {cls}."
        )

    class_pools[cls] = {
        "data": part,
        "positive_idx": positive_idx,
        "negative_idx": negative_idx,
        "n_pos": n_pos,
        "n_neg": n_neg,
    }

    print(
        f"{cls}: {n_pos} positif + {n_neg} negatif; "
        f"prevalensi={n_pos/(n_pos+n_neg):.4f}"
    )

group_ids = np.array(
    sorted(wide["source_group"].unique()),
    dtype=object,
)

rng = np.random.default_rng(SEED)

# ============================================================
# 7. PAIRED SOURCE-GROUP BOOTSTRAP
# ============================================================

boot_records = []
valid_bootstrap = 0
attempts_total = 0

for draw in range(N_BOOTSTRAP):

    group_counts = None

    # Dengan hanya 15 grup, beberapa sampel bootstrap dapat
    # kehilangan seluruh grup positif/negatif suatu kelas.
    # Sampel seperti itu ditolak dan diambil ulang.
    for _ in range(1000):

        attempts_total += 1

        sampled_groups = rng.choice(
            group_ids,
            size=len(group_ids),
            replace=True,
        )

        counts = (
            pd.Series(sampled_groups)
            .value_counts()
            .to_dict()
        )

        valid_draw = True

        for cls in CLASS_NAMES:

            cfg = class_pools[cls]
            part = cfg["data"]

            row_weights = (
                part["source_group"]
                .map(counts)
                .fillna(0)
                .to_numpy(dtype=float)
            )

            pos_weights = row_weights[
                cfg["positive_idx"]
            ]

            neg_weights = row_weights[
                cfg["negative_idx"]
            ]

            if (
                pos_weights.sum() <= 0
                or neg_weights.sum() <= 0
            ):
                valid_draw = False
                break

        if valid_draw:
            group_counts = counts
            break

    if group_counts is None:
        continue

    valid_bootstrap += 1

    for cls in CLASS_NAMES:

        cfg = class_pools[cls]
        part = cfg["data"]

        positive_idx = cfg["positive_idx"]
        negative_idx = cfg["negative_idx"]

        n_pos = cfg["n_pos"]
        n_neg = cfg["n_neg"]

        row_weights = (
            part["source_group"]
            .map(group_counts)
            .fillna(0)
            .to_numpy(dtype=float)
        )

        pos_weights = row_weights[positive_idx]
        neg_weights = row_weights[negative_idx]

        pos_probs = pos_weights / pos_weights.sum()
        neg_probs = neg_weights / neg_weights.sum()

        # Pool yang sama digunakan oleh baseline serta kedua
        # bobot LOF, sehingga perbandingannya berpasangan.
        selected_pos = rng.choice(
            positive_idx,
            size=n_pos,
            replace=True,
            p=pos_probs,
        )

        selected_neg = rng.choice(
            negative_idx,
            size=n_neg,
            replace=True,
            p=neg_probs,
        )

        pool_idx = np.concatenate([
            selected_pos,
            selected_neg,
        ])

        rng.shuffle(pool_idx)

        pool = part.iloc[pool_idx].copy()

        target = pool[
            "is_pseudo_unknown"
        ].to_numpy(dtype=int)

        assert int(target.sum()) == n_pos

        sample_idx_pool = pool[
            "sample_idx"
        ].to_numpy(dtype=int)

        conf_rank = pd.Series(
            pool["confidence_ood_k3"].to_numpy(dtype=float)
        ).rank(
            pct=True,
            method="average",
        ).to_numpy()

        lof_rank = pd.Series(
            pool["lof_ood_score"].to_numpy(dtype=float)
        ).rank(
            pct=True,
            method="average",
        ).to_numpy()

        for k in K_VALUES:

            knn_col = f"knn_distance_k{k}"

            knn_rank = pd.Series(
                pool[knn_col].to_numpy(dtype=float)
            ).rank(
                pct=True,
                method="average",
            ).to_numpy()

            baseline_score = (
                0.5 * conf_rank
                + 0.5 * knn_rank
            )

            base_order = np.lexsort((
                sample_idx_pool,
                -baseline_score,
            ))

            baseline_rp = float(
                target[base_order[:n_pos]].mean()
            )

            for alpha in LOF_WEIGHTS:

                # LOF menjadi komponen ketiga.
                # Confidence dan kNN tetap 50:50 pada
                # bobot sisa (1-alpha).
                candidate_score = (
                    (1.0 - alpha) * baseline_score
                    + alpha * lof_rank
                )

                new_order = np.lexsort((
                    sample_idx_pool,
                    -candidate_score,
                ))

                new_rp = float(
                    target[new_order[:n_pos]].mean()
                )

                boot_records.append({
                    "draw": draw,
                    "heldout_class": cls,
                    "k": k,
                    "lof_weight": alpha,
                    "baseline_r_precision": baseline_rp,
                    "new_r_precision": new_rp,
                    "delta_r_precision": (
                        new_rp - baseline_rp
                    ),
                })

if valid_bootstrap < 300:
    raise RuntimeError(
        f"Hanya {valid_bootstrap} bootstrap valid. "
        "Hasil tidak cukup stabil untuk dievaluasi."
    )

boot_df = pd.DataFrame(boot_records)

print("\n" + "=" * 70)
print("BOOTSTRAP SELESAI")
print("=" * 70)
print("Bootstrap valid:", valid_bootstrap, "/", N_BOOTSTRAP)
print("Percobaan group draw:", attempts_total)

# ============================================================
# 8. RINGKAS HASIL MAKRO DAN PER KELAS
# ============================================================

class_summary = (
    boot_df.groupby(
        ["k", "lof_weight", "heldout_class"],
        as_index=False,
    )
    .agg(
        baseline_r_precision=(
            "baseline_r_precision", "mean"
        ),
        new_r_precision=(
            "new_r_precision", "mean"
        ),
        mean_delta=(
            "delta_r_precision", "mean"
        ),
        sd_delta=(
            "delta_r_precision", "std"
        ),
    )
)

macro_draw = (
    boot_df.groupby(
        ["draw", "k", "lof_weight"],
        as_index=False,
    )["delta_r_precision"]
    .mean()
    .rename(
        columns={
            "delta_r_precision": "macro_delta"
        }
    )
)

summary_rows = []

for k in K_VALUES:
    for alpha in LOF_WEIGHTS:

        per_draw = macro_draw[
            (macro_draw["k"] == k)
            & (macro_draw["lof_weight"] == alpha)
        ]

        delta_values = per_draw[
            "macro_delta"
        ].to_numpy(dtype=float)

        ci_low, ci_high = np.quantile(
            delta_values,
            [0.025, 0.975],
        )

        cpart = class_summary[
            (class_summary["k"] == k)
            & (class_summary["lof_weight"] == alpha)
        ]

        summary_rows.append({
            "k": k,
            "lof_weight": alpha,
            "mean_macro_delta": float(
                delta_values.mean()
            ),
            "bootstrap_ci95_low": float(ci_low),
            "bootstrap_ci95_high": float(ci_high),
            "ci_excludes_zero_positive": bool(ci_low > 0),
            "worst_mean_class_delta": float(
                cpart["mean_delta"].min()
            ),
            "no_class_drop_over_003": bool(
                cpart["mean_delta"].min()
                >= -MAX_CLASS_DROP
            ),
        })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("MAKRO R-PRECISION PREVALENCE-MATCHED")
print("=" * 70)

display(summary_df.round(5))

print("\n" + "=" * 70)
print("R-PRECISION PER KELAS")
print("=" * 70)

display(
    class_summary.sort_values(
        ["lof_weight", "heldout_class", "k"]
    ).round(5)
)

# ============================================================
# 9. GERBANG OFFLINE 1–4 DAN GERBANG 5
# ============================================================

gate_rows = []

for alpha in LOF_WEIGHTS:

    by_k = {}

    for k in K_VALUES:

        row = summary_df[
            (summary_df["k"] == k)
            & (summary_df["lof_weight"] == alpha)
        ].iloc[0]

        by_k[k] = row

    # Gerbang 1: kenaikan macro R-precision >= 0.03 pada
    # kedua k yang dipertimbangkan.
    gain_ok = all(
        float(by_k[k]["mean_macro_delta"])
        >= MIN_MACRO_RPRECISION_GAIN
        for k in K_VALUES
    )

    # Gerbang 4: CI 95% selisih berpasangan harus berada
    # di atas nol untuk kedua k.
    ci_ok = all(
        float(by_k[k]["bootstrap_ci95_low"]) > 0
        for k in K_VALUES
    )

    # Gerbang 2: tidak ada kelas yang turun lebih dari 0.03
    # pada setiap k yang diperiksa.
    class_part = class_summary[
        class_summary["lof_weight"] == alpha
    ]

    class_ok = bool(
        (class_part["mean_delta"] >= -MAX_CLASS_DROP).all()
    )

    # Gerbang 3: gain dan CI diperiksa pada k=3 sekaligus k=7;
    # jika keduanya lolos, hasil lebih konsisten antar-k.
    offline_gates_ok = gain_ok and ci_ok and class_ok

    # Gerbang 5: hitung perubahan ID pada test untuk k=3.
    shift_row = test_shift_df[
        (test_shift_df["k"] == 3)
        & np.isclose(
            test_shift_df["lof_weight"],
            alpha,
        )
    ].iloc[0]

    shift_ok = bool(
        shift_row["symmetric_difference_ids"]
        >= MIN_UNKNOWN_ID_CHANGES
    )

    # Bobot 0.10 hanya untuk eksplorasi.
    # Hanya 1/3 yang ditetapkan sebelumnya sebagai kandidat
    # yang boleh dipertimbangkan untuk submission.
    is_predeclared_submission_weight = bool(
        np.isclose(alpha, 1 / 3)
    )

    passes_all = bool(
        is_predeclared_submission_weight
        and offline_gates_ok
        and shift_ok
    )

    gate_rows.append({
        "lof_weight": alpha,
        "role": (
            "submission_candidate_if_all_gates_pass"
            if is_predeclared_submission_weight
            else "exploratory_only"
        ),
        "gain_ge_003_both_k": gain_ok,
        "bootstrap_ci_positive_both_k": ci_ok,
        "no_class_drop_gt_003": class_ok,
        "passes_offline_gates_1_to_4": offline_gates_ok,
        "test_unknown_id_changes": int(
            shift_row["symmetric_difference_ids"]
        ),
        "test_shift_ge_40": shift_ok,
        "passes_all_gates": passes_all,
    })

gate_df = pd.DataFrame(gate_rows)

print("\n" + "=" * 70)
print("GERBANG KEPUTUSAN LOF")
print("=" * 70)

display(gate_df)

# ============================================================
# 10. SIMPAN ARTEFAK ANALISIS
# ============================================================
# Tidak menyimpan submission baru.

auc_lof_path = os.path.join(
    OUT,
    "beats_lof_standalone_auc.csv",
)

class_path = os.path.join(
    OUT,
    "beats_lof_third_signal_per_class.csv",
)

summary_path = os.path.join(
    OUT,
    "beats_lof_third_signal_prevalence_matched_summary.csv",
)

bootstrap_path = os.path.join(
    OUT,
    "beats_lof_third_signal_group_bootstrap_draws.csv",
)

shift_path = os.path.join(
    OUT,
    "beats_lof_third_signal_test_id_shift.csv",
)

gate_path = os.path.join(
    OUT,
    "beats_lof_third_signal_gate.csv",
)

auc_lof_df.to_csv(auc_lof_path, index=False)
class_summary.to_csv(class_path, index=False)
summary_df.to_csv(summary_path, index=False)
boot_df.to_csv(bootstrap_path, index=False)
test_shift_df.to_csv(shift_path, index=False)
gate_df.to_csv(gate_path, index=False)

print("\n" + "=" * 70)
print("SELESAI")
print("=" * 70)

print("AUC LOF            :", auc_lof_path)
print("Per kelas          :", class_path)
print("R-precision makro  :", summary_path)
print("Bootstrap          :", bootstrap_path)
print("Pergeseran test ID :", shift_path)
print("Gerbang keputusan  :", gate_path)

if gate_df["passes_all_gates"].any():
    print(
        "\nAda satu konfigurasi yang lolos semua gerbang. "
        "Tinjau hasilnya sebelum membuat kandidat submission."
    )
else:
    print(
        "\nTidak ada konfigurasi yang lolos semua gerbang. "
        "Jangan buat submission baru; pertahankan champion."
    )

print("\nChampion tidak diubah dan tidak ada submission yang dibuat.")

INPUT
Rank scores : /kaggle/working/triple_v4_unknown_count_rank_scores.csv
LOF OOF     : /kaggle/working/beats_lof_relative_density_oof.csv
LOF test    : /kaggle/working/beats_lof_relative_density_test.csv
Champion k=3: /kaggle/working/submission_triple_v4_conf_knn50_k3_u245.csv
Champion k=7: /kaggle/working/submission_triple_v4_conf_knn50_u245.csv

ID Unknown berbeda k=3 vs k=7: 12
VALIDASI BASELINE TEST
k=3: ID Unknown cocok 245/245
k=7: ID Unknown cocok 244/245
LOLOS: baseline k=3 cocok persis.
PERINGATAN: baseline k=7 berbeda dari champion pada satu atau beberapa ID.
Untuk perbandingan pergeseran terhadap champion, gunakan daftar Unknown k=7 asli.

PERGESERAN ID UNKNOWN TEST


,k,lof_weight,unknown_retained,unknown_out,unknown_in,symmetric_difference_ids,meets_40_id_shift
0,3,0.100000,232,13,13,26,False
1,7,0.100000,231,14,14,28,False
2,3,0.333333,211,34,34,68,True
3,7,0.333333,211,34,34,68,True



LOCO rows: 5478
Source groups: 15

AUC LOF SENDIRI


,heldout_class,auc_lof_only
0,Bondol Jawa,0.67850
1,Cendrawasih,0.92761
2,Kutilang,0.63692


Macro AUC LOF: 0.74768
Arah LOF lolos pemeriksaan rata-rata. Ini belum membuktikan bahwa fusion bertambah baik.
Bondol Jawa: 245 positif + 569 negatif; prevalensi=0.3010
Cendrawasih: 83 positif + 193 negatif; prevalensi=0.3007
Kutilang: 245 positif + 569 negatif; prevalensi=0.3010

BOOTSTRAP SELESAI
Bootstrap valid: 500 / 500
Percobaan group draw: 513

MAKRO R-PRECISION PREVALENCE-MATCHED


,k,lof_weight,mean_macro_delta,bootstrap_ci95_low,bootstrap_ci95_high,ci_excludes_zero_positive,worst_mean_class_delta,no_class_drop_over_003
0,3,0.10000,0.01124,-0.00956,0.03294,False,-0.00615,True
1,3,0.33333,0.02799,-0.01922,0.07243,False,-0.03170,False
2,7,0.10000,0.01099,-0.00813,0.03168,False,-0.00678,True
3,7,0.33333,0.02971,-0.01837,0.07300,False,-0.02757,True



R-PRECISION PER KELAS


,k,lof_weight,heldout_class,baseline_r_precision,new_r_precision,mean_delta,sd_delta
0,3,0.10000,Bondol Jawa,0.55641,0.55026,-0.00615,0.01530
6,7,0.10000,Bondol Jawa,0.54596,0.53918,-0.00678,0.01510
1,3,0.10000,Cendrawasih,0.67952,0.71586,0.03634,0.02493
7,7,0.10000,Cendrawasih,0.68258,0.71961,0.03704,0.02532
2,3,0.10000,Kutilang,0.45726,0.46079,0.00353,0.01126
8,7,0.10000,Kutilang,0.44740,0.45012,0.00272,0.01135
3,3,0.33333,Bondol Jawa,0.55641,0.52471,-0.03170,0.03478
9,7,0.33333,Bondol Jawa,0.54596,0.51839,-0.02757,0.03452
4,3,0.33333,Cendrawasih,0.67952,0.78848,0.10896,0.05776
10,7,0.33333,Cendrawasih,0.68258,0.79089,0.10831,0.05647



GERBANG KEPUTUSAN LOF


,lof_weight,role,gain_ge_003_both_k,bootstrap_ci_positive_both_k,no_class_drop_gt_003,passes_offline_gates_1_to_4,test_unknown_id_changes,test_shift_ge_40,passes_all_gates
0,0.100000,exploratory_only,False,False,True,False,26,False,False
1,0.333333,submission_candidate_if_all_gates_pass,False,False,False,False,68,True,False



SELESAI
AUC LOF            : /kaggle/working/beats_lof_standalone_auc.csv
Per kelas          : /kaggle/working/beats_lof_third_signal_per_class.csv
R-precision makro  : /kaggle/working/beats_lof_third_signal_prevalence_matched_summary.csv
Bootstrap          : /kaggle/working/beats_lof_third_signal_group_bootstrap_draws.csv
Pergeseran test ID : /kaggle/working/beats_lof_third_signal_test_id_shift.csv
Gerbang keputusan  : /kaggle/working/beats_lof_third_signal_gate.csv

Tidak ada konfigurasi yang lolos semua gerbang. Jangan buat submission baru; pertahankan champion.

Champion tidak diubah dan tidak ada submission yang dibuat.


In [28]:
# ============================================================
# ANALISIS 34 ID KELUAR + 34 ID MASUK UNKNOWN
# Champion: k=3, confidence 50%, kNN 50%, Unknown=245
# Kandidat diagnostik saja: tambah LOF dengan bobot 1/3.
#
# Tidak membuat submission.
# Tidak mengubah file champion.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

OUT = "/kaggle/working"
DATA_ROOT = "/kaggle/input/competitions/schematics-2026"

N_UNKNOWN = 245
ALPHA_LOF = 1 / 3

SAMPLE_PATH = os.path.join(
    DATA_ROOT, "sample_submission.csv"
)
RANK_PATH = os.path.join(
    OUT, "triple_v4_unknown_count_rank_scores.csv"
)
LOF_TEST_PATH = os.path.join(
    OUT, "beats_lof_relative_density_test.csv"
)
TRAIN_EMB_PATH = os.path.join(
    OUT, "beats_iter3plus_as2m_8s_train_embeddings.npy"
)
TEST_EMB_PATH = os.path.join(
    OUT, "beats_iter3plus_as2m_8s_test_embeddings.npy"
)
CHAMPION_PATH = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_k3_u245.csv"
)

for path in [
    SAMPLE_PATH, RANK_PATH, LOF_TEST_PATH,
    TRAIN_EMB_PATH, TEST_EMB_PATH, CHAMPION_PATH
]:
    assert os.path.isfile(path), f"File tidak ditemukan: {path}"

# ------------------------------------------------------------
# 1. Muat file dan selaraskan berdasarkan file_id
# ------------------------------------------------------------

sample = pd.read_csv(SAMPLE_PATH)
rank_df = pd.read_csv(RANK_PATH)
lof_df = pd.read_csv(LOF_TEST_PATH)
champion = pd.read_csv(CHAMPION_PATH)

for df in [sample, rank_df, lof_df, champion]:
    df["file_id"] = df["file_id"].astype(str)

assert len(sample) == 814
assert sample["file_id"].is_unique
assert rank_df["file_id"].is_unique
assert lof_df["file_id"].is_unique
assert champion["file_id"].is_unique

base = sample[["file_id"]].merge(
    rank_df[
        [
            "file_id",
            "triple_v4_pred_known_class",
            "confidence_uncertainty_rank",
        ]
    ],
    on="file_id",
    how="left",
    validate="one_to_one",
)

base = base.merge(
    lof_df[["file_id", "lof_ood_score"]],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert base["confidence_uncertainty_rank"].notna().all()
assert base["lof_ood_score"].notna().all()

champion_unknown = set(
    champion.loc[
        champion["label"] == "Unknown", "file_id"
    ]
)

assert len(champion_unknown) == N_UNKNOWN

# ------------------------------------------------------------
# 2. Rekonstruksi skor baseline k=3
# ------------------------------------------------------------

train_emb = np.load(TRAIN_EMB_PATH).astype(np.float32)
test_emb = np.load(TEST_EMB_PATH).astype(np.float32)

assert train_emb.shape == (1826, 768)
assert test_emb.shape == (814, 768)

train_norm = normalize(train_emb, norm="l2")
test_norm = normalize(test_emb, norm="l2")

nn = NearestNeighbors(
    n_neighbors=3,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)
nn.fit(train_norm)

distances, _ = nn.kneighbors(test_norm)
distance_k3 = distances[:, :3].mean(axis=1)

confidence_rank = base[
    "confidence_uncertainty_rank"
].to_numpy(dtype=float)

knn_rank = pd.Series(distance_k3).rank(
    pct=True, method="average"
).to_numpy()

lof_rank = pd.Series(
    base["lof_ood_score"].to_numpy(dtype=float)
).rank(
    pct=True, method="average"
).to_numpy()

baseline_score = (
    0.5 * confidence_rank + 0.5 * knn_rank
)

# Verifikasi bahwa baseline k=3 menghasilkan Unknown champion
# yang sama persis sebelum melanjutkan.
sample_ids = sample["file_id"].to_numpy()

baseline_idx = np.argsort(
    -baseline_score, kind="stable"
)[:N_UNKNOWN]

baseline_unknown = set(sample_ids[baseline_idx])

assert baseline_unknown == champion_unknown, (
    "Baseline k=3 tidak cocok dengan champion. "
    "Hentikan analisis agar hasil tidak menyesatkan."
)

# ------------------------------------------------------------
# 3. Bentuk pilihan Unknown hipotetis dengan LOF 1/3
# ------------------------------------------------------------

lof_candidate_score = (
    (1 - ALPHA_LOF) * baseline_score
    + ALPHA_LOF * lof_rank
)

candidate_idx = np.argsort(
    -lof_candidate_score, kind="stable"
)[:N_UNKNOWN]

candidate_unknown = set(sample_ids[candidate_idx])

out_ids = champion_unknown - candidate_unknown
in_ids = candidate_unknown - champion_unknown

print("=" * 70)
print("RINGKASAN PERUBAHAN UNKNOWN")
print("=" * 70)
print("Keluar dari Unknown:", len(out_ids))
print("Masuk ke Unknown  :", len(in_ids))
print("Total beda ID     :", len(out_ids) + len(in_ids))

assert len(out_ids) == len(in_ids)

# ------------------------------------------------------------
# 4. Kelompokkan berdasarkan kelas prediksi sebelum reject
# ------------------------------------------------------------

base["champion_unknown"] = base["file_id"].isin(
    champion_unknown
)
base["candidate_unknown"] = base["file_id"].isin(
    candidate_unknown
)

base["perubahan"] = np.select(
    [
        base["champion_unknown"] & ~base["candidate_unknown"],
        ~base["champion_unknown"] & base["candidate_unknown"],
    ],
    [
        "Keluar dari Unknown",
        "Masuk ke Unknown",
    ],
    default="Tidak berubah",
)

base["baseline_score"] = baseline_score
base["candidate_score"] = lof_candidate_score
base["confidence_rank"] = confidence_rank
base["knn_rank"] = knn_rank
base["lof_rank"] = lof_rank
base["knn_distance_k3"] = distance_k3

changed = base[
    base["perubahan"] != "Tidak berubah"
].copy()

summary = (
    changed.groupby(
        ["perubahan", "triple_v4_pred_known_class"]
    )
    .size()
    .unstack(fill_value=0)
)

print("\n" + "=" * 70)
print("KOMPOSISI BERDASARKAN KELAS PREDIKSI")
print("=" * 70)
display(summary)

print("\nJumlah per kelas dan arah perubahan:")
display(
    changed.groupby(
        ["triple_v4_pred_known_class", "perubahan"]
    )
    .size()
    .unstack(fill_value=0)
)

# ------------------------------------------------------------
# 5. Daftar sampel yang berubah dengan skor pendukung
# ------------------------------------------------------------

detail_cols = [
    "file_id",
    "triple_v4_pred_known_class",
    "perubahan",
    "confidence_rank",
    "knn_rank",
    "lof_rank",
    "baseline_score",
    "candidate_score",
]

print("\n" + "=" * 70)
print("DETAIL ID YANG BERUBAH")
print("=" * 70)

display(
    changed[detail_cols]
    .sort_values(
        ["perubahan", "triple_v4_pred_known_class", "candidate_score"],
        ascending=[True, True, False],
    )
    .round(6)
)

# Simpan diagnostik saja.
diagnostic_path = os.path.join(
    OUT, "lof_1third_unknown_class_shift_diagnostic.csv"
)

changed[detail_cols].to_csv(
    diagnostic_path, index=False
)

print("\nFile diagnostik tersimpan:", diagnostic_path)
print("Champion tidak diubah; tidak ada submission dibuat.")

RINGKASAN PERUBAHAN UNKNOWN
Keluar dari Unknown: 34
Masuk ke Unknown  : 34
Total beda ID     : 68

KOMPOSISI BERDASARKAN KELAS PREDIKSI


triple_v4_pred_known_class,Bondol Jawa,Cendrawasih,Kutilang
perubahan,,,
Keluar dari Unknown,12,10,12
Masuk ke Unknown,8,22,4



Jumlah per kelas dan arah perubahan:


perubahan,Keluar dari Unknown,Masuk ke Unknown
triple_v4_pred_known_class,,
Bondol Jawa,12,8
Cendrawasih,10,22
Kutilang,12,4



DETAIL ID YANG BERUBAH


,file_id,triple_v4_pred_known_class,perubahan,confidence_rank,knn_rank,lof_rank,baseline_score,candidate_score
453,00c2af08a4d5a8c267ee1b2866bbacdd445754fe75c90e...,Bondol Jawa,Keluar dari Unknown,0.667076,0.678133,0.541769,0.672604,0.628993
654,5dad7ff00959d3dbb800364888224655e2a18f30a64ed3...,Bondol Jawa,Keluar dari Unknown,0.707617,0.611794,0.566339,0.659705,0.628583
115,af6a2c3d40b277bc62a784b06c2512a64ef88c345acbbf...,Bondol Jawa,Keluar dari Unknown,0.776413,0.646192,0.457002,0.711302,0.626536
765,433e8bb5ba700468ed87892b6909775617e0ee3272f702...,Bondol Jawa,Keluar dari Unknown,0.749386,0.683047,0.422604,0.716216,0.618346
395,4673d6e064bd8023cc534309b4d7d7632dece79ce83120...,Bondol Jawa,Keluar dari Unknown,0.939803,0.341523,0.523342,0.640663,0.601556
...,...,...,...,...,...,...,...,...
3,337e8971e3cec5155ec0ea5887c7d5cb360379574ab9e6...,Cendrawasih,Masuk ke Unknown,0.233415,0.936118,0.815725,0.584767,0.661753
252,b90d497b1dce64f9126b72c29340b815f8535e183ea577...,Kutilang,Masuk ke Unknown,0.405405,0.835381,0.825553,0.620393,0.688780
105,92bc42c0ed6c389bbc149a28c92213d1323cb81dc592bf...,Kutilang,Masuk ke Unknown,0.514742,0.756757,0.776413,0.635749,0.682637
768,dcedccb4364813d600d0178eb73d02ee7cbf8cc8598551...,Kutilang,Masuk ke Unknown,0.466830,0.719902,0.850123,0.593366,0.678952



File diagnostik tersimpan: /kaggle/working/lof_1third_unknown_class_shift_diagnostic.csv
Champion tidak diubah; tidak ada submission dibuat.


In [29]:
# ============================================================
# BEATS CLASS-CONDITIONAL MAHALANOBIS
# PCA 128 + SHARED SHRINKAGE COVARIANCE
#
# Champion:
#   k=3, fusion 50:50, 245 Unknown, Public LB 0.68379
#
# Bobot Mahalanobis:
#   0.10  : eksplorasi
#   1/3   : kandidat yang boleh dipertimbangkan
#
# Tidak ada training ulang model audio.
# Tidak membuat atau mengubah submission.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score

# ============================================================
# 1. KONFIGURASI
# ============================================================

DATA_ROOT = "/kaggle/input/competitions/schematics-2026"
OUT = "/kaggle/working"

CLASS_NAMES = [
    "Bondol Jawa",
    "Cendrawasih",
    "Kutilang",
]

K_VALUES = [3, 7]
PCA_DIM = 128
MAHA_WEIGHTS = [0.10, 1 / 3]

N_UNKNOWN = 245
N_KNOWN = 569

N_BOOTSTRAP = 500
SEED = 20261020

MIN_MACRO_GAIN = 0.03
MAX_CLASS_DROP = 0.03
MIN_ID_SHIFT = 40

# ============================================================
# 2. CARI INPUT
# ============================================================

def find_file(filename, preferred_paths=None):
    preferred_paths = preferred_paths or []

    for path in preferred_paths:
        if path and os.path.isfile(path):
            return path

    matches = []

    for root in [OUT, "/kaggle/input"]:
        if os.path.isdir(root):
            matches.extend(
                glob.glob(
                    os.path.join(root, "**", filename),
                    recursive=True
                )
            )

    if matches:
        matches.sort(key=lambda p: (len(p), p))
        return matches[0]

    raise FileNotFoundError(
        f"File tidak ditemukan: {filename}"
    )


metadata_path = os.path.join(
    DATA_ROOT, "train", "metadata.csv"
)

fold_path = find_file(
    "bird_group_folds_seed16.csv",
    [
        "/kaggle/input/notebooks/radithiafatir/"
        "testttrisan/bird_group_folds_seed16.csv",
        os.path.join(OUT, "bird_group_folds_seed16.csv"),
    ],
)

sample_path = os.path.join(
    DATA_ROOT, "sample_submission.csv"
)

train_emb_path = find_file(
    "beats_iter3plus_as2m_8s_train_embeddings.npy"
)

test_emb_path = find_file(
    "beats_iter3plus_as2m_8s_test_embeddings.npy"
)

rank_path = os.path.join(
    OUT, "triple_v4_unknown_count_rank_scores.csv"
)

champion_k3_path = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_k3_u245.csv"
)

champion_k7_path = os.path.join(
    OUT, "submission_triple_v4_conf_knn50_u245.csv"
)

for path in [
    metadata_path, fold_path, sample_path,
    train_emb_path, test_emb_path, rank_path,
    champion_k3_path, champion_k7_path,
]:
    assert os.path.isfile(path), f"File tidak ditemukan: {path}"

print("=" * 70)
print("INPUT")
print("=" * 70)
print("Metadata :", metadata_path)
print("Folds    :", fold_path)
print("Train emb:", train_emb_path)
print("Test emb :", test_emb_path)
print("Rank CSV :", rank_path)

# ============================================================
# 3. LOAD DAN VALIDASI DATA
# ============================================================

meta = pd.read_csv(metadata_path)
fold_df = pd.read_csv(fold_path)
sample = pd.read_csv(sample_path)
rank_df = pd.read_csv(rank_path)

for df in [meta, fold_df, sample, rank_df]:
    df["file_id"] = df["file_id"].astype(str)
    assert df["file_id"].is_unique

meta = meta.merge(
    fold_df[["file_id", "fold"]],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert meta["fold"].notna().all()
meta["fold"] = meta["fold"].astype(int)

label_to_idx = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

y_series = meta["label"].map(label_to_idx)
assert y_series.notna().all()

y = y_series.to_numpy(dtype=int)
folds = meta["fold"].to_numpy()
fold_values = sorted(np.unique(folds))

train_emb = np.load(train_emb_path).astype(np.float32)
test_emb = np.load(test_emb_path).astype(np.float32)

assert train_emb.shape == (len(meta), 768)
assert test_emb.shape == (len(sample), 768)
assert len(sample) == 814

assert np.isfinite(train_emb).all()
assert np.isfinite(test_emb).all()

leakage = (
    meta.groupby("source_group")["fold"].nunique() > 1
).sum()

assert leakage == 0, "Terdeteksi source_group bocor antar-fold."

print("\nTrain:", len(meta))
print("Test:", len(sample))
print("Source group:", meta["source_group"].nunique())
print("Fold counts:", meta["fold"].value_counts().sort_index().to_dict())
print("Source-group leakage:", int(leakage))

# ============================================================
# 4. FUNGSI MAHALANOBIS
# ============================================================
# Fit scaler + PCA hanya pada training known.
# Estimasi kovarians shrinkage dilakukan pada residual
# terhadap mean kelas, dengan satu kovarians bersama.
#
# Skor OOD = jarak Mahalanobis minimum ke centroid kelas.
# Skor lebih tinggi berarti lebih jauh dari kelas dikenal.

def mahalanobis_ood_score(
    x_train,
    y_train,
    x_eval,
    n_components=PCA_DIM,
):
    scaler = StandardScaler()

    z_train = scaler.fit_transform(x_train)
    z_eval = scaler.transform(x_eval)

    n_comp = min(
        n_components,
        z_train.shape[1],
        z_train.shape[0] - 1,
    )

    pca = PCA(
        n_components=n_comp,
        svd_solver="randomized",
        random_state=SEED,
    )

    p_train = pca.fit_transform(z_train)
    p_eval = pca.transform(z_eval)

    classes = np.unique(y_train)
    if len(classes) < 2:
        raise ValueError(
            "Setidaknya dua kelas dikenal diperlukan."
        )

    means = {}
    residual_parts = []

    for cls in classes:
        class_vectors = p_train[y_train == cls]

        if len(class_vectors) < 2:
            raise ValueError(
                f"Sampel terlalu sedikit untuk kelas {cls}"
            )

        mean_vector = class_vectors.mean(axis=0)
        means[cls] = mean_vector

        residual_parts.append(
            class_vectors - mean_vector
        )

    # Kovarians bersama dari residual within-class.
    residuals = np.vstack(residual_parts)

    covariance_model = LedoitWolf(
        assume_centered=True
    )
    covariance_model.fit(residuals)

    precision = covariance_model.precision_

    distances = []

    for cls in classes:
        difference = p_eval - means[cls]

        d2 = np.einsum(
            "ij,jk,ik->i",
            difference,
            precision,
            difference,
        )

        distances.append(d2)

    distance_matrix = np.column_stack(distances)

    # Dekat ke salah satu kelas dikenal => bukan outlier kuat.
    return np.min(distance_matrix, axis=1)


# ============================================================
# 5. VALIDASI BASELINE TEST SEBELUM LOCO
# ============================================================

rank_required = {
    "file_id",
    "confidence_uncertainty_rank",
    "combined_rank_score_50_50",
}

assert rank_required.issubset(rank_df.columns), (
    f"Kolom rank CSV tidak lengkap: {rank_df.columns.tolist()}"
)

champion_k3 = pd.read_csv(champion_k3_path)
champion_k7 = pd.read_csv(champion_k7_path)

for df in [champion_k3, champion_k7]:
    df["file_id"] = df["file_id"].astype(str)
    assert len(df) == 814
    assert df["file_id"].is_unique

sample_ids = sample["file_id"].astype(str).to_numpy()

k3_unknown = set(
    champion_k3.loc[
        champion_k3["label"] == "Unknown", "file_id"
    ]
)

k7_unknown = set(
    champion_k7.loc[
        champion_k7["label"] == "Unknown", "file_id"
    ]
)

assert len(k3_unknown) == N_UNKNOWN
assert len(k7_unknown) == N_UNKNOWN

rank_test = sample[["file_id"]].merge(
    rank_df[
        [
            "file_id",
            "confidence_uncertainty_rank",
            "combined_rank_score_50_50",
        ]
    ],
    on="file_id",
    how="left",
    validate="one_to_one",
)

assert rank_test[
    "confidence_uncertainty_rank"
].notna().all()

assert rank_test[
    "combined_rank_score_50_50"
].notna().all()

confidence_rank_test = rank_test[
    "confidence_uncertainty_rank"
].to_numpy(dtype=float)

# Hitung kNN jarak pada test.
train_norm = normalize(train_emb, norm="l2")
test_norm = normalize(test_emb, norm="l2")

knn_test = NearestNeighbors(
    n_neighbors=max(K_VALUES),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

knn_test.fit(train_norm)
test_distances, _ = knn_test.kneighbors(test_norm)

baseline_test_scores = {}

# k=3: rekonstruksi dari pipeline yang sudah tervalidasi.
distance_k3_test = test_distances[:, :3].mean(axis=1)

knn_rank_k3_test = pd.Series(
    distance_k3_test
).rank(
    pct=True,
    method="average",
).to_numpy()

baseline_test_scores[3] = (
    0.5 * confidence_rank_test
    + 0.5 * knn_rank_k3_test
)

# k=7: gunakan skor fusion lama dari artefak asli.
# Ini menghindari menganggap hitung ulang jarak k=7 identik
# dengan pipeline lama, yang sebelumnya berbeda pada satu ID.
baseline_test_scores[7] = rank_test[
    "combined_rank_score_50_50"
].to_numpy(dtype=float)

def top_unknown_ids(scores):
    idx = np.argsort(
        -np.asarray(scores),
        kind="stable",
    )[:N_UNKNOWN]

    return set(sample_ids[idx])


baseline_k3_ids = top_unknown_ids(
    baseline_test_scores[3]
)

baseline_k7_ids = top_unknown_ids(
    baseline_test_scores[7]
)

print("\n" + "=" * 70)
print("VALIDASI BASELINE")
print("=" * 70)

print(
    "k=3 cocok dengan champion:",
    len(baseline_k3_ids & k3_unknown),
    "/", N_UNKNOWN,
)

print(
    "k=7 cocok dengan champion:",
    len(baseline_k7_ids & k7_unknown),
    "/", N_UNKNOWN,
)

assert baseline_k3_ids == k3_unknown, (
    "Baseline k=3 tidak cocok persis. "
    "Evaluasi dihentikan."
)

print("LOLOS: champion aktif k=3 direkonstruksi persis.")

# ============================================================
# 6. TEST MAHALANOBIS
# ============================================================

print("\n" + "=" * 70)
print("MAHALANOBIS TEST")
print("=" * 70)

maha_test = mahalanobis_ood_score(
    train_emb,
    y,
    test_emb,
    n_components=PCA_DIM,
)

assert len(maha_test) == len(sample)
assert np.isfinite(maha_test).all()

maha_rank_test = pd.Series(
    maha_test
).rank(
    pct=True,
    method="average",
).to_numpy()

test_shift_rows = []

for alpha in MAHA_WEIGHTS:
    for k in K_VALUES:

        candidate_score = (
            (1.0 - alpha) * baseline_test_scores[k]
            + alpha * maha_rank_test
        )

        candidate_ids = top_unknown_ids(candidate_score)

        # Referensi harus berupa file champion asli.
        reference_ids = (
            k3_unknown if k == 3 else k7_unknown
        )

        test_shift_rows.append({
            "k": k,
            "maha_weight": alpha,
            "unknown_retained": len(
                candidate_ids & reference_ids
            ),
            "unknown_out": len(
                reference_ids - candidate_ids
            ),
            "unknown_in": len(
                candidate_ids - reference_ids
            ),
            "symmetric_difference_ids": len(
                reference_ids ^ candidate_ids
            ),
        })

test_shift_df = pd.DataFrame(test_shift_rows)

print("\nPergeseran ID Unknown test:")
display(test_shift_df)

# ============================================================
# 7. GENERATE OOF LOCO MAHALANOBIS + BASELINE
# ============================================================
# Validasi per group-fold.
# Setiap heldout class dikeluarkan dari training known.
# Catatan: ini tetap simulasi pseudo-Unknown, bukan test Unknown asli.

print("\n" + "=" * 70)
print("GENERATE OOF LOCO")
print("=" * 70)

oof_records = []

for heldout_idx, heldout_name in enumerate(CLASS_NAMES):

    print(f"\nHeldout class: {heldout_name}")

    for fold in fold_values:

        tr_idx = np.flatnonzero(
            (folds != fold)
            & (y != heldout_idx)
        )

        va_idx = np.flatnonzero(folds == fold)

        y_train = y[tr_idx]
        x_train = train_emb[tr_idx]
        x_valid = train_emb[va_idx]

        # Confidence OOD yang sama seperti simulasi LOCO sebelumnya.
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.03,
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED,
            ),
        )

        clf.fit(x_train, y_train)

        p_valid = clf.predict_proba(x_valid)
        confidence_ood = 1.0 - p_valid.max(axis=1)

        # BEATs kNN: gunakan embedding L2-normalized dan cosine.
        x_train_norm = normalize(x_train, norm="l2")
        x_valid_norm = normalize(x_valid, norm="l2")

        knn = NearestNeighbors(
            n_neighbors=max(K_VALUES),
            metric="cosine",
            algorithm="brute",
            n_jobs=-1,
        )

        knn.fit(x_train_norm)

        fold_distances, _ = knn.kneighbors(
            x_valid_norm
        )

        knn_d3 = fold_distances[:, :3].mean(axis=1)
        knn_d7 = fold_distances[:, :7].mean(axis=1)

        # Sinyal Mahalanobis dilatih hanya dengan known train.
        maha_score = mahalanobis_ood_score(
            x_train,
            y_train,
            x_valid,
            n_components=PCA_DIM,
        )

        target = (
            y[va_idx] == heldout_idx
        ).astype(int)

        for j, sample_idx in enumerate(va_idx):

            oof_records.append({
                "sample_idx": int(sample_idx),
                "fold": int(fold),
                "heldout_class": heldout_name,
                "source_group": str(
                    meta.iloc[sample_idx]["source_group"]
                ),
                "is_pseudo_unknown": int(target[j]),
                "confidence_ood": float(
                    confidence_ood[j]
                ),
                "knn_distance_k3": float(knn_d3[j]),
                "knn_distance_k7": float(knn_d7[j]),
                "maha_ood_score": float(maha_score[j]),
            })

        print(
            f"  Fold {fold}: "
            f"train={len(tr_idx)}, validasi={len(va_idx)}"
        )

oof = pd.DataFrame(oof_records)

assert len(oof) == len(meta) * len(CLASS_NAMES)
assert not oof.duplicated(
    ["sample_idx", "fold", "heldout_class"]
).any()

print("\nShape OOF:", oof.shape)
print("Source groups:", oof["source_group"].nunique())

# ============================================================
# 8. AUC MAHALANOBIS SENDIRI
# ============================================================

auc_rows = []

for cls in CLASS_NAMES:

    part = oof[
        oof["heldout_class"] == cls
    ]

    auc = roc_auc_score(
        part["is_pseudo_unknown"],
        part["maha_ood_score"],
    )

    auc_rows.append({
        "heldout_class": cls,
        "auc_mahalanobis": auc,
    })

auc_df = pd.DataFrame(auc_rows)

print("\n" + "=" * 70)
print("AUC MAHALANOBIS SENDIRI")
print("=" * 70)

display(auc_df.round(5))

macro_auc_maha = float(
    auc_df["auc_mahalanobis"].mean()
)

print("Macro AUC Mahalanobis:", round(macro_auc_maha, 5))

if macro_auc_maha <= 0.5:
    raise RuntimeError(
        "Arah Mahalanobis tidak lebih baik dari acak secara rata-rata. "
        "Hentikan evaluasi fusion."
    )

# ============================================================
# 9. POOL PREVALENCE-MATCHED
# ============================================================

class_pools = {}

for cls in CLASS_NAMES:

    part = oof[
        oof["heldout_class"] == cls
    ].copy().reset_index(drop=True)

    target = part[
        "is_pseudo_unknown"
    ].to_numpy(dtype=int)

    positive_idx = np.flatnonzero(target == 1)
    negative_idx = np.flatnonzero(target == 0)

    n_pos = min(N_UNKNOWN, len(positive_idx))
    n_neg = int(round(n_pos * N_KNOWN / N_UNKNOWN))

    if n_neg > len(negative_idx):
        raise ValueError(
            f"Negatif tidak cukup untuk {cls}."
        )

    class_pools[cls] = {
        "data": part,
        "pos_idx": positive_idx,
        "neg_idx": negative_idx,
        "n_pos": n_pos,
        "n_neg": n_neg,
    }

    print(
        f"{cls}: {n_pos} pseudo-Unknown + {n_neg} known, "
        f"prevalensi={n_pos/(n_pos+n_neg):.4f}"
    )

# ============================================================
# 10. PAIRED SOURCE-GROUP BOOTSTRAP
# ============================================================

rng = np.random.default_rng(SEED)

groups = np.array(
    sorted(oof["source_group"].astype(str).unique()),
    dtype=object,
)

N_GROUPS = len(groups)

boot_rows = []
valid_bootstrap = 0
total_attempts = 0

for draw in range(N_BOOTSTRAP):

    group_counts = None

    # Retry hanya jika satu kelas kehilangan seluruh grup positif
    # atau negatif dalam bootstrap tersebut.
    for _ in range(1000):

        total_attempts += 1

        sampled_groups = rng.choice(
            groups,
            size=N_GROUPS,
            replace=True,
        )

        counts = (
            pd.Series(sampled_groups)
            .value_counts()
            .to_dict()
        )

        valid = True

        for cls in CLASS_NAMES:

            cfg = class_pools[cls]
            part = cfg["data"]

            row_weights = (
                part["source_group"]
                .map(counts)
                .fillna(0)
                .to_numpy(dtype=float)
            )

            if (
                row_weights[cfg["pos_idx"]].sum() <= 0
                or row_weights[cfg["neg_idx"]].sum() <= 0
            ):
                valid = False
                break

        if valid:
            group_counts = counts
            break

    if group_counts is None:
        continue

    valid_bootstrap += 1

    for cls in CLASS_NAMES:

        cfg = class_pools[cls]
        part = cfg["data"]

        pos_idx = cfg["pos_idx"]
        neg_idx = cfg["neg_idx"]

        n_pos = cfg["n_pos"]
        n_neg = cfg["n_neg"]

        row_weights = (
            part["source_group"]
            .map(group_counts)
            .fillna(0)
            .to_numpy(dtype=float)
        )

        pos_weights = row_weights[pos_idx]
        neg_weights = row_weights[neg_idx]

        pos_prob = pos_weights / pos_weights.sum()
        neg_prob = neg_weights / neg_weights.sum()

        # Pool berpasangan untuk baseline dan semua alpha.
        selected_pos = rng.choice(
            pos_idx,
            size=n_pos,
            replace=True,
            p=pos_prob,
        )

        selected_neg = rng.choice(
            neg_idx,
            size=n_neg,
            replace=True,
            p=neg_prob,
        )

        pool_idx = np.concatenate([
            selected_pos,
            selected_neg,
        ])

        rng.shuffle(pool_idx)

        pool = part.iloc[pool_idx]

        target = pool[
            "is_pseudo_unknown"
        ].to_numpy(dtype=int)

        sample_idx_pool = pool[
            "sample_idx"
        ].to_numpy(dtype=int)

        conf_rank = pd.Series(
            pool["confidence_ood"].to_numpy(dtype=float)
        ).rank(
            pct=True,
            method="average",
        ).to_numpy()

        maha_rank = pd.Series(
            pool["maha_ood_score"].to_numpy(dtype=float)
        ).rank(
            pct=True,
            method="average",
        ).to_numpy()

        R = int(target.sum())
        assert R == n_pos

        for k in K_VALUES:

            knn_col = f"knn_distance_k{k}"

            knn_rank = pd.Series(
                pool[knn_col].to_numpy(dtype=float)
            ).rank(
                pct=True,
                method="average",
            ).to_numpy()

            baseline_score = (
                0.5 * conf_rank
                + 0.5 * knn_rank
            )

            base_order = np.lexsort((
                sample_idx_pool,
                -baseline_score,
            ))

            baseline_rp = float(
                target[base_order[:R]].mean()
            )

            for alpha in MAHA_WEIGHTS:

                candidate_score = (
                    (1.0 - alpha) * baseline_score
                    + alpha * maha_rank
                )

                new_order = np.lexsort((
                    sample_idx_pool,
                    -candidate_score,
                ))

                new_rp = float(
                    target[new_order[:R]].mean()
                )

                boot_rows.append({
                    "draw": draw,
                    "heldout_class": cls,
                    "k": k,
                    "maha_weight": alpha,
                    "baseline_r_precision": baseline_rp,
                    "new_r_precision": new_rp,
                    "delta_r_precision": new_rp - baseline_rp,
                })

if valid_bootstrap < 300:
    raise RuntimeError(
        f"Bootstrap valid hanya {valid_bootstrap}/{N_BOOTSTRAP}. "
        "Hasil belum cukup stabil."
    )

boot_df = pd.DataFrame(boot_rows)

print("\nBootstrap valid:", valid_bootstrap, "/", N_BOOTSTRAP)
print("Total percobaan group draw:", total_attempts)

# ============================================================
# 11. AGREGASI MACRO, PER KELAS, DAN CI
# ============================================================

class_summary = (
    boot_df.groupby(
        ["k", "maha_weight", "heldout_class"],
        as_index=False,
    )
    .agg(
        baseline_r_precision=(
            "baseline_r_precision", "mean"
        ),
        new_r_precision=(
            "new_r_precision", "mean"
        ),
        mean_delta=(
            "delta_r_precision", "mean"
        ),
        sd_delta=(
            "delta_r_precision", "std"
        ),
    )
)

macro_draw = (
    boot_df.groupby(
        ["draw", "k", "maha_weight"],
        as_index=False,
    )["delta_r_precision"]
    .mean()
    .rename(
        columns={
            "delta_r_precision": "macro_delta"
        }
    )
)

summary_rows = []

for k in K_VALUES:
    for alpha in MAHA_WEIGHTS:

        part = macro_draw[
            (macro_draw["k"] == k)
            & (macro_draw["maha_weight"] == alpha)
        ]

        delta = part["macro_delta"].to_numpy(dtype=float)

        ci_low, ci_high = np.quantile(
            delta, [0.025, 0.975]
        )

        class_part = class_summary[
            (class_summary["k"] == k)
            & (class_summary["maha_weight"] == alpha)
        ]

        summary_rows.append({
            "k": k,
            "maha_weight": alpha,
            "mean_macro_delta": float(delta.mean()),
            "bootstrap_ci95_low": float(ci_low),
            "bootstrap_ci95_high": float(ci_high),
            "ci_lower_above_zero": bool(ci_low > 0),
            "worst_class_mean_delta": float(
                class_part["mean_delta"].min()
            ),
            "no_class_drop_over_003": bool(
                (class_part["mean_delta"] >= -MAX_CLASS_DROP).all()
            ),
        })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("HASIL MACRO R-PRECISION PREVALENCE-MATCHED")
print("=" * 70)

display(summary_df.round(5))

print("\n" + "=" * 70)
print("HASIL PER KELAS")
print("=" * 70)

display(
    class_summary.sort_values(
        ["maha_weight", "heldout_class", "k"]
    ).round(5)
)

# ============================================================
# 12. GERBANG KEPUTUSAN
# ============================================================

gate_rows = []

for alpha in MAHA_WEIGHTS:

    per_k = {
        k: summary_df[
            (summary_df["k"] == k)
            & (summary_df["maha_weight"] == alpha)
        ].iloc[0]
        for k in K_VALUES
    }

    gain_ok = all(
        per_k[k]["mean_macro_delta"] >= MIN_MACRO_GAIN
        for k in K_VALUES
    )

    ci_ok = all(
        per_k[k]["bootstrap_ci95_low"] > 0
        for k in K_VALUES
    )

    class_part = class_summary[
        class_summary["maha_weight"] == alpha
    ]

    class_ok = bool(
        (class_part["mean_delta"] >= -MAX_CLASS_DROP).all()
    )

    offline_ok = gain_ok and ci_ok and class_ok

    shift_row = test_shift_df[
        (test_shift_df["k"] == 3)
        & np.isclose(
            test_shift_df["maha_weight"],
            alpha,
        )
    ].iloc[0]

    shift_ok = bool(
        shift_row["symmetric_difference_ids"]
        >= MIN_ID_SHIFT
    )

    # Hanya bobot 1/3 yang sejak awal ditetapkan sebagai
    # kandidat yang boleh dipertimbangkan untuk submission.
    submission_weight = bool(
        np.isclose(alpha, 1 / 3)
    )

    passes_all = bool(
        submission_weight
        and offline_ok
        and shift_ok
    )

    gate_rows.append({
        "maha_weight": alpha,
        "role": (
            "candidate_if_all_gates_pass"
            if submission_weight
            else "exploratory_only"
        ),
        "gain_ge_003_both_k": gain_ok,
        "CI_positive_both_k": ci_ok,
        "no_class_drop_gt_003": class_ok,
        "passes_offline_gates_1_to_4": offline_ok,
        "test_unknown_id_changes_k3": int(
            shift_row["symmetric_difference_ids"]
        ),
        "test_shift_ge_40": shift_ok,
        "passes_all_gates": passes_all,
    })

gate_df = pd.DataFrame(gate_rows)

print("\n" + "=" * 70)
print("GERBANG KEPUTUSAN MAHALANOBIS")
print("=" * 70)

display(gate_df)

# ============================================================
# 13. SIMPAN ARTEFAK ANALISIS
# ============================================================
# Hanya diagnostik. Tidak ada submission yang dibuat.

files_to_save = {
    "beats_mahalanobis_oof.csv": oof,
    "beats_mahalanobis_auc_by_class.csv": auc_df,
    "beats_mahalanobis_prevalence_matched_summary.csv": summary_df,
    "beats_mahalanobis_per_class.csv": class_summary,
    "beats_mahalanobis_group_bootstrap_draws.csv": boot_df,
    "beats_mahalanobis_test_id_shift.csv": test_shift_df,
    "beats_mahalanobis_gate.csv": gate_df,
}

for filename, frame in files_to_save.items():
    path = os.path.join(OUT, filename)
    frame.to_csv(path, index=False)
    print("Tersimpan:", path)

print("\nTidak ada submission dibuat atau diubah.")

if gate_df["passes_all_gates"].any():
    print(
        "\nAda kandidat Mahalanobis yang lolos gerbang. "
        "Tinjau tabelnya sebelum membuat submission."
    )
else:
    print(
        "\nTidak ada konfigurasi Mahalanobis yang lolos semua gerbang. "
        "Pertahankan champion k=3, fusion 50:50, 245 Unknown."
    )

INPUT
Metadata : /kaggle/input/competitions/schematics-2026/train/metadata.csv
Folds    : /kaggle/input/notebooks/radithiafatir/testttrisan/bird_group_folds_seed16.csv
Train emb: /kaggle/working/beats_iter3plus_as2m_8s_train_embeddings.npy
Test emb : /kaggle/working/beats_iter3plus_as2m_8s_test_embeddings.npy
Rank CSV : /kaggle/working/triple_v4_unknown_count_rank_scores.csv

Train: 1826
Test: 814
Source group: 15
Fold counts: {0: 284, 1: 495, 2: 553, 3: 494}
Source-group leakage: 0

VALIDASI BASELINE
k=3 cocok dengan champion: 245 / 245
k=7 cocok dengan champion: 245 / 245
LOLOS: champion aktif k=3 direkonstruksi persis.

MAHALANOBIS TEST

Pergeseran ID Unknown test:


,k,maha_weight,unknown_retained,unknown_out,unknown_in,symmetric_difference_ids
0,3,0.100000,234,11,11,22
1,7,0.100000,232,13,13,26
2,3,0.333333,215,30,30,60
3,7,0.333333,215,30,30,60



GENERATE OOF LOCO

Heldout class: Bondol Jawa
  Fold 0: train=535, validasi=284
  Fold 1: train=521, validasi=495
  Fold 2: train=543, validasi=553
  Fold 3: train=507, validasi=494

Heldout class: Cendrawasih
  Fold 0: train=1494, validasi=284
  Fold 1: train=1251, validasi=495
  Fold 2: train=1206, validasi=553
  Fold 3: train=1278, validasi=494

Heldout class: Kutilang
  Fold 0: train=1055, validasi=284
  Fold 1: train=890, validasi=495
  Fold 2: train=797, validasi=553
  Fold 3: train=879, validasi=494

Shape OOF: (5478, 9)
Source groups: 15

AUC MAHALANOBIS SENDIRI


,heldout_class,auc_mahalanobis
0,Bondol Jawa,0.76847
1,Cendrawasih,0.93669
2,Kutilang,0.68963


Macro AUC Mahalanobis: 0.79826
Bondol Jawa: 245 pseudo-Unknown + 569 known, prevalensi=0.3010
Cendrawasih: 83 pseudo-Unknown + 193 known, prevalensi=0.3007
Kutilang: 245 pseudo-Unknown + 569 known, prevalensi=0.3010

Bootstrap valid: 500 / 500
Total percobaan group draw: 513

HASIL MACRO R-PRECISION PREVALENCE-MATCHED


,k,maha_weight,mean_macro_delta,bootstrap_ci95_low,bootstrap_ci95_high,ci_lower_above_zero,worst_class_mean_delta,no_class_drop_over_003
0,3,0.10000,0.01710,-0.00292,0.04227,False,0.00259,True
1,3,0.33333,0.04270,-0.00086,0.08524,False,0.00287,True
2,7,0.10000,0.01799,-0.00353,0.04097,False,0.00482,True
3,7,0.33333,0.04505,0.00647,0.08849,True,0.00845,True



HASIL PER KELAS


,k,maha_weight,heldout_class,baseline_r_precision,new_r_precision,mean_delta,sd_delta
0,3,0.10000,Bondol Jawa,0.55641,0.55900,0.00259,0.01303
6,7,0.10000,Bondol Jawa,0.54596,0.55078,0.00482,0.01287
1,3,0.10000,Cendrawasih,0.67952,0.72186,0.04234,0.02985
7,7,0.10000,Cendrawasih,0.68258,0.72347,0.04089,0.02921
2,3,0.10000,Kutilang,0.45726,0.46362,0.00637,0.01100
8,7,0.10000,Kutilang,0.44740,0.45565,0.00824,0.01123
3,3,0.33333,Bondol Jawa,0.55641,0.55928,0.00287,0.02908
9,7,0.33333,Bondol Jawa,0.54596,0.55441,0.00845,0.02915
4,3,0.33333,Cendrawasih,0.67952,0.79007,0.11055,0.05856
10,7,0.33333,Cendrawasih,0.68258,0.79034,0.10776,0.05859



GERBANG KEPUTUSAN MAHALANOBIS


,maha_weight,role,gain_ge_003_both_k,CI_positive_both_k,no_class_drop_gt_003,passes_offline_gates_1_to_4,test_unknown_id_changes_k3,test_shift_ge_40,passes_all_gates
0,0.100000,exploratory_only,False,False,True,False,22,False,False
1,0.333333,candidate_if_all_gates_pass,True,False,True,False,60,True,False


Tersimpan: /kaggle/working/beats_mahalanobis_oof.csv
Tersimpan: /kaggle/working/beats_mahalanobis_auc_by_class.csv
Tersimpan: /kaggle/working/beats_mahalanobis_prevalence_matched_summary.csv
Tersimpan: /kaggle/working/beats_mahalanobis_per_class.csv
Tersimpan: /kaggle/working/beats_mahalanobis_group_bootstrap_draws.csv
Tersimpan: /kaggle/working/beats_mahalanobis_test_id_shift.csv
Tersimpan: /kaggle/working/beats_mahalanobis_gate.csv

Tidak ada submission dibuat atau diubah.

Tidak ada konfigurasi Mahalanobis yang lolos semua gerbang. Pertahankan champion k=3, fusion 50:50, 245 Unknown.


In [30]:
# ============================================================
# MAHALANOBIS: KONFIRMASI BOOTSTRAP 3000 REPLIKASI
# ============================================================
# Tidak menghitung ulang embedding.
# Tidak mengubah bobot atau gerbang keputusan.
# Tidak membuat submission.
#
# Menggunakan:
#   beats_mahalanobis_oof.csv
#   beats_mahalanobis_test_id_shift.csv
#
# Bobot kandidat: Mahalanobis 1/3.
# Baseline: confidence/kNN 50:50.
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. KONFIGURASI TETAP
# ------------------------------------------------------------

OUT = "/kaggle/working"

CLASS_NAMES = [
    "Bondol Jawa",
    "Cendrawasih",
    "Kutilang",
]

K_VALUES = [3, 7]
ALPHA = 1 / 3

N_UNKNOWN = 245
N_KNOWN = 569

N_BOOTSTRAP = 3000
SEED = 20261020

MIN_MACRO_GAIN = 0.03
MAX_CLASS_DROP = 0.03
MIN_ID_SHIFT = 40

OOF_PATH = os.path.join(
    OUT, "beats_mahalanobis_oof.csv"
)

SHIFT_PATH = os.path.join(
    OUT, "beats_mahalanobis_test_id_shift.csv"
)

assert os.path.isfile(OOF_PATH), OOF_PATH
assert os.path.isfile(SHIFT_PATH), SHIFT_PATH

# ------------------------------------------------------------
# 2. BACA OOF YANG SUDAH DIHITUNG
# ------------------------------------------------------------

oof = pd.read_csv(OOF_PATH)

required = {
    "sample_idx",
    "heldout_class",
    "source_group",
    "is_pseudo_unknown",
    "confidence_ood",
    "knn_distance_k3",
    "knn_distance_k7",
    "maha_ood_score",
}

missing = required - set(oof.columns)

if missing:
    raise ValueError(
        f"Kolom OOF tidak lengkap: {missing}"
    )

oof["sample_idx"] = oof["sample_idx"].astype(int)
oof["heldout_class"] = oof["heldout_class"].astype(str)
oof["source_group"] = oof["source_group"].astype(str)
oof["is_pseudo_unknown"] = (
    oof["is_pseudo_unknown"].astype(int)
)

assert len(oof) == 1826 * 3
assert oof["source_group"].nunique() == 15

key_cols = ["sample_idx", "heldout_class"]

assert not oof.duplicated(key_cols).any()

numeric_cols = [
    "confidence_ood",
    "knn_distance_k3",
    "knn_distance_k7",
    "maha_ood_score",
]

assert np.isfinite(
    oof[numeric_cols].to_numpy(dtype=float)
).all()

print("OOF shape:", oof.shape)
print("Source groups:", oof["source_group"].nunique())
print("Bootstrap target:", N_BOOTSTRAP)

# ------------------------------------------------------------
# 3. PERSIAPKAN POOL PREVALENCE-MATCHED
# ------------------------------------------------------------

class_pools = {}

for cls in CLASS_NAMES:

    part = oof[
        oof["heldout_class"] == cls
    ].copy().reset_index(drop=True)

    target = part[
        "is_pseudo_unknown"
    ].to_numpy(dtype=int)

    pos_idx = np.flatnonzero(target == 1)
    neg_idx = np.flatnonzero(target == 0)

    n_pos = min(N_UNKNOWN, len(pos_idx))
    n_neg = int(round(
        n_pos * N_KNOWN / N_UNKNOWN
    ))

    if n_neg > len(neg_idx):
        raise ValueError(
            f"Negatif tidak cukup untuk {cls}"
        )

    class_pools[cls] = {
        "data": part,
        "pos_idx": pos_idx,
        "neg_idx": neg_idx,
        "n_pos": n_pos,
        "n_neg": n_neg,
    }

    print(
        f"{cls}: {n_pos} positif, {n_neg} negatif"
    )

groups = np.array(
    sorted(oof["source_group"].unique()),
    dtype=object,
)

rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# 4. BOOTSTRAP BERPASANGAN BERDASARKAN SOURCE_GROUP
# ------------------------------------------------------------

boot_rows = []
valid_draws = 0
attempts = 0

for draw in range(N_BOOTSTRAP):

    group_counts = None

    # Bootstrap group ditolak jika suatu kelas kehilangan
    # semua group positif atau semua group negatif.
    for _ in range(1000):

        attempts += 1

        sampled_groups = rng.choice(
            groups,
            size=len(groups),
            replace=True,
        )

        counts = (
            pd.Series(sampled_groups)
            .value_counts()
            .to_dict()
        )

        valid = True

        for cls in CLASS_NAMES:

            cfg = class_pools[cls]
            part = cfg["data"]

            row_weights = (
                part["source_group"]
                .map(counts)
                .fillna(0)
                .to_numpy(dtype=float)
            )

            if (
                row_weights[cfg["pos_idx"]].sum() <= 0
                or row_weights[cfg["neg_idx"]].sum() <= 0
            ):
                valid = False
                break

        if valid:
            group_counts = counts
            break

    if group_counts is None:
        continue

    valid_draws += 1

    for cls in CLASS_NAMES:

        cfg = class_pools[cls]
        part = cfg["data"]

        pos_idx = cfg["pos_idx"]
        neg_idx = cfg["neg_idx"]

        n_pos = cfg["n_pos"]
        n_neg = cfg["n_neg"]

        row_weights = (
            part["source_group"]
            .map(group_counts)
            .fillna(0)
            .to_numpy(dtype=float)
        )

        pos_weights = row_weights[pos_idx]
        neg_weights = row_weights[neg_idx]

        pos_probs = pos_weights / pos_weights.sum()
        neg_probs = neg_weights / neg_weights.sum()

        # Pool berpasangan: baseline dan kandidat memakai
        # sampel yang sama di setiap kelas dan draw.
        selected_pos = rng.choice(
            pos_idx,
            size=n_pos,
            replace=True,
            p=pos_probs,
        )

        selected_neg = rng.choice(
            neg_idx,
            size=n_neg,
            replace=True,
            p=neg_probs,
        )

        pool_idx = np.concatenate([
            selected_pos,
            selected_neg,
        ])

        rng.shuffle(pool_idx)

        pool = part.iloc[pool_idx]

        target = pool[
            "is_pseudo_unknown"
        ].to_numpy(dtype=int)

        R = int(target.sum())
        assert R == n_pos

        sample_ids = pool[
            "sample_idx"
        ].to_numpy(dtype=int)

        # Ranking dihitung di dalam pool.
        confidence_rank = pd.Series(
            pool["confidence_ood"].to_numpy(dtype=float)
        ).rank(
            pct=True,
            method="average",
        ).to_numpy()

        maha_rank = pd.Series(
            pool["maha_ood_score"].to_numpy(dtype=float)
        ).rank(
            pct=True,
            method="average",
        ).to_numpy()

        for k in K_VALUES:

            knn_col = f"knn_distance_k{k}"

            knn_rank = pd.Series(
                pool[knn_col].to_numpy(dtype=float)
            ).rank(
                pct=True,
                method="average",
            ).to_numpy()

            # Champion: confidence/kNN 50:50.
            baseline_score = (
                0.5 * confidence_rank
                + 0.5 * knn_rank
            )

            # Kandidat diagnostik: Mahalanobis 1/3.
            candidate_score = (
                (1 - ALPHA) * baseline_score
                + ALPHA * maha_rank
            )

            # Tie-break deterministik.
            baseline_order = np.lexsort((
                sample_ids,
                -baseline_score,
            ))

            candidate_order = np.lexsort((
                sample_ids,
                -candidate_score,
            ))

            baseline_rp = float(
                target[baseline_order[:R]].mean()
            )

            candidate_rp = float(
                target[candidate_order[:R]].mean()
            )

            boot_rows.append({
                "draw": draw,
                "heldout_class": cls,
                "k": k,
                "baseline_r_precision": baseline_rp,
                "candidate_r_precision": candidate_rp,
                "delta_r_precision": (
                    candidate_rp - baseline_rp
                ),
            })

if valid_draws < 0.95 * N_BOOTSTRAP:
    raise RuntimeError(
        f"Hanya {valid_draws}/{N_BOOTSTRAP} bootstrap valid."
    )

boot_df = pd.DataFrame(boot_rows)

print("\nBootstrap valid:", valid_draws, "/", N_BOOTSTRAP)
print("Total percobaan group draw:", attempts)

# ------------------------------------------------------------
# 5. RINGKAS HASIL PER KELAS
# ------------------------------------------------------------

class_summary = (
    boot_df.groupby(
        ["k", "heldout_class"],
        as_index=False,
    )
    .agg(
        baseline_r_precision=(
            "baseline_r_precision", "mean"
        ),
        candidate_r_precision=(
            "candidate_r_precision", "mean"
        ),
        mean_delta=(
            "delta_r_precision", "mean"
        ),
        sd_delta=(
            "delta_r_precision", "std"
        ),
    )
)

print("\n" + "=" * 70)
print("HASIL PER KELAS")
print("=" * 70)

display(class_summary.round(6))

# ------------------------------------------------------------
# 6. MACRO CI PADA SETIAP k
# ------------------------------------------------------------

macro_draw = (
    boot_df.groupby(
        ["draw", "k"],
        as_index=False,
    )["delta_r_precision"]
    .mean()
    .rename(
        columns={
            "delta_r_precision": "macro_delta"
        }
    )
)

summary_rows = []

for k in K_VALUES:

    part = macro_draw[
        macro_draw["k"] == k
    ]

    values = part[
        "macro_delta"
    ].to_numpy(dtype=float)

    ci_low, ci_high = np.quantile(
        values,
        [0.025, 0.975],
    )

    class_part = class_summary[
        class_summary["k"] == k
    ]

    summary_rows.append({
        "k": k,
        "maha_weight": ALPHA,
        "mean_macro_delta": float(values.mean()),
        "bootstrap_ci95_low": float(ci_low),
        "bootstrap_ci95_high": float(ci_high),
        "ci_lower_above_zero": bool(ci_low > 0),
        "worst_class_mean_delta": float(
            class_part["mean_delta"].min()
        ),
        "no_class_drop_over_003": bool(
            (class_part["mean_delta"] >= -MAX_CLASS_DROP).all()
        ),
    })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("HASIL KONFIRMASI BOOTSTRAP 3000")
print("=" * 70)

display(summary_df.round(6))

# ------------------------------------------------------------
# 7. GERBANG TETAP — TIDAK DIUBAH
# ------------------------------------------------------------

gain_ok = bool(
    (summary_df["mean_macro_delta"] >= MIN_MACRO_GAIN).all()
)

ci_ok = bool(
    (summary_df["bootstrap_ci95_low"] > 0).all()
)

class_ok = bool(
    summary_df["no_class_drop_over_003"].all()
)

shift_df = pd.read_csv(SHIFT_PATH)

shift_row = shift_df[
    (shift_df["k"] == 3)
    & np.isclose(
        shift_df["maha_weight"],
        ALPHA,
    )
]

if len(shift_row) != 1:
    raise ValueError(
        "Baris pergeseran ID k=3, bobot 1/3 tidak ditemukan."
    )

id_changes = int(
    shift_row.iloc[0]["symmetric_difference_ids"]
)

shift_ok = id_changes >= MIN_ID_SHIFT

gate = pd.DataFrame([{
    "gain_ge_003_both_k": gain_ok,
    "CI_positive_both_k": ci_ok,
    "no_class_drop_over_003": class_ok,
    "test_unknown_id_changes": id_changes,
    "test_shift_ge_40": shift_ok,
    "passes_all_gates": gain_ok and ci_ok and class_ok and shift_ok,
}])

print("\n" + "=" * 70)
print("GERBANG FINAL")
print("=" * 70)

display(gate)

# ------------------------------------------------------------
# 8. SIMPAN HASIL DIAGNOSTIK
# ------------------------------------------------------------

boot_path = os.path.join(
    OUT,
    "beats_mahalanobis_group_bootstrap_3000.csv",
)

summary_path = os.path.join(
    OUT,
    "beats_mahalanobis_group_bootstrap_3000_summary.csv",
)

class_path = os.path.join(
    OUT,
    "beats_mahalanobis_group_bootstrap_3000_per_class.csv",
)

gate_path = os.path.join(
    OUT,
    "beats_mahalanobis_group_bootstrap_3000_gate.csv",
)

boot_df.to_csv(boot_path, index=False)
summary_df.to_csv(summary_path, index=False)
class_summary.to_csv(class_path, index=False)
gate.to_csv(gate_path, index=False)

print("\nFile tersimpan:")
print(boot_path)
print(summary_path)
print(class_path)
print(gate_path)

if gate["passes_all_gates"].iloc[0]:
    print(
        "\nSEMUA GERBANG LOLOS. "
        "Kandidat boleh dipertimbangkan untuk evaluasi lanjutan, "
        "tetapi belum otomatis harus disubmit."
    )
else:
    print(
        "\nGERBANG BELUM LOLOS. "
        "Jangan submit; pertahankan champion."
    )

OOF shape: (5478, 9)
Source groups: 15
Bootstrap target: 3000
Bondol Jawa: 245 positif, 569 negatif
Cendrawasih: 83 positif, 193 negatif
Kutilang: 245 positif, 569 negatif

Bootstrap valid: 3000 / 3000
Total percobaan group draw: 3069

HASIL PER KELAS


,k,heldout_class,baseline_r_precision,candidate_r_precision,mean_delta,sd_delta
0,3,Bondol Jawa,0.556188,0.560554,0.004366,0.029169
1,3,Cendrawasih,0.682112,0.792715,0.110602,0.059288
2,3,Kutilang,0.458777,0.472736,0.013959,0.021208
3,7,Bondol Jawa,0.546354,0.555332,0.008978,0.029629
4,7,Cendrawasih,0.684839,0.793365,0.108526,0.059491
5,7,Kutilang,0.448702,0.467033,0.018331,0.021306



HASIL KONFIRMASI BOOTSTRAP 3000


,k,maha_weight,mean_macro_delta,bootstrap_ci95_low,bootstrap_ci95_high,ci_lower_above_zero,worst_class_mean_delta,no_class_drop_over_003
0,3,0.333333,0.042976,0.001162,0.084665,True,0.004366,True
1,7,0.333333,0.045278,0.003669,0.087452,True,0.008978,True



GERBANG FINAL


,gain_ge_003_both_k,CI_positive_both_k,no_class_drop_over_003,test_unknown_id_changes,test_shift_ge_40,passes_all_gates
0,True,True,True,60,True,True



File tersimpan:
/kaggle/working/beats_mahalanobis_group_bootstrap_3000.csv
/kaggle/working/beats_mahalanobis_group_bootstrap_3000_summary.csv
/kaggle/working/beats_mahalanobis_group_bootstrap_3000_per_class.csv
/kaggle/working/beats_mahalanobis_group_bootstrap_3000_gate.csv

SEMUA GERBANG LOLOS. Kandidat boleh dipertimbangkan untuk evaluasi lanjutan, tetapi belum otomatis harus disubmit.


In [31]:
import os
import pandas as pd

FILES_TO_CHECK = {
    "Ranking fusion champion": (
        "/kaggle/working/triple_v4_unknown_count_rank_scores.csv"
    ),
    "ID perubahan Mahalanobis": (
        "/kaggle/working/beats_mahalanobis_test_id_shift.csv"
    ),
    "Gate Mahalanobis": (
        "/kaggle/working/beats_mahalanobis_gate.csv"
    ),
    "Submission champion": (
        "/kaggle/working/submission_triple_v4_conf_knn50_k3_u245.csv"
    ),
    "Sample submission": (
        "/kaggle/input/competitions/schematics-2026/sample_submission.csv"
    ),
}

for name, path in FILES_TO_CHECK.items():
    print("\n" + "=" * 70)
    print(name)
    print("Path:", path)
    print("Ada:", os.path.exists(path))

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Kolom:", df.columns.tolist())
        display(df.head(3))


Ranking fusion champion
Path: /kaggle/working/triple_v4_unknown_count_rank_scores.csv
Ada: True
Shape: (814, 43)
Kolom: ['file_id', 'triple_v4_pred_known_class', 'triple_v4_confidence', 'triple_v4_uncertainty', 'triple_v4_entropy_norm', 'triple_v4_disagreement_tv', 'beats_cosine_proto_distance', 'beats_knn_cosine_distance', 'beats_diag_mahalanobis', 'rms', 'rms_db', 'zero_crossing_rate', 'spectral_flatness_median', 'spectral_flatness_mean', 'spectral_entropy_mean', 'spectral_centroid_hz', 'low_band_fraction_lt500hz', 'high_band_fraction_ge2500hz', 'spectral_flatness_p90', 'rank_uncertainty', 'rank_entropy', 'rank_disagreement_tv', 'rank_margin_low_is_ood', 'rank_beats_cosine_proto_distance', 'rank_beats_knn_cosine_distance', 'rank_beats_diag_mahalanobis', 'rank_acoustic_rms', 'rank_acoustic_rms_db', 'rank_acoustic_zero_crossing_rate', 'rank_acoustic_spectral_flatness_median', 'rank_acoustic_spectral_flatness_mean', 'rank_acoustic_spectral_entropy_mean', 'rank_acoustic_spectral_centroi

,file_id,triple_v4_pred_known_class,triple_v4_confidence,triple_v4_uncertainty,triple_v4_entropy_norm,triple_v4_disagreement_tv,beats_cosine_proto_distance,beats_knn_cosine_distance,beats_diag_mahalanobis,rms,rms_db,zero_crossing_rate,spectral_flatness_median,spectral_flatness_mean,spectral_entropy_mean,spectral_centroid_hz,low_band_fraction_lt500hz,high_band_fraction_ge2500hz,spectral_flatness_p90,rank_uncertainty,rank_entropy,rank_disagreement_tv,rank_margin_low_is_ood,rank_beats_cosine_proto_distance,rank_beats_knn_cosine_distance,rank_beats_diag_mahalanobis,rank_acoustic_rms,rank_acoustic_rms_db,rank_acoustic_zero_crossing_rate,rank_acoustic_spectral_flatness_median,rank_acoustic_spectral_flatness_mean,rank_acoustic_spectral_entropy_mean,rank_acoustic_spectral_centroid_hz,rank_acoustic_low_band_fraction_lt500hz,rank_acoustic_high_band_fraction_ge2500hz,rank_acoustic_spectral_flatness_p90,ood_rank_score_triple_v4_uncertainty_disagreement_60_40,ood_rank_score_beats_embedding_only,ood_rank_score_triple_plus_beats_embedding,ood_rank_score_all_signals_exploratory,confidence_uncertainty_rank,knn_distance_rank,combined_rank_score_50_50
0,7e05954cc1e90e75330575982ab9dac4717fad2fb70599...,Cendrawasih,0.871516,0.128484,0.412344,0.122858,0.137142,0.159475,0.978920,0.090990,-20.820170,0.435886,0.190125,0.180382,0.821792,3312.4229,0.136696,0.611131,0.237828,0.756846,0.757941,0.738226,0.759036,0.340088,0.969880,0.481380,0.645674,0.645674,0.684556,0.847755,0.817087,0.880066,0.618839,0.469880,0.537240,0.708653,0.749398,0.597116,0.695016,0.720564,0.253071,0.808354,0.530713
1,ac5e9f292b26bb1e0557beb235b9d6bcb066cc10834d98...,Cendrawasih,0.866702,0.133298,0.439079,0.115825,0.137704,0.165595,0.948896,0.069712,-23.133884,0.352589,0.104021,0.097222,0.705516,2539.5370,0.113075,0.445324,0.153526,0.761227,0.772727,0.725630,0.748631,0.347207,0.976451,0.384995,0.342278,0.342278,0.489047,0.466046,0.392662,0.306134,0.416758,0.411829,0.354874,0.398138,0.746988,0.569551,0.686665,0.612749,0.266585,0.829238,0.547912
2,e8ace030a12a1b16784f6b64a47d79793e3fdeb432d292...,Cendrawasih,0.893650,0.106350,0.369140,0.078286,0.149420,0.187454,0.981871,0.089074,-21.004997,0.386159,0.177687,0.172561,0.788604,2890.5830,0.174828,0.558349,0.223095,0.722892,0.730011,0.624863,0.722344,0.469880,0.989595,0.492333,0.619934,0.619934,0.573932,0.815991,0.787514,0.727820,0.510953,0.560241,0.481380,0.671961,0.683680,0.650602,0.682284,0.693231,0.213759,0.901720,0.557740



ID perubahan Mahalanobis
Path: /kaggle/working/beats_mahalanobis_test_id_shift.csv
Ada: True
Shape: (4, 6)
Kolom: ['k', 'maha_weight', 'unknown_retained', 'unknown_out', 'unknown_in', 'symmetric_difference_ids']


,k,maha_weight,unknown_retained,unknown_out,unknown_in,symmetric_difference_ids
0,3,0.100000,234,11,11,22
1,7,0.100000,232,13,13,26
2,3,0.333333,215,30,30,60



Gate Mahalanobis
Path: /kaggle/working/beats_mahalanobis_gate.csv
Ada: True
Shape: (2, 9)
Kolom: ['maha_weight', 'role', 'gain_ge_003_both_k', 'CI_positive_both_k', 'no_class_drop_gt_003', 'passes_offline_gates_1_to_4', 'test_unknown_id_changes_k3', 'test_shift_ge_40', 'passes_all_gates']


,maha_weight,role,gain_ge_003_both_k,CI_positive_both_k,no_class_drop_gt_003,passes_offline_gates_1_to_4,test_unknown_id_changes_k3,test_shift_ge_40,passes_all_gates
0,0.100000,exploratory_only,False,False,True,False,22,False,False
1,0.333333,candidate_if_all_gates_pass,True,False,True,False,60,True,False



Submission champion
Path: /kaggle/working/submission_triple_v4_conf_knn50_k3_u245.csv
Ada: True
Shape: (814, 2)
Kolom: ['file_id', 'label']


,file_id,label
0,7e05954cc1e90e75330575982ab9dac4717fad2fb70599...,Cendrawasih
1,ac5e9f292b26bb1e0557beb235b9d6bcb066cc10834d98...,Cendrawasih
2,e8ace030a12a1b16784f6b64a47d79793e3fdeb432d292...,Cendrawasih



Sample submission
Path: /kaggle/input/competitions/schematics-2026/sample_submission.csv
Ada: True
Shape: (814, 2)
Kolom: ['file_id', 'label']


,file_id,label
0,7e05954cc1e90e75330575982ab9dac4717fad2fb70599...,NaN
1,ac5e9f292b26bb1e0557beb235b9d6bcb066cc10834d98...,NaN
2,e8ace030a12a1b16784f6b64a47d79793e3fdeb432d292...,NaN


In [32]:
import os
import glob
import pandas as pd

pattern = "/kaggle/working/*mahalanobis*"
files = sorted(glob.glob(pattern))

print("=" * 75)
print("FILE MAHALANOBIS YANG TERSEDIA")
print("=" * 75)

for path in files:
    print(f"\nFile: {os.path.basename(path)}")
    print(f"Ukuran: {os.path.getsize(path):,} bytes")

    if path.lower().endswith(".csv"):
        try:
            df = pd.read_csv(path)
            print("Shape:", df.shape)
            print("Kolom:", df.columns.tolist())

            if len(df) > 0:
                display(df.head(2))
        except Exception as e:
            print("Gagal membaca CSV:", repr(e))

FILE MAHALANOBIS YANG TERSEDIA

File: beats_mahalanobis_auc_by_class.csv
Ukuran: 120 bytes
Shape: (3, 2)
Kolom: ['heldout_class', 'auc_mahalanobis']


,heldout_class,auc_mahalanobis
0,Bondol Jawa,0.768468
1,Cendrawasih,0.936690



File: beats_mahalanobis_gate.csv
Ukuran: 307 bytes
Shape: (2, 9)
Kolom: ['maha_weight', 'role', 'gain_ge_003_both_k', 'CI_positive_both_k', 'no_class_drop_gt_003', 'passes_offline_gates_1_to_4', 'test_unknown_id_changes_k3', 'test_shift_ge_40', 'passes_all_gates']


,maha_weight,role,gain_ge_003_both_k,CI_positive_both_k,no_class_drop_gt_003,passes_offline_gates_1_to_4,test_unknown_id_changes_k3,test_shift_ge_40,passes_all_gates
0,0.100000,exploratory_only,False,False,True,False,22,False,False
1,0.333333,candidate_if_all_gates_pass,True,False,True,False,60,True,False



File: beats_mahalanobis_group_bootstrap_3000.csv
Ukuran: 1,351,813 bytes
Shape: (18000, 6)
Kolom: ['draw', 'heldout_class', 'k', 'baseline_r_precision', 'candidate_r_precision', 'delta_r_precision']


,draw,heldout_class,k,baseline_r_precision,candidate_r_precision,delta_r_precision
0,0,Bondol Jawa,3,0.591837,0.620408,0.028571
1,0,Bondol Jawa,7,0.587755,0.628571,0.040816


... (12 outputs left out)


File: beats_mahalanobis_prevalence_matched_summary.csv
Ukuran: 568 bytes
Shape: (4, 8)
Kolom: ['k', 'maha_weight', 'mean_macro_delta', 'bootstrap_ci95_low', 'bootstrap_ci95_high', 'ci_lower_above_zero', 'worst_class_mean_delta', 'no_class_drop_over_003']


,k,maha_weight,mean_macro_delta,bootstrap_ci95_low,bootstrap_ci95_high,ci_lower_above_zero,worst_class_mean_delta,no_class_drop_over_003
0,3,0.100000,0.017097,-0.002921,0.042267,False,0.002588,True
1,3,0.333333,0.042696,-0.000863,0.085235,False,0.002873,True



File: beats_mahalanobis_test_id_shift.csv
Ukuran: 185 bytes
Shape: (4, 6)
Kolom: ['k', 'maha_weight', 'unknown_retained', 'unknown_out', 'unknown_in', 'symmetric_difference_ids']


,k,maha_weight,unknown_retained,unknown_out,unknown_in,symmetric_difference_ids
0,3,0.1,234,11,11,22
1,7,0.1,232,13,13,26


In [33]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors

# ============================================================
# 1. PATH DAN KONFIGURASI
# ============================================================

WORK = "/kaggle/working"
COMP = "/kaggle/input/competitions/schematics-2026"

TRAIN_EMB_PATH = os.path.join(
    WORK, "beats_iter3plus_as2m_8s_train_embeddings.npy"
)
TEST_EMB_PATH = os.path.join(
    WORK, "beats_iter3plus_as2m_8s_test_embeddings.npy"
)
RANK_PATH = os.path.join(
    WORK, "triple_v4_unknown_count_rank_scores.csv"
)
CHAMPION_PATH = os.path.join(
    WORK, "submission_triple_v4_conf_knn50_k3_u245.csv"
)
SAMPLE_PATH = os.path.join(COMP, "sample_submission.csv")
META_PATH = os.path.join(COMP, "train/metadata.csv")

OUT_PATH = os.path.join(
    WORK, "submission_triple_v4_conf_knn50_k3_maha33_u245.csv"
)
SCORE_PATH = os.path.join(
    WORK, "beats_mahalanobis_test_scores_reconstructed.csv"
)

N_UNKNOWN = 245
MAHA_WEIGHT = 1.0 / 3.0
KNN_K = 3
PCA_COMPONENTS = 128
PCA_SEED = int(globals().get("SEED", 42))

# ============================================================
# 2. LOAD DATA DAN PERIKSA ALIGNMENT
# ============================================================

for path in [
    TRAIN_EMB_PATH, TEST_EMB_PATH, RANK_PATH,
    CHAMPION_PATH, SAMPLE_PATH, META_PATH
]:
    assert os.path.exists(path), f"File tidak ditemukan: {path}"

train_emb = np.load(TRAIN_EMB_PATH).astype(np.float64)
test_emb = np.load(TEST_EMB_PATH).astype(np.float64)

train_meta = pd.read_csv(META_PATH)
rank_df = pd.read_csv(RANK_PATH)
champion = pd.read_csv(CHAMPION_PATH)
sample_sub = pd.read_csv(SAMPLE_PATH)

# Gunakan train_df apabila tersedia agar urutan label sesuai
# dengan embedding yang dibuat pada notebook saat ini.
if (
    "train_df" in globals()
    and isinstance(globals()["train_df"], pd.DataFrame)
    and len(globals()["train_df"]) == len(train_emb)
    and "label" in globals()["train_df"].columns
):
    labels = globals()["train_df"]["label"].astype(str).to_numpy()
    label_source = "train_df di notebook"
else:
    assert len(train_meta) == len(train_emb), (
        "Jumlah metadata train tidak sama dengan jumlah embedding."
    )
    labels = train_meta["label"].astype(str).to_numpy()
    label_source = "metadata.csv"

assert train_emb.ndim == 2 and test_emb.ndim == 2
assert train_emb.shape[1] == test_emb.shape[1]
assert len(labels) == len(train_emb)
assert len(sample_sub) == len(test_emb) == 814
assert len(rank_df) == len(test_emb)
assert rank_df["file_id"].is_unique
assert sample_sub["file_id"].is_unique
assert champion["file_id"].is_unique

assert {
    "file_id", "triple_v4_pred_known_class",
    "confidence_uncertainty_rank"
}.issubset(rank_df.columns)

assert set(sample_sub["file_id"]) == set(rank_df["file_id"])
assert set(sample_sub["file_id"]) == set(champion["file_id"])

classes = np.unique(labels)
assert len(classes) == 3, f"Harus ada 3 kelas, ditemukan: {classes}"

print("Train embedding:", train_emb.shape)
print("Test embedding :", test_emb.shape)
print("Sumber label   :", label_source)
print("Kelas          :", classes.tolist())

# Selaraskan semua data test berdasarkan file_id sample submission.
rank_df = (
    sample_sub[["file_id"]]
    .merge(rank_df, on="file_id", how="left", validate="one_to_one")
)

champion = (
    sample_sub[["file_id"]]
    .merge(
        champion[["file_id", "label"]],
        on="file_id",
        how="left",
        validate="one_to_one"
    )
)

assert not rank_df["confidence_uncertainty_rank"].isna().any()
assert not champion["label"].isna().any()

# ============================================================
# 3. RECONSTRUCT KNN k=3 UNTUK DATA TEST
# ============================================================

knn = NearestNeighbors(
    n_neighbors=KNN_K,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)
knn.fit(train_emb)

knn_distances, _ = knn.kneighbors(test_emb)
knn_k3_distance = knn_distances.mean(axis=1)

# Rank lebih tinggi = lebih berpotensi Unknown.
knn_k3_rank = pd.Series(knn_k3_distance).rank(
    method="average", pct=True
).to_numpy()

confidence_rank = rank_df[
    "confidence_uncertainty_rank"
].to_numpy(dtype=float)

# Baseline fusion confidence/kNN 50:50.
baseline_score = 0.5 * confidence_rank + 0.5 * knn_k3_rank

# ============================================================
# 4. FIT MAHALANOBIS PADA SELURUH TRAIN
# ============================================================

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_emb)
test_scaled = scaler.transform(test_emb)

pca = PCA(
    n_components=PCA_COMPONENTS,
    random_state=PCA_SEED
)
train_z = pca.fit_transform(train_scaled)
test_z = pca.transform(test_scaled)

# Centroid setiap kelas di ruang PCA.
centroids = {}
for cls in classes:
    centroids[cls] = train_z[labels == cls].mean(axis=0)

# Kovarians bersama dihitung dari residual dalam kelas.
residuals = np.empty_like(train_z)
for cls in classes:
    mask = labels == cls
    residuals[mask] = train_z[mask] - centroids[cls]

cov_model = LedoitWolf()
cov_model.fit(residuals)
precision = cov_model.precision_

# Skor OOD = jarak Mahalanobis kuadrat terkecil ke centroid kelas.
# Skor lebih tinggi berarti lebih jauh dari semua centroid.
maha_distances = []

for cls in classes:
    diff = test_z - centroids[cls]
    d2 = np.einsum(
        "ij,jk,ik->i",
        diff, precision, diff,
        optimize=True
    )
    maha_distances.append(d2)

maha_ood_score = np.min(
    np.column_stack(maha_distances),
    axis=1
)

maha_rank = pd.Series(maha_ood_score).rank(
    method="average", pct=True
).to_numpy()

# ============================================================
# 5. VALIDASI BASELINE TERHADAP CHAMPION
# ============================================================

base_top = set(
    rank_df.assign(_score=baseline_score)
    .sort_values(
        ["_score", "file_id"],
        ascending=[False, True],
        kind="mergesort"
    )
    .head(N_UNKNOWN)["file_id"]
)

champion_unknown = set(
    champion.loc[champion["label"] == "Unknown", "file_id"]
)

baseline_overlap = len(base_top & champion_unknown)
baseline_symmetric_diff = len(base_top ^ champion_unknown)

print("\n" + "=" * 70)
print("VALIDASI BASELINE k=3")
print("=" * 70)
print("Unknown pada champion       :", len(champion_unknown))
print("Unknown pada baseline baru  :", len(base_top))
print("ID yang sama                :", baseline_overlap)
print("Symmetric difference        :", baseline_symmetric_diff)

# Jangan lanjut apabila baseline tidak mereproduksi champion.
if baseline_symmetric_diff != 0:
    raise RuntimeError(
        "Baseline k=3 tidak mereproduksi champion secara persis. "
        "Submission tidak dibuat. Periksa metode ranking kNN "
        "dan confidence_uncertainty_rank sebelum melanjutkan."
    )

# ============================================================
# 6. FUSION MAHALANOBIS DENGAN BOBOT 1/3
# ============================================================

candidate_score = (
    (1.0 - MAHA_WEIGHT) * baseline_score
    + MAHA_WEIGHT * maha_rank
)

candidate_unknown = set(
    rank_df.assign(_score=candidate_score)
    .sort_values(
        ["_score", "file_id"],
        ascending=[False, True],
        kind="mergesort"
    )
    .head(N_UNKNOWN)["file_id"]
)

symmetric_diff = len(candidate_unknown ^ champion_unknown)
unknown_out = len(champion_unknown - candidate_unknown)
unknown_in = len(candidate_unknown - champion_unknown)

print("\n" + "=" * 70)
print("HASIL KANDIDAT MAHALANOBIS")
print("=" * 70)
print("Bobot Mahalanobis           :", MAHA_WEIGHT)
print("Jumlah Unknown kandidat     :", len(candidate_unknown))
print("Unknown keluar dari champion:", unknown_out)
print("Unknown masuk ke kandidat   :", unknown_in)
print("Symmetric difference        :", symmetric_diff)
print("PCA explained variance      :", round(
    pca.explained_variance_ratio_.sum(), 5
))

# Simpan diagnostik skor, bukan submission terlebih dahulu.
test_scores = rank_df[
    ["file_id", "triple_v4_pred_known_class"]
].copy()
test_scores["knn_k3_distance_reconstructed"] = knn_k3_distance
test_scores["baseline_score_reconstructed"] = baseline_score
test_scores["maha_ood_score_reconstructed"] = maha_ood_score
test_scores["maha_rank_reconstructed"] = maha_rank
test_scores["candidate_score_reconstructed"] = candidate_score
test_scores.to_csv(SCORE_PATH, index=False)

# Eksperimen sebelumnya melaporkan symmetric difference = 60.
# Jika tidak cocok, tahan submission untuk menghindari pipeline keliru.
if symmetric_diff != 60:
    raise RuntimeError(
        f"Pergeseran ID = {symmetric_diff}, bukan 60 seperti hasil "
        "eksperimen sebelumnya. Skor diagnostik sudah disimpan, "
        "tetapi submission sengaja TIDAK dibuat. Kirim output ini "
        "untuk mencocokkan kembali pipeline Mahalanobis."
    )

# ============================================================
# 7. BENTUK SUBMISSION BARU, CHAMPION TETAP AMAN
# ============================================================

candidate = sample_sub[["file_id"]].merge(
    rank_df[["file_id", "triple_v4_pred_known_class"]],
    on="file_id",
    how="left",
    validate="one_to_one"
)

candidate = candidate.rename(
    columns={"triple_v4_pred_known_class": "label"}
)
candidate.loc[
    candidate["file_id"].isin(candidate_unknown), "label"
] = "Unknown"

valid_labels = set(classes.tolist()) | {"Unknown"}

assert candidate.shape == (814, 2)
assert candidate["file_id"].is_unique
assert candidate["label"].notna().all()
assert set(candidate["label"]).issubset(valid_labels)
assert (candidate["label"] == "Unknown").sum() == N_UNKNOWN

# Prediksi kelas known harus sama dengan champion untuk semua
# baris yang sebelumnya bukan Unknown.
old_known = champion["label"] != "Unknown"
old_known_ids = set(champion.loc[old_known, "file_id"])

champion_known_map = champion.set_index("file_id")["label"]
candidate_known_map = candidate.set_index("file_id")["label"]

assert all(
    candidate_known_map.loc[fid] == champion_known_map.loc[fid]
    for fid in old_known_ids
), "Prediksi kelas known berubah; submission dibatalkan."

candidate.to_csv(OUT_PATH, index=False)

# Validasi ulang file sesudah disimpan.
check = pd.read_csv(OUT_PATH)
assert check.shape == (814, 2)
assert (check["label"] == "Unknown").sum() == 245
assert set(check["file_id"]) == set(sample_sub["file_id"])

print("\n" + "=" * 70)
print("KANDIDAT BERHASIL DIBUAT")
print("=" * 70)
print("Submission :", OUT_PATH)
print("Skor test  :", SCORE_PATH)
print("\nDistribusi label:")
print(check["label"].value_counts())
print("\nChampion lama tetap:", os.path.exists(CHAMPION_PATH))
print("Champion lama tidak ditimpa.")

Train embedding: (1826, 768)
Test embedding : (814, 768)
Sumber label   : metadata.csv
Kelas          : ['Bondol Jawa', 'Cendrawasih', 'Kutilang']

VALIDASI BASELINE k=3
Unknown pada champion       : 245
Unknown pada baseline baru  : 245
ID yang sama                : 245
Symmetric difference        : 0

HASIL KANDIDAT MAHALANOBIS
Bobot Mahalanobis           : 0.3333333333333333
Jumlah Unknown kandidat     : 245
Unknown keluar dari champion: 30
Unknown masuk ke kandidat   : 30
Symmetric difference        : 60
PCA explained variance      : 0.92199


AssertionError: Prediksi kelas known berubah; submission dibatalkan.

In [34]:
import os
import numpy as np
import pandas as pd

WORK = "/kaggle/working"
COMP = "/kaggle/input/competitions/schematics-2026"

SAMPLE_PATH = os.path.join(COMP, "sample_submission.csv")
META_PATH = os.path.join(COMP, "train/metadata.csv")
RANK_PATH = os.path.join(
    WORK, "triple_v4_unknown_count_rank_scores.csv"
)
CHAMPION_PATH = os.path.join(
    WORK, "submission_triple_v4_conf_knn50_k3_u245.csv"
)
SCORE_PATH = os.path.join(
    WORK, "beats_mahalanobis_test_scores_reconstructed.csv"
)

# Nama baru: champion tidak disentuh dan file kandidat sebelumnya
# tidak akan tertimpa.
OUT_PATH = os.path.join(
    WORK,
    "submission_triple_v4_conf_knn50_k3_maha33_u245_validated.csv"
)

N_UNKNOWN = 245

# ============================================================
# 1. BACA DAN SELARASKAN FILE
# ============================================================

for path in [SAMPLE_PATH, META_PATH, RANK_PATH,
             CHAMPION_PATH, SCORE_PATH]:
    assert os.path.exists(path), f"File tidak ditemukan: {path}"

sample = pd.read_csv(SAMPLE_PATH)
meta = pd.read_csv(META_PATH)
rank = pd.read_csv(RANK_PATH)
champion = pd.read_csv(CHAMPION_PATH)
scores = pd.read_csv(SCORE_PATH)

assert len(sample) == 814
assert sample["file_id"].is_unique
assert rank["file_id"].is_unique
assert champion["file_id"].is_unique
assert scores["file_id"].is_unique

assert set(sample["file_id"]) == set(champion["file_id"])
assert set(sample["file_id"]) == set(rank["file_id"])
assert set(sample["file_id"]) == set(scores["file_id"])

# Urutkan sesuai sample_submission.
champion = sample[["file_id"]].merge(
    champion[["file_id", "label"]],
    on="file_id",
    how="left",
    validate="one_to_one"
)

rank = sample[["file_id"]].merge(
    rank[["file_id", "triple_v4_pred_known_class"]],
    on="file_id",
    how="left",
    validate="one_to_one"
)

scores = sample[["file_id"]].merge(
    scores[["file_id", "candidate_score_reconstructed"]],
    on="file_id",
    how="left",
    validate="one_to_one"
)

assert champion["label"].notna().all()
assert rank["triple_v4_pred_known_class"].notna().all()
assert scores["candidate_score_reconstructed"].notna().all()
assert np.isfinite(
    scores["candidate_score_reconstructed"].to_numpy()
).all()

known_classes = set(meta["label"].astype(str).unique())
assert len(known_classes) == 3

predicted_known = set(
    rank["triple_v4_pred_known_class"].astype(str).unique()
)
assert predicted_known.issubset(known_classes), (
    f"Ada kelas prediksi yang tidak valid: "
    f"{predicted_known - known_classes}"
)

# ============================================================
# 2. PILIH TEPAT 245 UNKNOWN DARI SKOR KANDIDAT
# ============================================================

scoring = scores.sort_values(
    ["candidate_score_reconstructed", "file_id"],
    ascending=[False, True],
    kind="mergesort"
)

candidate_unknown = set(
    scoring.head(N_UNKNOWN)["file_id"]
)

champion_unknown = set(
    champion.loc[champion["label"] == "Unknown", "file_id"]
)

unknown_out = champion_unknown - candidate_unknown
unknown_in = candidate_unknown - champion_unknown
symmetric_difference = len(unknown_out) + len(unknown_in)

print("=" * 65)
print("VALIDASI PEMILIHAN UNKNOWN")
print("=" * 65)
print("Unknown champion        :", len(champion_unknown))
print("Unknown kandidat        :", len(candidate_unknown))
print("Keluar dari Unknown     :", len(unknown_out))
print("Masuk ke Unknown        :", len(unknown_in))
print("Symmetric difference    :", symmetric_difference)

assert len(champion_unknown) == N_UNKNOWN
assert len(candidate_unknown) == N_UNKNOWN
assert len(unknown_out) == 30
assert len(unknown_in) == 30
assert symmetric_difference == 60

# ============================================================
# 3. PERTAHANKAN LABEL LAIN; UBAH HANYA 60 ID
# ============================================================

candidate = champion.copy()

# 30 ID masuk ke Unknown.
candidate.loc[
    candidate["file_id"].isin(unknown_in), "label"
] = "Unknown"

# 30 ID keluar dari Unknown: gunakan kelas prediksi known model.
pred_map = rank.set_index("file_id")[
    "triple_v4_pred_known_class"
]

released_ids = candidate["file_id"].isin(unknown_out)
candidate.loc[released_ids, "label"] = (
    candidate.loc[released_ids, "file_id"].map(pred_map)
)

# Validasi bahwa label Unknown sama persis dengan daftar skor.
actual_unknown = set(
    candidate.loc[candidate["label"] == "Unknown", "file_id"]
)
assert actual_unknown == candidate_unknown

# Validasi label dan jumlah perubahan.
valid_labels = known_classes | {"Unknown"}

assert candidate.shape == (814, 2)
assert candidate["file_id"].is_unique
assert candidate["label"].notna().all()
assert set(candidate["label"].astype(str)).issubset(valid_labels)
assert (candidate["label"] == "Unknown").sum() == 245

changed = candidate.merge(
    champion,
    on="file_id",
    suffixes=("_new", "_old"),
    validate="one_to_one"
)

changed_count = int(
    (changed["label_new"] != changed["label_old"]).sum()
)

print("\n" + "=" * 65)
print("VALIDASI LABEL")
print("=" * 65)
print("Jumlah label yang berubah:", changed_count)
print("Distribusi kandidat:")
print(candidate["label"].value_counts())

assert changed_count == 60

# Jangan menimpa file kandidat yang mungkin sudah ada.
assert not os.path.exists(OUT_PATH), (
    f"File tujuan sudah ada: {OUT_PATH}. "
    "Pilih nama output baru agar tidak menimpa file."
)
assert os.path.abspath(OUT_PATH) != os.path.abspath(CHAMPION_PATH)

# ============================================================
# 4. SIMPAN DAN BACA ULANG UNTUK VALIDASI AKHIR
# ============================================================

candidate.to_csv(OUT_PATH, index=False)
check = pd.read_csv(OUT_PATH)

assert check.shape == (814, 2)
assert check["file_id"].is_unique
assert check["label"].notna().all()
assert (check["label"] == "Unknown").sum() == 245
assert set(check["file_id"]) == set(sample["file_id"])

# Champion harus tetap tersedia dan tidak diubah oleh cell ini.
assert os.path.exists(CHAMPION_PATH)

print("\n" + "=" * 65)
print("SUBMISSION KANDIDAT BERHASIL DIBUAT")
print("=" * 65)
print("File:", OUT_PATH)
print("Shape:", check.shape)
print("Jumlah Unknown:", int((check["label"] == "Unknown").sum()))
print("Jumlah ID berubah:", changed_count)
print("Champion tetap aman:", CHAMPION_PATH)
print("Validasi akhir: LULUS")

VALIDASI PEMILIHAN UNKNOWN
Unknown champion        : 245
Unknown kandidat        : 245
Keluar dari Unknown     : 30
Masuk ke Unknown        : 30
Symmetric difference    : 60

VALIDASI LABEL
Jumlah label yang berubah: 60
Distribusi kandidat:
label
Kutilang       269
Unknown        245
Bondol Jawa    237
Cendrawasih     63
Name: count, dtype: int64

SUBMISSION KANDIDAT BERHASIL DIBUAT
File: /kaggle/working/submission_triple_v4_conf_knn50_k3_maha33_u245_validated.csv
Shape: (814, 2)
Jumlah Unknown: 245
Jumlah ID berubah: 60
Champion tetap aman: /kaggle/working/submission_triple_v4_conf_knn50_k3_u245.csv
Validasi akhir: LULUS
